# Restoring the register after a lesion (FLUX.1-dev)

This code runs lesion and rescue experiments on the register token in the
FLUX.1-dev image stream. The dominant channel c\* is 154 and carries the
register direction v\*. We break one part of the register at a single block
and denoising step, put one candidate mediator back to its clean-run value
and nothing else, and measure whether the register's attention sink comes
back. Every earlier step is replayed exactly from the baseline run, and the
later steps run without intervention.

There are three lesion and rescue families, all applied to the register
token.

1. c\* suppression with v\* rescue (`cstar_zero`, `cstar_zero__vstar`).
   Channel 154 of the register is set to zero at the lesion block. At the
   next residual state only the projection onto v\* (one scalar) is restored
   to its clean value. As a lower bound, `cstar_zero__norm` restores the
   clean norm and keeps the lesioned direction. As an upper bound,
   `cstar_zero__full` restores the whole clean vector.
2. v\* destruction with natural-key rescue (`vstar_destroy`,
   `vstar_destroy__key`). The v\* component is removed at constant norm. In
   the attention of the following blocks only the register's final key
   (after QK normalization and rotary embedding) is replaced by the clean
   key, and the residual state stays lesioned. The upper bound is
   `vstar_destroy__full`.
3. Register removal with channel rescue (`reg_remove`, `reg_remove__cstar`).
   The register is replaced by the mean ordinary token, and at the next
   residual state only channel 154 is restored. The upper bound is
   `reg_remove__full`.

With `baseline` and a `sham` condition (the stored clean register vector,
key and value written back unchanged) this makes 12 default conditions.
More controls can be added by name through `EXTRA_CONDITIONS`, for example
a partial v\* dose (`cstar_zero__vstar_f0.5`), a value instead of a key
rescue (`vstar_destroy__value`), control channels (`reg_remove__chan_ctrl`)
or the channel write at a random ordinary token
(`reg_remove__cstar_at_random`).

At the intervened step we read out, at every block, how much attention the
register receives, its key rank, how often it is the top sink, how far the
attention pattern moved from the clean run, and the register geometry. We
also keep the velocity prediction at that step, the distance of the
ordinary tokens to the clean state at two reference blocks, and the final
image.

Endpoints are paired within a scenario (one prompt and seed). Rescue
fractions get prompt-clustered bootstrap intervals, and the controls are
judged against equivalence margins fixed in advance. The smoke test checks
that the replay is exact and that every rescue hits its clean value, and
the audit has to pass before the analysis runs.

## How to run

1. Use an A100 GPU runtime with gated access to FLUX.1-dev and a read token
   in the Colab secret `HF_TOKEN`. Outputs go to the Drive folder
   `MyDrive/flux_ma_channels`, which must already hold the outputs of the
   channel recurrence run: `meta/prompts.json`, `meta/pilot_decision.json`,
   the prompt-embedding cache and the five pilot npz files. v\* comes from
   one of two places. If the earlier dissolution experiment's frozen files
   `meta/intervention_decision_v3.json` and `meta/vstar_v3.npz` are there,
   they are reused, so v\* and the channel identities are exactly the same
   as in that experiment. Otherwise v\* is refitted with the same definition
   from the recurrence run's full tensors `main/*__Xfull_*.npy`. If anything
   is missing, the code stops and says what.
2. Run the cells top to bottom. The gates, in order, are the pipeline smoke
   test (S3c), the frozen decision and v\* (R2), the CPU
   operator unit tests (R3), and the rescue smoke test (R5), which checks
   replay exactness and every lesion and rescue. Then come the capture (R6),
   the audit (R7) and the analysis (R8 to R10).
3. `scenario_scope` in R1b defaults to `"full"` (100 scenarios). Set it to
   `"pilot"` for a five-scenario dry run. The capture is resumable. After a
   disconnect, run from the top through R6 and it continues from the last
   finished run.
4. To add conditions, list their names in `EXTRA_CONDITIONS` (R1a). This
   only adds runs, since the condition list is not part of the run hash or
   the frozen decision. The intervention steps are part of the decision, so
   adding a step (for example `("tstar", "early")`) needs
   `force_redecide=True` and redoes the finished runs. Changing the
   decision, v\* or any hashed config field (`_HASH_EXCLUDE` in R1b lists
   the fields that are not hashed) also invalidates finished runs.

On an A100 40 GB the default set of 12 conditions times 100 scenarios is
1200 generations. The baseline runs all 28 steps (about 22 s).
Every other run replays the clean prefix and computes only the 7 steps from
step 21 on, about 8 to 10 s including the all-block attention probe. That's
about 3 hours in total, and the extended set adds about 4.5 hours. With
every optional array enabled the outputs take about 20 GiB on Drive, or
about 9 GiB without the later-step token norms and the stored attention
distributions.

## S0. Environment

Installs the libraries with version bounds (Colab's preinstalled torch is left
alone), mounts Drive, checks free space and picks the precision mode from the
GPU. On a GPU with at least 35 GiB (A100) the model runs in bf16. Smaller GPUs
fall back to quantized weights, which is recorded in every output file as a
possible confound. The cell also checks access to the gated FLUX.1-dev
weights before any compute is spent.

In [ ]:
# =====================================================================
# S0. Environment setup
# =====================================================================
import importlib, importlib.util, subprocess, sys, os

def _pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[setup][warn] pip install {' '.join(pkgs)} failed:\n{r.stderr[-800:]}")

# Leave torch alone. Colab's preinstalled build matches its GPUs.
_pip("diffusers>=0.32.0,<0.40", "transformers>=4.44.0", "accelerate>=0.33.0",
     "safetensors", "sentencepiece", "protobuf", "huggingface_hub>=0.24.0",
     "pyarrow", "pandas")
# Quantization back ends, only needed for the fallback on GPUs below 35 GiB.
_pip("torchao", "bitsandbytes")

import gc, json, math, time, glob, re, shutil, hashlib, getpass, platform, datetime, csv
import numpy as np
import torch

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# ---------------------------------------------------------------- Drive
BASE = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = "/content/drive/MyDrive/flux_ma_channels"
    except Exception as e:
        print(f"[setup][warn] Drive mount failed ({e}); falling back to local disk "
              f"-- outputs will NOT persist across sessions!")
if BASE is None:
    BASE = os.path.abspath("./flux_ma_channels")

PATHS = {
    "base":    BASE,
    "meta":    os.path.join(BASE, "meta"),
    "embeds":  os.path.join(BASE, "prompt_embeds"),
    "pilot":   os.path.join(BASE, "pilot"),
    "main":    os.path.join(BASE, "main"),
    "smoke":   os.path.join(BASE, "smoke"),
    "figures": os.path.join(BASE, "figures"),
    "analysis": os.path.join(BASE, "analysis"),
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
RUN_LOG_PATH   = os.path.join(PATHS["meta"], "run_log.csv")
ERROR_LOG_PATH = os.path.join(PATHS["meta"], "errors.log")
print(f"[setup] output root: {BASE}")

# ------------------------------------------------------- Drive preflight
try:
    du = shutil.disk_usage(BASE)
    free_gib = du.free / 2**30
    print(f"[setup] free space at output root: {free_gib:.1f} GiB "
          f"(the intervention experiment needs ~15 GiB)")
    if free_gib < 16:
        print("[setup][WARN] <16 GiB free -- clear Drive space before the capture loop.")
except Exception as e:
    print(f"[setup][warn] could not check disk usage: {e}")

# ---------------------------------------------------------- GPU detect
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible. In Colab: Runtime -> Change runtime type -> "
        "GPU (A100 preferred, L4 fallback), then restart and re-run this cell.")
GPU_PROPS  = torch.cuda.get_device_properties(0)
GPU_NAME   = GPU_PROPS.name
TOTAL_GIB  = GPU_PROPS.total_memory / 2**30
CC         = (GPU_PROPS.major, GPU_PROPS.minor)
DTYPE      = torch.bfloat16 if CC >= (8, 0) else torch.float16
DEVICE     = "cuda"

# bf16 on GPUs with at least 35 GiB (A100), quantized weights otherwise.
if TOTAL_GIB >= 35.0:
    PRECISION_MODE = "bf16_full"
elif CC >= (8, 9):                     # L4 and other Ada GPUs, fp8 weight-only
    PRECISION_MODE = "quant_fp8"
else:                                  # T4, V100, A10 and so on, NF4 as a last resort
    PRECISION_MODE = "quant_nf4"

print(f"[setup] GPU: {GPU_NAME} | {TOTAL_GIB:.1f} GiB | compute capability {CC[0]}.{CC[1]}")
print(f"[setup] precision mode: {PRECISION_MODE} | activation dtype at hooks: fp32 "
      f"(model dtype {DTYPE})")
if PRECISION_MODE != "bf16_full":
    print("[setup][SCIENCE CAVEAT] Quantized fallback active: weight-only quantization "
          "perturbs the very activations we measure. Channel *identities* are expected "
          "to survive; magnitudes will shift. This flag is recorded in every output "
          "file and in the run manifest. If an A100 becomes available later, re-run "
          "the 5-scenario pilot in bf16 and compare top-10 sets (see manifest notes).")
    if PRECISION_MODE == "quant_nf4" and TOTAL_GIB < 20:
        print("[setup][WARN] Small GPU (likely T4): expect ~10 min/image at 1024^2. "
              "The main run will need multiple resumed sessions.")

# ------------------------------------------------------ Hugging Face auth
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        if HF_TOKEN:
            print("[setup] HF token loaded from Colab secret 'HF_TOKEN'.")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None
if not HF_TOKEN:
    print("No HF token found (Colab secret 'HF_TOKEN' or env var). Paste one now.")
    print("The token account must have ACCEPTED the license of "
          "black-forest-labs/FLUX.1-dev (gated model).")
    tok = getpass.getpass("HF_TOKEN (input hidden): ").strip()
    HF_TOKEN = tok or None

MODEL_ID = "black-forest-labs/FLUX.1-dev"
from huggingface_hub import model_info
try:
    _ = model_info(MODEL_ID, token=HF_TOKEN)
    print(f"[setup] Hugging Face access to {MODEL_ID}: OK")
except Exception as e:
    raise RuntimeError(
        f"Cannot access {MODEL_ID}. FLUX.1-dev is a *gated* model:\n"
        f"  1) visit https://huggingface.co/{MODEL_ID} and accept the license,\n"
        f"  2) create a READ token at https://huggingface.co/settings/tokens,\n"
        f"  3) store it as a Colab secret named HF_TOKEN (key icon in left sidebar),\n"
        f"  4) re-run this cell.\nUnderlying error: {e}") from e

# ------------------------------------------------- library version record
def _ver(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "n/a"
LIB_VERSIONS = {k: _ver(k) for k in
                ("torch", "diffusers", "transformers", "accelerate",
                 "numpy", "huggingface_hub", "torchao", "bitsandbytes")}
LIB_VERSIONS["python"] = platform.python_version()
print("[setup] versions:", json.dumps(LIB_VERSIONS))


## S1. Experiment config

All experiment parameters live in one frozen dataclass, and its hash is
stored in every output file. The rules for picking the analysis block and
step are part of this config and are fixed before the pilot. They are the
search band (blocks 18 to 39), the tie priority (block 22, then 18), the
analysis step (half way through sampling, or three quarters as the
fallback), the statistic-agreement thresholds and the anchor channels 154 and
1446. The pilot only supplies the data these rules are applied to.

In [ ]:
# =====================================================================
# S1. Experiment config
#
# The config hash is stored in every output file.
# =====================================================================
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class ExperimentConfig:
    # -- model and sampling -----------------------------------------------
    model_id: str = "black-forest-labs/FLUX.1-dev"
    height: int = 1024                    # -> 64x64 = 4096 image tokens
    width: int = 1024
    num_steps: int = 28                   # common FLUX.1-dev setting
    guidance_scale: float = 3.5           # FLUX.1-dev embedded-guidance default
    max_seq_len: int = 512                # T5 text tokens, the image tokens come after these
    # -- scenario grid -----------------------------------------------------
    seeds: tuple = (0, 42, 100, 1234, 2024)
    n_prompts: int = 20
    prompt_source_seed: int = 1234        # RNG seed for DiffusionDB stratified sampling
    # -- architecture expectations (FLUX.1-dev) ----------------------------
    d_model: int = 3072
    n_blocks_expected: int = 57           # 19 dual-stream (MMDiT) + 38 single-stream
    n_dual_expected: int = 19
    # -- analysis block and step selection (rules fixed before the pilot) ---
    band: tuple = (18, 39)                # high-norm band, the search range for l*
    block_tie_priority: tuple = (22, 18)  # ties go to block 22 (Gan et al.), then band onset 18
    block_tie_window_log10: float = 0.1   # blocks within 0.1 log10 of best count as ties
    t_star_default: int = 14              # 0-indexed, half way through the 28 steps
    t_star_fallback: int = 21             # 0-indexed, three quarters of the way
    tstar_mag_frac_min: float = 0.6       # top-1 magnitude at t* >= 60% of the final step's, pilot median
    tstar_stab_jaccard_min: float = 0.6   # median top-10 Jaccard, t* vs final step
    # -- statistic-agreement gate (P4, checked on the pilot before Phase B) ---
    primary_stat_default: str = "abs_signed_mean"   # |mean| (Turri et al., Eq. 1)
    stat_gate_jaccard_min: float = 0.8    # keep |mean| if median J(top10 |mean|, top10 mean|x|) >= this
    sign_consistency_min: float = 0.9     # top-10 channels by mean|x| below this count as mixed-sign
    # -- external anchor check (P5) -----------------------------------------
    dg_anchor_channels: tuple = (154, 1446)  # MA channels reported for FLUX by Gan et al.
    allow_missing_dg_anchor: bool = False    # only set True after debugging by hand
    # -- logging ------------------------------------------------------------
    secondary_blocks: tuple = (6, 11, 18, 22, 30, 45)
    # 6 early control, 11 and 22 used in prior work, 18 band onset,
    # 30 mid-band, 45 post-band control
    k_list: tuple = (1, 5, 10)
    top_save: int = 20
    save_full_tensor: bool = True         # fp16 [4096,3072] at (l*, t*) per scenario (~24 MB)
    # -- pilot & smoke ---------------------------------------------------------
    pilot_pairs: tuple = (("p00", 42), ("p01", 0), ("p02", 1234),
                          ("p03", 100), ("p04", 2024))
    smoke_steps: int = 4
    recurring_set_threshold: float = 0.9  # G_k = {d : f_k(d) >= this}

CFG = ExperimentConfig()
N_IMG = (CFG.height // 8 // 2) * (CFG.width // 8 // 2)   # VAE /8, patchify 2x2
assert N_IMG == 4096, f"unexpected token count {N_IMG}; resolution changed?"

CONFIG_HASH = hashlib.sha256(
    json.dumps(asdict(CFG), sort_keys=True, default=str).encode()).hexdigest()[:12]

print(f"[config] {CFG.n_prompts} prompts x {len(CFG.seeds)} seeds = "
      f"{CFG.n_prompts * len(CFG.seeds)} scenarios | {CFG.num_steps} steps @ "
      f"{CFG.height}x{CFG.width} ({N_IMG} image tokens, D={CFG.d_model})")
print(f"[config] hash: {CONFIG_HASH}")
for k, v in asdict(CFG).items():
    print(f"    {k:26s} = {v}")


### LIB A. Core utilities

Small helpers used throughout. They cover Jaccard, top-k and rank functions,
the primary score and sign consistency, atomic writes (write a temp file,
then rename), scenario paths, the completeness check that lets a run resume,
and the run-log columns.


In [ ]:
# =====================================================================
# LIB A. Core utilities
#
# numpy and the standard library only.
# =====================================================================
import os, json, csv, hashlib, datetime
import numpy as np

STAT_NAMES = ("signed_mean", "mean_abs", "max_abs", "p99_abs", "frac_pos")

# ------------------------------------------------------------ set math
def jaccard(a, b):
    """Jaccard overlap of two iterables of channel ids."""
    A, B = set(int(x) for x in a), set(int(x) for x in b)
    if not A and not B:
        return 1.0
    return len(A & B) / len(A | B)

def topk_ids(score_vec, k):
    """Indices of the k largest scores, largest first. Ties go to the lower id."""
    s = np.asarray(score_vec, dtype=np.float64)
    # stable sort, so tied ids stay in ascending order
    order = np.argsort(-s, kind="stable")
    return order[:k].astype(np.int64)

def rank_of(score_vec, channel):
    """Competition rank (1 = largest) of `channel` within score_vec."""
    s = np.asarray(score_vec, dtype=np.float64)
    return 1 + int((s > s[int(channel)]).sum())

def primary_score(stats_at_t, primary_stat):
    """Primary score [D] (float64) at one block and step.

    stats_at_t maps a stat name to its [D] vector. primary_stat is
    'abs_signed_mean' or 'mean_abs'."""
    if primary_stat == "abs_signed_mean":
        return np.abs(np.asarray(stats_at_t["signed_mean"], dtype=np.float64))
    if primary_stat == "mean_abs":
        return np.asarray(stats_at_t["mean_abs"], dtype=np.float64)
    raise ValueError(f"unknown primary_stat {primary_stat!r}")

def sign_consistency(frac_pos):
    """1.0 if the channel keeps one sign on every token, 0.5 if fully mixed."""
    f = np.asarray(frac_pos, dtype=np.float64)
    return np.maximum(f, 1.0 - f)

# ------------------------------------------------------------ atomic IO
def save_json_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

def save_npz_atomic(path, **arrays):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)

def save_npy_atomic(path, arr):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)

def save_png_atomic(pil_image, path):
    tmp = path + ".tmp.png"
    pil_image.save(tmp, format="PNG")
    os.replace(tmp, path)

def meta_to_npz_field(meta_dict):
    return np.array(json.dumps(meta_dict, default=str))

def meta_from_npz(z):
    return json.loads(str(z["meta_json"].item()))

# --------------------------------------------------- scenario bookkeeping
def scenario_id(pid, seed):
    return f"{pid}_s{seed}"

def scenario_paths(out_dir, sid, save_tensor, lstar=None, tstar=None):
    p = {"npz": os.path.join(out_dir, f"{sid}.npz"),
         "png": os.path.join(out_dir, f"{sid}.png"),
         "tensor": None}
    if save_tensor:
        assert lstar is not None and tstar is not None, \
            "tensor capture requested but (l*, t*) not set"
        p["tensor"] = os.path.join(out_dir, f"{sid}__Xfull_l{lstar:02d}_t{tstar:02d}.npy")
    return p

def is_scenario_complete(paths, need_tensor, expect=None):
    """True if the npz and png (and the tensor, when needed) exist and the npz
    opens with the required keys. If `expect` holds meta fields such as lstar,
    tstar or primary_stat, the stored meta has to match them too, so a resume
    doesn't keep results made under an older pilot decision."""
    if not (os.path.exists(paths["npz"]) and os.path.exists(paths["png"])):
        return False
    if need_tensor and not (paths["tensor"] and os.path.exists(paths["tensor"])):
        return False
    try:
        with np.load(paths["npz"], allow_pickle=False) as z:
            if not (("meta_json" in z.files) and ("sigmas" in z.files)):
                return False
            if expect:
                m = meta_from_npz(z)
                for k, v in expect.items():
                    if m.get(k) != v:
                        return False
            return True
    except Exception:
        return False

# ---------------------------------------------------------- run log CSV
RUN_LOG_FIELDS = [
    "timestamp", "tag", "sid", "prompt_id", "seed", "status", "wall_s",
    "peak_vram_gib", "fires_ok", "nan_flag", "hook_errors", "tensor_captured",
    "top1_channel", "top1_score", "top1_over_median", "top1_sign_consistency",
    "jacc_primary_vs_meanabs_top10", "precision_mode",
]

def append_run_log(csv_path, row_dict):
    new = not os.path.exists(csv_path)
    row = {k: row_dict.get(k, "") for k in RUN_LOG_FIELDS}
    row["timestamp"] = row.get("timestamp") or datetime.datetime.now().isoformat(timespec="seconds")
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RUN_LOG_FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)

def append_error_log(path, sid, err_text):
    with open(path, "a") as f:
        f.write(f"[{datetime.datetime.now().isoformat(timespec='seconds')}] {sid}: {err_text}\n\n")

def sha256_of_text(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

print("[lib-core] loaded:", ", ".join(
    ["jaccard", "topk_ids", "rank_of", "primary_score", "sign_consistency",
     "atomic IO", "scenario paths", "run log"]))


### LIB F. Figure style

`save_fig` doesn't add a title. The figure-finishing routine of the
intervention cells, `save_paper_fig`, writes a titled and an untitled copy of
every figure, and caption drafts go to `.caption.txt` files next to them for
use in LaTeX. Categorical colors use the colorblind-safe Okabe-Ito palette.
Magnitude heatmaps use sequential colormaps with a log norm, and the diverging
colormap is only used for signed data. Each figure is saved as a vector PDF
and a 600 dpi PNG, with TrueType (Type 42) fonts sized for a single column
(3.35 in) or the full page width (7 in).

In [ ]:
# =====================================================================
# LIB F. Figure style
#
# save_fig doesn't add a title. save_paper_fig, defined with the
# intervention config, writes a titled and an untitled copy of each figure.
# Caption drafts go to <stem>.caption.txt for LaTeX.
# Categorical colors use the Okabe-Ito palette. Magnitude heatmaps use sequential
# colormaps with a log norm, and signed data uses a dark-centered diverging
# colormap. Each figure is saved as a vector PDF plus a 600 dpi PNG, with
# TrueType (Type 42) fonts sized for a column or the full page.
#
# For color coding, each channel is put in a category by comparing a
# per-channel magnitude statistic with the median channel. This follows the
# massive-activations habit of calling a value an outlier when it is orders
# of magnitude above the typical one.
#   massive   statistic >= MASSIVE_MULT x median   (default 100x)
#   high      statistic >= HIGH_MULT x median      (default 10x)
#   typical   everything else
# Captions that use these colors state the multipliers.
# =====================================================================
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm

FIG_COL, FIG_PAGE = 3.35, 7.00      # inches, single column and full width
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00",
             "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
_BASE_PT = 8.0

def use_paper_style():
    mpl.rcParams.update({
        "figure.constrained_layout.use": True,
        "figure.dpi": 110,
        "pdf.fonttype": 42, "ps.fonttype": 42,
        "font.size": _BASE_PT,
        "axes.titlesize": _BASE_PT, "axes.labelsize": _BASE_PT,
        "xtick.labelsize": _BASE_PT - 1, "ytick.labelsize": _BASE_PT - 1,
        "legend.fontsize": _BASE_PT - 1, "legend.frameon": False,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.major.size": 2.5, "ytick.major.size": 2.5,
        "lines.linewidth": 1.0,
        "axes.prop_cycle": mpl.cycler(color=OKABE_ITO),
    })

use_paper_style()

MODEL_LABEL = {
    "black-forest-labs/FLUX.1-dev": "FLUX.1-dev",
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS": "PixArt-Sigma",
    "stabilityai/stable-diffusion-3-medium-diffusers": "SD3-Medium",
}.get(CFG.model_id, CFG.model_id)

BLOCK_AXIS_LABEL = {
    "FLUX.1-dev": "block index (0 to 18 dual, 19 to 56 single)",
}.get(MODEL_LABEL, "block index")

PRIMARY_TEX = {"mean_abs": r"mean $|x|$",
               "abs_signed_mean": r"$|\mathrm{mean}\,x|$"}
PRIMARY_WORDS = {"mean_abs": "the per-channel mean absolute activation",
                 "abs_signed_mean": "the absolute per-channel signed mean"}

# ---------------- channel categorization (high / massive color coding)
HIGH_MULT, MASSIVE_MULT = 10.0, 100.0
# profile figures on a white background use warm accent colors
CAT_COLORS = {0: "#6A51A3", 1: "#E69F00", 2: "#D55E00"}
# 3D bar landscape uses a blue floor and purples for the outlier tiers
BAR3D_COLORS = {0: "#6BAED6", 1: "#9E8FD0", 2: "#5C2D91"}
CAT_NAMES = {0: "typical",
             1: f"high (>= {HIGH_MULT:g}x median)",
             2: f"massive (>= {MASSIVE_MULT:g}x median)"}
CAT_CAPTION = (f"Channels are categorized relative to the median channel of "
               f"the same statistic: massive is at least {MASSIVE_MULT:g} "
               f"times the median, high is at least {HIGH_MULT:g} times the "
               f"median.")

def classify_channels(vec, high_mult=None, massive_mult=None):
    """[D] magnitude statistic -> (categories [D] int8, median).

    2 is massive, 1 high, 0 typical. Thresholds are multiples of the median
    channel and default to HIGH_MULT and MASSIVE_MULT."""
    v = np.asarray(vec, np.float64)
    med = max(float(np.median(v)), 1e-12)
    hm = HIGH_MULT if high_mult is None else float(high_mult)
    mm = MASSIVE_MULT if massive_mult is None else float(massive_mult)
    cat = np.zeros(v.size, np.int8)
    cat[v >= hm * med] = 1
    cat[v >= mm * med] = 2
    return cat, med

def _make_signed_cmap():
    """Black-centered diverging colormap. The positive half is magma, as in
    the magnitude heatmaps, and the negative half is a matching ramp from
    black through blue and cyan to pale cyan, so all landscapes look alike."""
    from matplotlib.colors import ListedColormap, LinearSegmentedColormap
    pos = plt.get_cmap("magma")(np.linspace(0.0, 1.0, 256))
    neg_ramp = LinearSegmentedColormap.from_list(
        "negside", ["#000004", "#1D1147", "#1F4287", "#2D7DD2",
                    "#40C9E0", "#B5F2F5"])
    neg = neg_ramp(np.linspace(0.0, 1.0, 256))[::-1]
    return ListedColormap(np.vstack([neg, pos]), name="signed_magma")

SIGNED_CMAP = _make_signed_cmap()

def fig_stem(name):
    return os.path.join(PATHS["figures"], name)

def save_fig(fig, stem, caption=None, show=True, tight=True):
    """Write <stem>.pdf, <stem>.png and, with a caption, <stem>.caption.txt
    atomically.

    Use tight=False for 3D axes. The tight bounding box under-measures 3D
    axis labels and crops them, and their placement drifts a little with
    the raster dpi, so 3D figures skip the tight box and set their margins
    with ax.set_position instead."""
    for ext, dpi in (("pdf", None), ("png", 600)):
        tmp = f"{stem}.tmp.{ext}"
        fig.savefig(tmp, bbox_inches="tight" if tight else None, dpi=dpi)
        os.replace(tmp, f"{stem}.{ext}")
    if caption:
        tmp = f"{stem}.caption.txt.tmp"
        with open(tmp, "w") as f:
            f.write(caption.strip() + "\n")
        os.replace(tmp, f"{stem}.caption.txt")
    print(f"[fig] saved {os.path.basename(stem)}"
          + (" (.pdf/.png/.caption.txt)" if caption else " (.pdf/.png)"))
    if show:
        plt.show()

# Color floor percentile for the magnitude heatmaps. At 1.0 the whole line
# texture of the landscape stays visible. Raise it toward 90 to black out
# the weaker lines and show only the strongest channels.
HEAT_FLOOR_PCTL = 1.0

def _cluster_sides(xs_sorted, span, sep_frac=0.04):
    """Pick a label side for each sorted point. Points closer than sep_frac
    of the axis span form a cluster. A lone point is labeled centered
    above, a pair gets left and right, and a larger cluster gets left, then
    center for the middle points, then right."""
    sep = sep_frac * float(span)
    sides, cluster = [], [0]

    def _flush(c):
        if len(c) == 1:
            sides.append("center")
        elif len(c) == 2:
            sides.extend(["left", "right"])
        else:
            sides.append("left")
            sides.extend(["center"] * (len(c) - 2))
            sides.append("right")

    for i in range(1, len(xs_sorted)):
        if xs_sorted[i] - xs_sorted[i - 1] <= sep:
            cluster.append(i)
        else:
            _flush(cluster)
            cluster = [i]
    _flush(cluster)
    return sides

def annotate_points(ax, xs, ys, labels):
    """Channel-id labels at their points. A lone channel is labeled right
    above its point. Two neighboring channels are labeled to the left and
    right of their points, so the pair stays readable without extra marks."""
    xs = np.asarray(xs, dtype=np.float64)
    ys = np.asarray(ys, dtype=np.float64)
    order = np.argsort(xs)
    xmin, xmax = ax.get_xlim()
    sides = _cluster_sides([float(xs[k]) for k in order], xmax - xmin)
    off = {"left": (-5, 1), "center": (0, 5), "right": (5, 1)}
    ha = {"left": "right", "center": "center", "right": "left"}
    for k, side in zip(order, sides):
        ax.annotate(str(labels[k]), (xs[k], ys[k]),
                    xytext=off[side], textcoords="offset points",
                    ha=ha[side], va="bottom", fontsize=_BASE_PT - 1.5)

def label_channels_top(ax, positions, labels, color="0.15"):
    """Channel-id labels in one row above the axes. Labels that would overlap
    are pushed sideways just enough to clear each other, using widths
    estimated from the font size and axes width. A thin leader line ties
    each label to its exact channel position."""
    if len(positions) == 0:
        return
    xs = np.asarray(positions, dtype=np.float64)
    order = np.argsort(xs)
    xs = xs[order]
    labs = [str(int(np.asarray(labels)[k])) for k in order]
    xmin, xmax = sorted(ax.get_xlim())
    fs = _BASE_PT - 3
    ax_pt = max(ax.figure.get_figwidth() * ax.get_position().width * 72.0,
                1.0)
    upp = (xmax - xmin) / ax_pt                    # data units per point
    wid = [(0.62 * fs * len(t) + 3.0) * upp for t in labs]
    pos = list(xs)
    for _ in range(200):                           # 1D dodge until no labels overlap
        moved = False
        for i in range(1, len(pos)):
            need = 0.5 * (wid[i - 1] + wid[i])
            if pos[i] - pos[i - 1] < need:
                mid = 0.5 * (pos[i] + pos[i - 1])
                pos[i - 1] = mid - 0.5 * need
                pos[i] = mid + 0.5 * need
                moved = True
        for i in range(len(pos)):                  # stay inside the axes
            pos[i] = min(max(pos[i], xmin + 0.5 * wid[i]),
                         xmax - 0.5 * wid[i])
        if not moved:
            break
    for x0, xl, t in zip(xs, pos, labs):
        ax.annotate(t, xy=(x0, 1.0), xycoords=("data", "axes fraction"),
                    xytext=((xl - x0) / upp, 8.0),
                    textcoords="offset points",
                    ha="center", va="bottom", fontsize=fs, color=color,
                    annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", lw=0.5, color=color,
                                    shrinkA=1.0, shrinkB=0.0))

def legend_above(ax, ncol=2, title=None):
    """Legend in a strip above the axes, never inside the data area."""
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.02), ncol=ncol,
                  borderaxespad=0.0, frameon=False, columnspacing=1.0,
                  handletextpad=0.4, title=title,
                  title_fontsize=_BASE_PT - 1)

def category_profile(ax, vec, cats, top_ids=None, label_top=True):
    """Log-scale per-channel profile with category color coding."""
    D = len(vec)
    ax.semilogy(np.arange(D), vec, lw=0.4, color="0.75", zorder=1)
    for c in (1, 2):
        idx = np.where(cats == c)[0]
        if len(idx):
            ax.scatter(idx, np.asarray(vec)[idx], s=10 if c == 2 else 5,
                       color=CAT_COLORS[c], zorder=3, label=CAT_NAMES[c],
                       linewidths=0)
    ax.set_xlim(0, D)
    if top_ids is not None and label_top:
        annotate_points(ax, list(top_ids),
                        [float(np.asarray(vec)[c]) for c in top_ids],
                        [int(c) for c in top_ids])

print(f"[lib-fig] paper style active | model label: {MODEL_LABEL} | "
      f"signed cmap: {SIGNED_CMAP.name}")


## S2. Prompt manifest and scenario grid
Prompts p00 to p02 are three fixed bridge prompts, used as written. Prompts
p03 to p19 come from the DiffusionDB metadata table (text and NSFW scores
only, no images). We keep ASCII prompts of 5 to 25 words with both NSFW
scores below 0.2, drop anything on a small blocklist or with more than four
comma-separated parts, and remove duplicates after normalizing case and
punctuation. The survivors are sampled round-robin over eight content
buckets so the set isn't dominated by one kind of scene.

If the download fails, a curated fallback list is used instead. In both cases
the list is written once to `prompts.json` with a sha-256 digest, and later
sessions reload that file instead of sampling again.

In [ ]:
# =====================================================================
# S2. Prompt manifest and scenario grid
# =====================================================================
# p00 to p02 are the three fixed bridge prompts. p03 to p19 are filtered
# from poloclub/diffusiondb metadata.parquet and sampled over 8 content
# buckets, with a curated fallback list if that fails. The list is written
# to prompts.json on the first build and reloaded, not resampled, after that.
import random

TABLE6_PROMPTS = [
    "a cat holding hello world sign",
    "an astronaut in the moon holding the USA flag",
    "a protestor in a city square holding a 'change' sign",
]

FALLBACK_PROMPTS = [
    "a close-up portrait of an elderly fisherman with a weathered face, dramatic lighting",
    "a red fox standing in a snowy birch forest at dawn",
    "a wide mountain valley with a winding river under storm clouds",
    "a cozy cluttered artist studio with paint tubes and canvases by a window",
    "a vintage brass pocket watch on a wooden desk, macro photograph",
    "a rusty pickup truck parked beside a desert gas station at sunset",
    "a neon sign that says 'open all night' above a diner entrance",
    "an isometric illustration of a tiny floating island village, pastel colors",
    "a chef plating dessert in a busy restaurant kitchen",
    "a lighthouse on a cliff during a thunderstorm, long exposure",
    "a black cat sleeping on a stack of old books",
    "an astronaut greenhouse on mars with rows of green plants",
    "a watercolor painting of a rainy tokyo street at night",
    "a medieval knight's helmet on a stone pedestal in a museum",
    "a hot air balloon festival over lavender fields",
    "a robot barista serving coffee in a futuristic cafe",
    "an oil painting of a stormy sea with a small sailboat",
]

BUCKET_KEYWORDS = {
    "portrait":  ("portrait", "face", " man ", " woman ", " girl ", " boy ", "person"),
    "animal":    ("cat", "dog", "fox", "bird", "horse", "wolf", "tiger", "lion", "deer", "animal"),
    "landscape": ("mountain", "forest", "valley", "landscape", "river", "desert",
                  "beach", "sunset", "field", "waterfall"),
    "indoor":    ("room", "kitchen", "interior", "office", "studio", "library", "bedroom"),
    "object":    ("watch", "bottle", "chair", "table", "lamp", "book", "sword",
                  "cup", "clock", "vase"),
    "vehicle":   ("car", "truck", "ship", "train", "plane", "motorcycle", "boat", "bus"),
    "text_in_image": ("sign", " text", "poster", "billboard", "label", "graffiti"),
    "stylized":  ("watercolor", "oil painting", "isometric", "pixel art",
                  "illustration", "anime", "sketch", "low poly"),
}

_BLOCKLIST_RE = re.compile(
    r"\b(nsfw|nude|naked|topless|porn|sexy|erotic|hentai|lingerie|gore|blood|"
    r"beheading|corpse|mutilat\w*|loli|shota)\b", re.IGNORECASE)

def _normalize_prompt(p):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(p).lower())).strip()

def _passes_filters(p, p_nsfw, i_nsfw):
    if not isinstance(p, str):
        return False
    p = p.strip()
    if not (p and p.isascii()):
        return False
    n_words = len(p.split())
    if not (5 <= n_words <= 25):
        return False
    try:
        if float(p_nsfw) >= 0.2 or float(i_nsfw) >= 0.2:
            return False
    except (TypeError, ValueError):
        return False
    if _BLOCKLIST_RE.search(p):
        return False
    if p.count(",") + 1 > 4:            # at most 4 comma-separated segments
        return False
    return True

def _bucket_of(p):
    q = f" {p.lower()} "
    for b, kws in BUCKET_KEYWORDS.items():
        if any(kw in q for kw in kws):
            return b
    return "other"

def select_diffusiondb_prompts(n_needed, rng_seed, exclude_norms):
    """Download the DiffusionDB metadata (prompt text and NSFW scores, no
    images), filter and dedupe it, then pick n_needed prompts round-robin
    over the content buckets."""
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq
    print("[prompts] downloading DiffusionDB metadata.parquet (~0.5 GB, one time) ...")
    fp = hf_hub_download("poloclub/diffusiondb", "metadata.parquet", repo_type="dataset")
    tbl = pq.read_table(fp, columns=["prompt", "prompt_nsfw", "image_nsfw"])
    df = tbl.to_pandas()
    print(f"[prompts] {len(df):,} raw rows")
    df = df[df["prompt"].map(lambda p: _passes_filters(p, 0.0, 0.0))]  # cheap text filters first
    mask = [( _passes_filters(p, pn, im)) for p, pn, im in
            zip(df["prompt"], df["prompt_nsfw"], df["image_nsfw"])]
    df = df[np.asarray(mask, dtype=bool)]
    print(f"[prompts] {len(df):,} rows after filters")
    seen, rows = set(exclude_norms), []
    for p in df["prompt"]:
        k = _normalize_prompt(p)
        if k and k not in seen:
            seen.add(k)
            rows.append(p.strip())
    buckets = {b: [] for b in list(BUCKET_KEYWORDS) + ["other"]}
    for p in rows:
        buckets[_bucket_of(p)].append(p)
    rng = random.Random(rng_seed)
    for b in buckets:
        rng.shuffle(buckets[b])
    order = list(BUCKET_KEYWORDS) + ["other"]
    picked, provenance = [], []
    while len(picked) < n_needed:
        progressed = False
        for b in order:
            if buckets[b] and len(picked) < n_needed:
                p = buckets[b].pop()
                picked.append(p)
                provenance.append(b)
                progressed = True
        if not progressed:
            raise RuntimeError("DiffusionDB pool exhausted before filling the manifest")
    return picked, provenance

def build_or_load_prompts(cfg, meta_dir):
    """Return ({pid: text}, manifest dict). The manifest is fixed after the
    first build."""
    path = os.path.join(meta_dir, "prompts.json")
    if os.path.exists(path):
        with open(path) as f:
            man = json.load(f)
        prompts = {e["pid"]: e["text"] for e in man["prompts"]}
        assert len(prompts) == cfg.n_prompts, \
            f"frozen prompts.json has {len(prompts)} prompts, config expects {cfg.n_prompts}"
        got = sha256_of_text("\n".join(prompts[f"p{i:02d}"] for i in range(cfg.n_prompts)))
        assert got == man["sha256"], "prompts.json content/sha mismatch -- file corrupted?"
        print(f"[prompts] loaded FROZEN manifest ({man['source']}, sha {man['sha256']})")
        return prompts, man
    n_extra = cfg.n_prompts - len(TABLE6_PROMPTS)
    exclude = {_normalize_prompt(p) for p in TABLE6_PROMPTS}
    try:
        extra, prov = select_diffusiondb_prompts(n_extra, cfg.prompt_source_seed, exclude)
        source = "table6+diffusiondb"
    except Exception as e:
        print(f"[prompts][warn] DiffusionDB path failed ({type(e).__name__}: {e})")
        print("[prompts] using curated fallback list (source recorded in manifest).")
        extra, prov = FALLBACK_PROMPTS[:n_extra], ["curated"] * n_extra
        source = "table6+curated_fallback"
    texts = TABLE6_PROMPTS + extra
    assert len(texts) == cfg.n_prompts, \
        f"prompt build produced {len(texts)} prompts, config expects {cfg.n_prompts}"
    provs = ["table6_bridge"] * len(TABLE6_PROMPTS) + prov
    entries = [{"pid": f"p{i:02d}", "text": t, "bucket": b}
               for i, (t, b) in enumerate(zip(texts, provs))]
    man = {"source": source, "rng_seed": cfg.prompt_source_seed,
           "built": datetime.datetime.now().isoformat(timespec="seconds"),
           "sha256": sha256_of_text("\n".join(texts)), "prompts": entries}
    save_json_atomic(man, path)
    print(f"[prompts] built & FROZE manifest -> {path} (source={source})")
    return {e["pid"]: e["text"] for e in entries}, man

PROMPTS, PROMPT_MANIFEST = build_or_load_prompts(CFG, PATHS["meta"])

# ---------------- scenario grid, prompt-major (all seeds of p00, then p01, ...)
SCENARIOS = [(scenario_id(pid, s), pid, s, PROMPTS[pid])
             for pid in sorted(PROMPTS) for s in CFG.seeds]
import pandas as pd
scen_df = pd.DataFrame(SCENARIOS, columns=["sid", "prompt_id", "seed", "prompt"])
scen_df.to_csv(os.path.join(PATHS["meta"], "scenarios.csv"), index=False)
print(f"[prompts] {len(SCENARIOS)} scenarios written to scenarios.csv")
for pid in sorted(PROMPTS):
    print(f"    {pid}: {PROMPTS[pid]}")


## S3. Model loading and smoke test
Loading happens in two phases. S3a loads only the text encoders (T5-XXL and
CLIP), encodes all 20 prompts once, saves the fp32 embeddings to Drive and
unloads the encoders. That frees about 10 GiB of VRAM, gives every seed of a
prompt exactly the same conditioning, and means a resumed session never
loads T5 at all. S3b then loads the generation pipeline without text
encoders.

S3c is a 4-step smoke run with hooks on all 57 blocks. It checks the block
count, that the image stream is found in both dual and single blocks, that
every hook fired the right number of times, that the statistics are finite,
and that the saved tensor and token norms load back from disk. Nothing else
should be run until it passes.

In [ ]:
# =====================================================================
# S3a. Prompt embedding cache
# =====================================================================
# Encode all 20 prompts once with T5-XXL and CLIP, save the fp32 embeddings
# to Drive and delete the text encoders. This frees about 10 GiB of VRAM,
# keeps the conditioning identical across seeds, and lets a resumed session
# skip T5.

def build_prompt_embed_cache(cfg, prompts, embed_dir, token, enc_dtype):
    embed_path = lambda pid: os.path.join(embed_dir, f"{pid}.npz")

    def _valid(pid):
        p = embed_path(pid)
        if not os.path.exists(p):
            return False
        try:
            with np.load(p, allow_pickle=False) as z:
                m = meta_from_npz(z)
                return (m["text_sha"] == sha256_of_text(prompts[pid])
                        and z["prompt_embeds"].shape[1] == cfg.max_seq_len)
        except Exception:
            return False

    missing = [pid for pid in sorted(prompts) if not _valid(pid)]
    if not missing:
        print(f"[embeds] all {len(prompts)} prompt embeddings cached & verified -- "
              f"text encoders will NOT be loaded this session.")
        return
    print(f"[embeds] encoding {len(missing)} prompt(s): {missing}")
    from diffusers import FluxPipeline
    pipe_text = FluxPipeline.from_pretrained(
        cfg.model_id, transformer=None, vae=None,
        torch_dtype=enc_dtype, token=token)
    pipe_text.to("cuda")
    with torch.inference_mode():
        for pid in missing:
            text = prompts[pid]
            pe, pp, _text_ids = pipe_text.encode_prompt(
                prompt=text, prompt_2=text, device="cuda",
                max_sequence_length=cfg.max_seq_len)
            assert pe.ndim == 3 and pe.shape[0] == 1 and pe.shape[1] == cfg.max_seq_len, \
                f"unexpected prompt_embeds shape {tuple(pe.shape)}"
            assert pp.ndim == 2 and pp.shape[0] == 1, \
                f"unexpected pooled shape {tuple(pp.shape)}"
            save_npz_atomic(
                embed_path(pid),
                prompt_embeds=pe.float().cpu().numpy(),
                pooled=pp.float().cpu().numpy(),
                meta_json=meta_to_npz_field({
                    "pid": pid, "text": text, "text_sha": sha256_of_text(text),
                    "enc_dtype": str(enc_dtype), "max_seq_len": cfg.max_seq_len,
                    "model_id": cfg.model_id}))
            print(f"    {pid}: encoded & saved  (T5 tokens={pe.shape[1]})")
    del pipe_text
    gc.collect(); torch.cuda.empty_cache()
    print("[embeds] text encoders unloaded; VRAM freed for generation.")

def load_embed_cache(prompts, embed_dir):
    """Return {pid: (prompt_embeds [1,512,4096], pooled [1,768])}, fp32 on CPU."""
    out = {}
    for pid in sorted(prompts):
        with np.load(os.path.join(embed_dir, f"{pid}.npz"), allow_pickle=False) as z:
            m = meta_from_npz(z)
            assert m["text_sha"] == sha256_of_text(prompts[pid]), \
                f"{pid}: cached embedding text mismatch -- delete {embed_dir} and re-run"
            out[pid] = (torch.from_numpy(z["prompt_embeds"].copy()),
                        torch.from_numpy(z["pooled"].copy()))
    return out

build_prompt_embed_cache(CFG, PROMPTS, PATHS["embeds"], HF_TOKEN, DTYPE)
EMBEDS = load_embed_cache(PROMPTS, PATHS["embeds"])
N_TXT = int(next(iter(EMBEDS.values()))[0].shape[1])
assert N_TXT == CFG.max_seq_len
print(f"[embeds] loaded {len(EMBEDS)} embeddings into RAM "
      f"(n_txt={N_TXT}, ~{len(EMBEDS)*8.4:.0f} MB)")


In [ ]:
# =====================================================================
# S3b. Generation pipeline
# =====================================================================
# The text encoders stay unloaded because the embeddings are cached.
#   bf16_full      everything on the GPU (A100 40/80 GB), the main path
#   bf16_offload   used automatically if bf16_full runs out of memory
#   quant_fp8      torchao float8 weight-only transformer (L4, Ada)
#   quant_nf4      bitsandbytes NF4 transformer, last resort (T4 etc.)
# The quantized modes change the activations we measure, so every saved
# file records `precision_mode`.

def load_generation_pipeline(cfg, mode, dtype, token):
    from diffusers import FluxPipeline
    no_text = dict(text_encoder=None, tokenizer=None,
                   text_encoder_2=None, tokenizer_2=None)

    def _bf16():
        pipe = FluxPipeline.from_pretrained(cfg.model_id, torch_dtype=dtype,
                                            token=token, **no_text)
        try:
            pipe.to("cuda")
            return pipe, "bf16_full"
        except torch.cuda.OutOfMemoryError:
            print("[pipe][warn] full-GPU load OOM'd -> enabling model CPU offload")
            gc.collect(); torch.cuda.empty_cache()
            pipe.enable_model_cpu_offload()
            return pipe, "bf16_offload"

    def _fp8():
        from diffusers import FluxTransformer2DModel, TorchAoConfig
        last_err = None
        for qname in ("float8wo", "float8_weight_only"):
            try:
                qc = TorchAoConfig(qname)
                tr = FluxTransformer2DModel.from_pretrained(
                    cfg.model_id, subfolder="transformer",
                    quantization_config=qc, torch_dtype=dtype, token=token)
                pipe = FluxPipeline.from_pretrained(
                    cfg.model_id, transformer=tr, torch_dtype=dtype,
                    token=token, **no_text)
                pipe.to("cuda")
                return pipe, "quant_fp8"
            except Exception as e:
                last_err = e
        raise RuntimeError(f"torchao fp8 path failed: {last_err}")

    def _nf4():
        from diffusers import FluxTransformer2DModel, BitsAndBytesConfig
        qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=dtype)
        tr = FluxTransformer2DModel.from_pretrained(
            cfg.model_id, subfolder="transformer",
            quantization_config=qc, torch_dtype=dtype, token=token)
        pipe = FluxPipeline.from_pretrained(
            cfg.model_id, transformer=tr, torch_dtype=dtype,
            token=token, **no_text)
        try:                       # bnb weights are already on the GPU, move the rest
            pipe.to("cuda")
        except Exception:
            pipe.vae.to("cuda")
        return pipe, "quant_nf4"

    if mode == "bf16_full":
        return _bf16()
    if mode == "quant_fp8":
        try:
            return _fp8()
        except Exception as e:
            print(f"[pipe][warn] fp8 unavailable ({e}); cascading to NF4")
            gc.collect(); torch.cuda.empty_cache()
            return _nf4()
    if mode == "quant_nf4":
        return _nf4()
    raise ValueError(f"unknown precision mode {mode!r}")

print(f"[pipe] loading FLUX.1-dev generation pipeline (mode={PRECISION_MODE}, "
      f"dtype={DTYPE}) -- first load downloads ~24 GB of transformer/VAE weights ...")
_t0 = time.time()
pipe, PRECISION_MODE = load_generation_pipeline(CFG, PRECISION_MODE, DTYPE, HF_TOKEN)
pipe.set_progress_bar_config(disable=True)
try:
    _pdev = next(pipe.transformer.parameters()).device
    _pdt = next(pipe.transformer.parameters()).dtype
except StopIteration:
    _pdev = _pdt = "n/a"
print(f"[pipe] ready in {time.time()-_t0:.0f}s | effective mode={PRECISION_MODE} | "
      f"transformer on {_pdev} ({_pdt}) | "
      f"VRAM allocated {torch.cuda.memory_allocated()/2**30:.1f} GiB")


### LIB B and LIB C. Measurement core
Forward hooks sit on the block outputs. The image stream is found by its
shape, not by its position in the output tuple. Dual blocks return two
streams and we take the one with sequence length 4096. Single blocks may
return the joined text-then-image sequence, and then the first 512 text
tokens are sliced off. Statistics are computed in fp32 on the GPU, so only
five D-dimensional vectors per block and step are copied to the CPU.

There is exactly one transformer forward per denoising step, so each
block's hook counter doubles as the step index. The counters are checked
when the recorder is finalized.

`run_scenario` is the one function used by the smoke test, the pilot and
the main run, so every phase measures the same way. It saves the image, then
the tensor, then the npz. The npz is written last and marks a scenario as
complete when resuming.

In [ ]:
# =====================================================================
# LIB B. Activation recorder
# =====================================================================
# We measure the residual-stream hidden state at each block output. Only
# the image stream is used, batch 1, cast to fp32 on the GPU before any
# reduction.
#
# The image stream is picked by shape, not by tuple position.
#   dual (MMDiT) blocks      separate streams, take the one with seq_len n_img
#   single blocks            joined [text | image] sequence, take [:, n_txt:, :]
#   tuple from single block  (newer diffusers) same rule as dual blocks
# Any other output raises inside the hook. Hook errors are logged and never
# stop the generation.
import math
import numpy as np
import torch

STAT_NAMES = ("signed_mean", "mean_abs", "max_abs", "p99_abs", "frac_pos")

def enumerate_flux_blocks(transformer):
    """List of (combined_index, module, kind). Dual blocks are 0..18, single
    blocks 19..56."""
    blocks = []
    for m in transformer.transformer_blocks:
        blocks.append((len(blocks), m, "dual"))
    for m in transformer.single_transformer_blocks:
        blocks.append((len(blocks), m, "single"))
    return blocks

def compute_channel_stats(X):
    """Per-channel statistics over tokens. X is a float32 [N_tokens, D]
    tensor on any device. Returns a dict of [D] float32 tensors.

    signed_mean    mean_n X[n,d], its absolute value is the default ranking statistic
    mean_abs       mean_n |X[n,d]|, used for the landscape
    max_abs        max_n |X[n,d]|, sensitive to single-token spikes
    p99_abs        99th percentile of |X[.,d]|, less spike-sensitive
    frac_pos       fraction of tokens with X[n,d] > 0, shows sign structure
    """
    N = X.shape[0]
    absX = X.abs()
    k99 = min(N, max(1, int(math.ceil(0.99 * N))))
    return {
        "signed_mean": X.mean(dim=0),
        "mean_abs":    absX.mean(dim=0),
        "max_abs":     absX.amax(dim=0),
        "p99_abs":     absX.kthvalue(k99, dim=0).values,
        "frac_pos":    (X > 0).to(torch.float32).mean(dim=0),
    }

class ActivationRecorder:
    def __init__(self, transformer, n_steps, n_img, n_txt, d_model, stat_blocks,
                 lstar=None, tstar=None, capture_full=False,
                 capture_token_norms=False):
        self.all_blocks = enumerate_flux_blocks(transformer)
        self.n_steps, self.n_img, self.n_txt, self.d_model = \
            int(n_steps), int(n_img), int(n_txt), int(d_model)
        self.stat_blocks = sorted({int(b) for b in stat_blocks})
        bad = [b for b in self.stat_blocks if not (0 <= b < len(self.all_blocks))]
        if bad:
            raise ValueError(f"stat_blocks out of range: {bad} "
                             f"(model has {len(self.all_blocks)} blocks)")
        self.lstar = None if lstar is None else int(lstar)
        self.tstar = None if tstar is None else int(tstar)
        self.capture_full = bool(capture_full)
        self.capture_token_norms = bool(capture_token_norms)
        if self.capture_full or self.capture_token_norms:
            assert self.lstar in self.stat_blocks, \
                "capture requested but lstar not among stat_blocks"
        self._handles = []
        self.reset()

    def reset(self):
        self.stats = {b: {s: np.zeros((self.n_steps, self.d_model), np.float32)
                          for s in STAT_NAMES} for b in self.stat_blocks}
        self.counters = {b: 0 for b in self.stat_blocks}
        self.overflow = {b: 0 for b in self.stat_blocks}
        self.token_norms = (np.zeros((self.n_steps, self.n_img), np.float16)
                            if (self.capture_token_norms and self.lstar is not None)
                            else None)
        self.full_tensor = None
        self.full_clamped = 0
        self.errors = []

    # ------------------------------------------------ stream extraction
    def _extract_image_stream(self, output):
        if isinstance(output, (tuple, list)):
            cands = [o for o in output if torch.is_tensor(o) and o.dim() == 3]
            for o in cands:
                if o.shape[1] == self.n_img:
                    return o
            raise RuntimeError(
                "no image stream (seq_len=%d) in tuple output; got shapes %s"
                % (self.n_img, [tuple(o.shape) for o in cands]))
        if torch.is_tensor(output) and output.dim() == 3:
            L = output.shape[1]
            if L == self.n_img:
                return output
            if L == self.n_img + self.n_txt:      # FLUX joins as [text | image]
                return output[:, self.n_txt:, :]
            raise RuntimeError(
                f"unexpected sequence length {L} "
                f"(expected {self.n_img} or {self.n_txt + self.n_img})")
        raise RuntimeError(f"unexpected hook output type {type(output)}")

    # ------------------------------------------------------------ hooks
    def _make_hook(self, bidx):
        def hook(module, args, output):
            step = self.counters[bidx]
            self.counters[bidx] = step + 1
            if step >= self.n_steps:
                self.overflow[bidx] += 1
                return
            try:
                img = self._extract_image_stream(output)
                if img.shape[0] != 1:
                    raise RuntimeError(f"expected batch 1, got shape {tuple(img.shape)}")
                if img.shape[2] != self.d_model:
                    raise RuntimeError(f"hidden size {img.shape[2]} != d_model {self.d_model}")
                X = img[0].to(torch.float32)
                st = compute_channel_stats(X)
                packed = torch.stack([st[s] for s in STAT_NAMES], dim=0) \
                              .to("cpu", torch.float32).numpy()   # one device-to-host copy
                for si, s in enumerate(STAT_NAMES):
                    self.stats[bidx][s][step] = packed[si]
                if self.token_norms is not None and bidx == self.lstar:
                    self.token_norms[step] = \
                        X.norm(dim=1).to("cpu", torch.float16).numpy()
                if (self.capture_full and bidx == self.lstar
                        and self.tstar is not None and step == self.tstar):
                    n_over = int((X.abs() > 65504.0).sum().item())
                    if n_over:                      # would overflow fp16, so clamp and count
                        self.full_clamped = n_over
                    self.full_tensor = X.clamp(-65504.0, 65504.0) \
                                        .to("cpu", torch.float16).numpy()
            except Exception as e:   # log it, don't crash the generation
                self.errors.append(
                    f"block {bidx} step {step}: {type(e).__name__}: {e}")
        return hook

    def attach(self):
        assert not self._handles, "recorder already attached"
        hooked = set(self.stat_blocks)
        for i, mod, _kind in self.all_blocks:
            if i in hooked:
                self._handles.append(mod.register_forward_hook(self._make_hook(i)))
        return self

    def detach(self):
        for h in self._handles:
            h.remove()
        self._handles = []

    def finalize(self, expected_steps=None):
        exp = int(self.n_steps if expected_steps is None else expected_steps)
        fires_ok = (all(self.counters[b] == exp for b in self.stat_blocks)
                    and all(v == 0 for v in self.overflow.values()))
        nan_flag = False
        for b in self.stat_blocks:
            for s in STAT_NAMES:
                if not np.isfinite(self.stats[b][s][:exp]).all():
                    nan_flag = True
        return {"fires_ok": bool(fires_ok),
                "nan_flag": bool(nan_flag),
                "hook_errors": len(self.errors),
                "errors": list(self.errors),
                "counters": {int(b): int(c) for b, c in self.counters.items()},
                "tensor_captured": self.full_tensor is not None,
                "full_tensor_clamped": int(self.full_clamped)}

print(f"[lib-recorder] loaded (stats: {', '.join(STAT_NAMES)})")


In [ ]:
# =====================================================================
# LIB C. Scenario runner
# =====================================================================
# run_scenario does one generation with the recorder attached and saves the
# results atomically. The smoke test, the pilot (phase A) and the main run
# (phase B) all call it. Files are saved as png, then tensor, then npz. The
# npz comes last and marks the scenario as done, so a scenario that was cut
# off halfway is simply run again.
import gc, time, datetime
import numpy as np
import torch

def run_scenario(pipe, cfg, sid, pid, prompt_text, seed, embeds, out_dir,
                 stat_blocks, n_steps, n_img, pipe_dtype, device="cuda",
                 lstar=None, tstar=None, primary_stat=None,
                 capture_full=False, capture_token_norms=False,
                 tag="main", extra_meta=None, force=False):
    extra_meta = dict(extra_meta or {})
    need_tensor = bool(capture_full and cfg.save_full_tensor)
    paths = scenario_paths(out_dir, sid, need_tensor, lstar, tstar)
    expect = None
    if lstar is not None and tstar is not None and primary_stat is not None:
        expect = {"lstar": int(lstar), "tstar": int(tstar),
                  "primary_stat": primary_stat}
    if not force and is_scenario_complete(paths, need_tensor=need_tensor,
                                          expect=expect):
        return {"tag": tag, "sid": sid, "prompt_id": pid, "seed": seed,
                "status": "skipped",
                "precision_mode": extra_meta.get("precision_mode", "")}

    pe, pp = embeds[pid]
    n_txt = int(pe.shape[1])
    rec = ActivationRecorder(
        pipe.transformer, n_steps=n_steps, n_img=n_img, n_txt=n_txt,
        d_model=cfg.d_model, stat_blocks=stat_blocks,
        lstar=lstar, tstar=tstar,
        capture_full=need_tensor, capture_token_norms=capture_token_norms)

    torch.manual_seed(seed)                         # global RNG as well
    gen = torch.Generator("cpu").manual_seed(seed)  # CPU generator gives the same noise on any device
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    rec.attach()
    try:
        with torch.inference_mode():
            out = pipe(prompt_embeds=pe.to(device, pipe_dtype),
                       pooled_prompt_embeds=pp.to(device, pipe_dtype),
                       height=cfg.height, width=cfg.width,
                       num_inference_steps=n_steps,
                       guidance_scale=cfg.guidance_scale,
                       generator=gen, output_type="pil")
    finally:
        rec.detach()
    wall = time.time() - t0
    peak = (torch.cuda.max_memory_allocated() / 2**30
            if torch.cuda.is_available() else 0.0)
    image = out.images[0]
    fin = rec.finalize(expected_steps=n_steps)
    if fin["errors"]:
        append_error_log(ERROR_LOG_PATH, sid,
                         "hook errors:\n  " + "\n  ".join(fin["errors"][:20]))

    sig = (pipe.scheduler.sigmas.detach().float().cpu().numpy()
           if hasattr(pipe.scheduler, "sigmas") else np.zeros(0, np.float32))
    tsv = (pipe.scheduler.timesteps.detach().float().cpu().numpy()
           if hasattr(pipe.scheduler, "timesteps") else np.zeros(0, np.float32))

    arrays = {}
    for b in rec.stat_blocks:
        for s in STAT_NAMES:
            arrays[f"b{b:02d}__{s}"] = rec.stats[b][s]
    if rec.token_norms is not None:
        arrays["token_norms_lstar"] = rec.token_norms

    diag = {"tag": tag, "sid": sid, "prompt_id": pid, "seed": seed,
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "fires_ok": fin["fires_ok"], "nan_flag": fin["nan_flag"],
            "hook_errors": fin["hook_errors"],
            "tensor_captured": fin["tensor_captured"],
            "precision_mode": extra_meta.get("precision_mode", "")}

    # ---- per-scenario top-k under the primary statistic and under mean|x|, and their overlap
    if lstar is not None and tstar is not None and primary_stat is not None:
        stats_t = {s: rec.stats[lstar][s][tstar] for s in STAT_NAMES}
        sp = primary_score(stats_t, primary_stat)
        sa = np.asarray(stats_t["mean_abs"], dtype=np.float64)
        ids_p = topk_ids(sp, cfg.top_save)
        ids_a = topk_ids(sa, cfg.top_save)
        arrays["top20_ids"] = ids_p.astype(np.int32)
        arrays["top20_scores"] = sp[ids_p].astype(np.float32)
        arrays["top20_ids_mean_abs"] = ids_a.astype(np.int32)
        arrays["top20_scores_mean_abs"] = sa[ids_a].astype(np.float32)
        cons = sign_consistency(stats_t["frac_pos"])
        top1 = int(ids_p[0])
        med = max(float(np.median(sp)), 1e-12)
        diag.update(
            top1_channel=top1,
            top1_score=round(float(sp[top1]), 4),
            top1_over_median=round(float(sp[top1] / med), 1),
            top1_sign_consistency=round(float(cons[top1]), 4),
            jacc_primary_vs_meanabs_top10=round(jaccard(ids_p[:10], ids_a[:10]), 3))

    meta = {"sid": sid, "prompt_id": pid, "seed": int(seed), "prompt": prompt_text,
            "tag": tag, "model_id": cfg.model_id,
            "height": cfg.height, "width": cfg.width, "num_steps": int(n_steps),
            "guidance_scale": cfg.guidance_scale,
            "n_img": int(n_img), "n_txt": int(n_txt), "d_model": cfg.d_model,
            "stat_blocks": [int(b) for b in rec.stat_blocks],
            "lstar": lstar, "tstar": tstar, "primary_stat": primary_stat,
            "counters": fin["counters"], "fires_ok": fin["fires_ok"],
            "nan_flag": fin["nan_flag"], "hook_errors": fin["hook_errors"],
            "hook_error_msgs": fin["errors"][:10],
            "tensor_captured": fin["tensor_captured"],
            "full_tensor_clamped": fin["full_tensor_clamped"],
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            **extra_meta}
    arrays["sigmas"] = sig.astype(np.float32)
    arrays["timesteps"] = tsv.astype(np.float32)
    arrays["meta_json"] = meta_to_npz_field(meta)

    save_png_atomic(image, paths["png"])
    if need_tensor and rec.full_tensor is not None:
        save_npy_atomic(paths["tensor"], rec.full_tensor)
    save_npz_atomic(paths["npz"], **arrays)   # written last, marks completion

    ok = (fin["fires_ok"] and not fin["nan_flag"] and fin["hook_errors"] == 0
          and (fin["tensor_captured"] or not need_tensor))
    diag["status"] = "ok" if ok else "warn"
    return diag

print("[lib-run] run_scenario ready")


In [ ]:
# =====================================================================
# S3c. Smoke test
# =====================================================================
# One 4-step generation with hooks on all 57 blocks, about 15 to 60 s. It
# asserts that there are 19 dual and 38 single blocks, that the image stream
# (4096 tokens, D=3072) is found in both block types, that each hook fired
# exactly `smoke_steps` times without errors, that the stats are finite, and
# that the full tensor and token norms load back from Drive.
# It also checks, as a warning only, that the top mean|x| channel carries
# over the boundary from the last dual block (18) to the first single
# block (19).
print("=" * 70)
print("SMOKE TEST")
print("=" * 70)

_blocks = enumerate_flux_blocks(pipe.transformer)
_n_dual = sum(1 for _, _, k in _blocks if k == "dual")
assert len(_blocks) == CFG.n_blocks_expected and _n_dual == CFG.n_dual_expected, (
    f"block census mismatch: found {len(_blocks)} total / {_n_dual} dual, "
    f"expected {CFG.n_blocks_expected}/{CFG.n_dual_expected}. diffusers "
    f"{LIB_VERSIONS.get('diffusers')} may have restructured FLUX -- inspect "
    f"pipe.transformer before proceeding.")
print(f"[smoke] block census OK: {_n_dual} dual + {len(_blocks)-_n_dual} single")

SMOKE_SID = "smoke__p00_s0"
SMOKE_LSTAR, SMOKE_TSTAR = 18, CFG.smoke_steps - 1
for _f in glob.glob(os.path.join(PATHS["smoke"], "smoke__*")):
    os.remove(_f)                       # always a fresh smoke run

_diag = run_scenario(
    pipe, CFG, SMOKE_SID, "p00", PROMPTS["p00"], 0, EMBEDS, PATHS["smoke"],
    stat_blocks=list(range(len(_blocks))), n_steps=CFG.smoke_steps, n_img=N_IMG,
    pipe_dtype=DTYPE, device=DEVICE, lstar=SMOKE_LSTAR, tstar=SMOKE_TSTAR,
    primary_stat=CFG.primary_stat_default,
    capture_full=True, capture_token_norms=True, tag="smoke",
    extra_meta={"precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH},
    force=True)
print(f"[smoke] generation done in {_diag['wall_s']}s "
      f"(peak {_diag['peak_vram_gib']} GiB) status={_diag['status']}")
assert _diag["status"] == "ok", f"smoke diagnostics not clean: {_diag}"

_paths = scenario_paths(PATHS["smoke"], SMOKE_SID, True, SMOKE_LSTAR, SMOKE_TSTAR)
with np.load(_paths["npz"], allow_pickle=False) as _z:
    _meta = meta_from_npz(_z)
    assert _meta["n_txt"] == CFG.max_seq_len, f"n_txt={_meta['n_txt']}"
    assert _meta["n_img"] == N_IMG, f"n_img={_meta['n_img']}"
    assert all(int(c) == CFG.smoke_steps for c in _meta["counters"].values()), \
        f"hook fire counts wrong: {_meta['counters']}"
    _stat_keys = [k for k in _z.files if k.startswith("b") and "__" in k]
    assert len(_stat_keys) == len(_blocks) * len(STAT_NAMES), \
        f"{len(_stat_keys)} stat arrays, expected {len(_blocks)*len(STAT_NAMES)}"
    _tn = _z["token_norms_lstar"]
    assert _tn.shape == (CFG.smoke_steps, N_IMG) and _tn.dtype == np.float16
    assert _z["sigmas"].shape[0] in (0, CFG.smoke_steps + 1)
    for _k in _stat_keys:
        assert np.isfinite(_z[_k]).all(), f"non-finite values in {_k}"
    _v18 = _z["b18__mean_abs"][SMOKE_TSTAR].copy()
    _v19 = _z["b19__mean_abs"][SMOKE_TSTAR].copy()

_X = np.load(_paths["tensor"])
assert _X.shape == (N_IMG, CFG.d_model) and _X.dtype == np.float16, \
    f"tensor round-trip: shape {_X.shape} dtype {_X.dtype}"
_Xf = _X.astype(np.float32)
assert np.isfinite(_Xf).all() and float(np.abs(_Xf).max()) > 0.0
del _X, _Xf

_t18, _t19 = int(np.argmax(_v18)), int(np.argmax(_v19))
_top10_19 = set(topk_ids(_v19, 10).tolist())
if _t18 == _t19:
    print(f"[smoke] residual-stream continuity OK: top mean|x| channel {_t18} "
          f"identical across dual->single boundary (blocks 18->19)")
elif _t18 in _top10_19:
    print(f"[smoke][note] block-18 top-1 (ch {_t18}) is in block-19 top-10 "
          f"(top-1 there: ch {_t19}) -- acceptable")
else:
    print(f"[smoke][WARN] block-18 top-1 (ch {_t18}) absent from block-19 top-10 "
          f"({_t19} leads there). Usually the residual stream carries the MA "
          f"channel across the boundary; if the pilot's DG-anchor check (P5) "
          f"also fails, suspect the [text|image] concat order in single blocks.")

import matplotlib.pyplot as plt
from PIL import Image
plt.figure(figsize=(4, 4))
plt.imshow(Image.open(_paths["png"])); plt.axis("off")
plt.title(f"smoke image ({CFG.smoke_steps} steps -- low quality is expected)")
plt.show()
print("SMOKE TEST PASSED -- safe to start Phase A (pilot).")


## R1. Rescue experiment configuration

R1a holds the model-specific values, such as the expected channel
identities, the lesion block, the probe blocks and their roles, the
secondary intervention step and any extra conditions. R1b is shared. It
defines a frozen config dataclass whose hash goes into every output (a few
run-control fields such as the condition list and the intervention steps
are left out), the output folders, the prerequisite checks and the
condition grammar with readable labels. Every operation gets its own line
style. R1b also saves the depth and step settings as fractions of the
model's depth and step count, and defines the figure helpers, which place
the legend above the panels, save copies with and without a title, and
check every figure for overlapping text and clashing line styles.

In [ ]:
# =====================================================================
# R1a. Model-specific parameters (FLUX.1-dev)
#
# The expected channels come from the channel recurrence run. c* = 154
# carries v*, and 1446, the catching-up channel, is the massive-channel
# control. R2 recomputes both from the pilot data and stops on a mismatch
# unless allow_ch_mismatch is set.
#
# Depths, fixed before any rescue run:
#   lesion block b_L = 22     the analysis block l*, four blocks after the
#                             band onset. The register is fully formed there
#                             (about 20x the median norm) and its key is a
#                             top sink in most heads.
#   rescue block b_R = 23     the next residual state (gap 1)
#   primary probe = 24        first attention call that sees the rescued
#                             residual (b_R + 1)
#   maintained range end 39   band end, where the register dissolves
#
# PROBE_BLOCKS are where the full attention distributions are stored, one
# per role in PROBE_ROLES. The scalar routing readouts are taken at every
# block anyway.
# =====================================================================
MODEL_FAMILY = "flux"        # branches the pipeline call and batch handling
EXPECTED_CH1 = 154           # dominant channel c*, carries v*
EXPECTED_CH2 = 1446          # massive-channel control, the catching-up channel
REG_EXPECT_TOKEN = None      # FLUX register position varies across prompts and seeds
LAST_BLOCK = CFG.n_blocks_expected - 1   # 56
LESION_BLOCK_DEFAULT = 22    # = l*. R2 stops if the lesion block is before the pilot l*
EARLY_STEP = 10              # secondary intervention step (0-indexed, of 28)
PROBE_BLOCKS = (22, 23, 24, 26, 30, 35, 39, 45)
PROBE_ROLES = ("lesion_block", "lesion_readout", "first_post_rescue", "post_rescue_2",
               "mid_band", "overtake", "band_end", "post_band")
STATE_REF_BLOCKS_RULE = ("first_post_rescue", "band_end")   # full-state references, stored as fp16 by the baseline
N_HEADS_EXPECTED = 24
EXTRA_CONDITIONS = ()        # extra condition names, grammar in R1b

def trajectory_vector(out_tensor):
    """The part of the transformer output that drives the denoising step.
    FLUX has embedded guidance and batch 1, so it's the whole prediction."""
    return out_tensor[0].to(torch.float32)

print(f"[q8-params] family={MODEL_FAMILY} c*={EXPECTED_CH1} control ch2={EXPECTED_CH2} | "
      f"blocks 0..{LAST_BLOCK} | band {CFG.band} | lesion block {LESION_BLOCK_DEFAULT} | "
      f"probe blocks {PROBE_BLOCKS} | early step {EARLY_STEP} | heads {N_HEADS_EXPECTED}")


In [ ]:
# =====================================================================
# R1b. Rescue config, output folders, prerequisites, condition grammar
# and figure helpers
#
# RescueConfig is hashed into every output, except the fields listed in
# _HASH_EXCLUDE.
#
# Condition names (parsed by parse_condition)
#
#   baseline                          no intervention
#   sham                              writes the stored clean vector back in
#                                     the lesion window and the rescue range,
#                                     and the clean key and value inside the
#                                     attention of the rescue range. It must
#                                     match the baseline bit for bit, which
#                                     checks the hooks and that the stored
#                                     clean values are lossless.
#   <lesion>[_maint]                  lesion only
#   <lesion>[_maint]__<rescue>[opts]  lesion, then rescue
#
# Lesions hit the register in blocks b_L to b_R - 1, or up to the maintained
# end (the band end by default) with _maint.
#   cstar_zero          channel c* set to 0
#   vstar_destroy       v* component removed, norm kept
#   vstar_random_dir    direction replaced by a random direction orthogonal
#                       to v*, norm kept
#   reg_remove          register replaced by the mean ordinary token
#
# Residual rescues act at block b_R. Attention rescues act at every block
# from b_R to the maintained end.
#   vstar[_f<frac>]           projection onto v* set to frac x clean value
#   cstar                     channel c* set to its clean value
#   chan_c2, chan_ctrl        a control channel set to its clean value
#   norm                      clean norm, direction kept
#   full                      whole clean vector
#   key, value, key_value     the register's final key and/or value inside
#                             attention set to the clean one
#
# Options after the rescue
#   _maint        residual rescue at every block from b_R to the maintained end
#   _single       attention rescue at block b_R only
#   _at_random    the rescue is written at a random ordinary token
#   _wrong        attention rescues only, writes the random token's clean key
#                 and/or value at the register position
# _at_random and _wrong can't be combined with _maint, _single or a
# fraction, and a fraction only works with vstar.
# =====================================================================
import re, itertools, textwrap as _textwrap
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class RescueConfig:
    # -- where and when -----------------------------------------------------
    lesion_block: str = "model_default"   # "model_default" (from R1a), "lstar" or an int
    rescue_gap: int = 1                   # b_R = b_L + gap, lesion window [b_L, b_R - 1]
    maintain_end: str = "band_end"        # end of maintained windows, "band_end" or "last"
    intervention_steps: tuple = ("tstar",)   # ("tstar", "early") doubles the runs, part of the R2 decision
    # -- targets -------------------------------------------------------------
    n_targets: int = 1                    # 1 = the max-norm token, up to 8 high-norm tokens
    target_norm_mult: float = 3.0         # extra targets must be >= this x median norm
    ordinary_norm_mult: float = 2.0       # ordinary tokens (random and mean token) have norm < this x median
    random_token_seed: int = 7
    # -- conditions (baseline first, R1a adds EXTRA_CONDITIONS) --------------
    conditions: tuple = ("baseline", "sham",
                         "cstar_zero", "cstar_zero__vstar", "cstar_zero__norm", "cstar_zero__full",
                         "vstar_destroy", "vstar_destroy__key", "vstar_destroy__full",
                         "reg_remove", "reg_remove__cstar", "reg_remove__full") + tuple(EXTRA_CONDITIONS)
    scenario_scope: str = "full"          # "full" (100) | "prompts20" | "pilot" (5)
    # -- replay and storage --------------------------------------------------
    replay_prefix: bool = True            # replay the clean prefix from the baseline's cached outputs
    store_tn_later_steps: bool = True     # token norms at every block for steps after the intervention
    store_probe_rel: bool = True          # full incoming-attention distributions at the probe blocks
    store_state_ref: bool = True          # baseline stores the fp16 full state at the two reference blocks
    store_kv_conditions: bool = True      # keys/values at the tracked tokens in condition runs, for checking
    # -- readouts --------------------------------------------------------------
    probe_chunk: int = 256
    geom_cos_thr: float = 0.9
    # -- v* ---------------------------------------------------------------------
    vstar_source: str = "q6_frozen_or_refit"
    vstar_norm_mult: float = 3.0
    vstar_max_per_scenario: int = 8
    vstar_min_scenarios: int = 20
    # -- channels ----------------------------------------------------------------
    ctrl_exclude_mult: float = 10.0       # ordinary control channel must be below this x median channel at the lesion block
    expected_ch1: int = EXPECTED_CH1
    expected_ch2: int = EXPECTED_CH2
    allow_ch_mismatch: bool = False
    force_redecide: bool = False
    force_refit_vstar: bool = False
    # -- analysis, fixed in advance ------------------------------------------------
    rf_gap_floor_log10: float = 0.3       # rescue fractions need a clean-lesion gap of at least 0.3 log10 (x2)
    rf_equiv_margin: float = 0.2          # a control has "no effect" if the CI of the median RF is inside +/- this
    rf_full_margin: float = 0.2           # "full rescue" if the CI of the median RF is inside 1 +/- this
    share_equiv_log10: float = 0.15       # equivalence margin for log10 share differences
    n_boot: int = 10000

Q8 = RescueConfig()
assert Q8.conditions[0] == "baseline", "baseline must be the first condition"
assert len(set(Q8.conditions)) == len(Q8.conditions), "duplicate condition names"
_HASH_EXCLUDE = {"conditions", "scenario_scope", "intervention_steps", "allow_ch_mismatch",
                 "force_redecide", "force_refit_vstar", "store_tn_later_steps", "store_probe_rel",
                 "store_state_ref", "store_kv_conditions", "n_boot"}
Q8_HASH = hashlib.sha256(json.dumps(
    {k: v for k, v in asdict(Q8).items() if k not in _HASH_EXCLUDE},
    sort_keys=True, default=str).encode()).hexdigest()[:12]

# ------------------------------------------------- output tree (under BASE)
Q8_PATHS = {
    "runs":     os.path.join(BASE, "q8_rescue"),
    "smoke":    os.path.join(BASE, "q8_rescue_smoke"),
    "figs":     os.path.join(BASE, "figures_q8_rescue"),
    "analysis": os.path.join(BASE, "analysis_q8_rescue"),
}
for _p in Q8_PATHS.values():
    os.makedirs(_p, exist_ok=True)
Q8_DEC_PATH = os.path.join(PATHS["meta"], "q8_decision.json")
Q8_VSTAR_PATH = os.path.join(PATHS["meta"], "vstar_q8.npz")
Q6_DEC_PATH = os.path.join(PATHS["meta"], "intervention_decision_v3.json")
Q6_VSTAR_PATH = os.path.join(PATHS["meta"], "vstar_v3.npz")

def q8_fig_stem(name):
    return os.path.join(Q8_PATHS["figs"], name)

# ------------------------------------------------- prerequisites from the channel recurrence run
_dec_path = os.path.join(PATHS["meta"], "pilot_decision.json")
if not os.path.exists(_dec_path):
    raise RuntimeError("pilot_decision.json missing: complete the v4 recurrence notebook's "
                       "Phase A (S4) and decision cell (S5) on this Drive first.")
with open(_dec_path) as _f:
    PILOT_DECISION = json.load(_f)
L_STAR = int(PILOT_DECISION["l_star"])
T_STAR = int(PILOT_DECISION["t_star"])
PRIMARY_STAT = PILOT_DECISION["primary_stat"]
PILOT_SIDS = [f"pilot__{scenario_id(p, s)}" for p, s in CFG.pilot_pairs]
for _sid in PILOT_SIDS:
    if not os.path.exists(os.path.join(PATHS["pilot"], f"{_sid}.npz")):
        raise RuntimeError(f"pilot capture missing: {_sid}.npz (run the v4 notebook's Phase A first)")

N_TXT_IN_SEQ = int(CFG.max_seq_len) if MODEL_FAMILY == "flux" else int(CFG.n_txt_in_seq)
EXP_BATCH = 1 if MODEL_FAMILY == "flux" else (
    2 if (CFG.guidance_mode == "cfg_batch2" and CFG.guidance_scale > 1.0) else 1)
SELECT_ROW = 0 if MODEL_FAMILY == "flux" else int(CFG.cond_row)

# ------------------------------------------------- blocks and steps
_BAND_LO, _BAND_HI = int(CFG.band[0]), int(CFG.band[1])
if Q8.lesion_block == "model_default":
    B_LESION = int(LESION_BLOCK_DEFAULT)
elif Q8.lesion_block == "lstar":
    B_LESION = L_STAR
else:
    B_LESION = int(Q8.lesion_block)
B_RESCUE = B_LESION + int(Q8.rescue_gap)
B_MAINT_END = _BAND_HI if Q8.maintain_end == "band_end" else LAST_BLOCK
assert 1 <= B_LESION < B_RESCUE <= B_MAINT_END <= LAST_BLOCK, (B_LESION, B_RESCUE, B_MAINT_END)
assert B_RESCUE + 1 <= LAST_BLOCK, "need at least one block after the rescue block"
assert len(PROBE_ROLES) == len(PROBE_BLOCKS)
assert PROBE_BLOCKS[0] == B_LESION and PROBE_BLOCKS[1] == B_LESION + 1 and PROBE_BLOCKS[2] == B_RESCUE + 1, \
    "the first three probe blocks must be the lesion block, its readout block and the first post-rescue block"
assert list(PROBE_BLOCKS) == sorted(set(PROBE_BLOCKS)), "probe blocks must be sorted and unique"
PROBE_BLOCKS_L = [int(b) for b in PROBE_BLOCKS]
PRIMARY_PROBE = B_RESCUE + 1
STATE_REF_BLOCKS = tuple(PROBE_BLOCKS_L[PROBE_ROLES.index(r)] for r in STATE_REF_BLOCKS_RULE)
INT_STEPS = tuple(sorted({T_STAR if s == "tstar" else (int(EARLY_STEP) if s == "early" else int(s))
                          for s in Q8.intervention_steps}))
assert all(0 <= s < CFG.num_steps for s in INT_STEPS), INT_STEPS
assert T_STAR in INT_STEPS, "t* must be among the intervention steps (primary analysis)"

# ------------------------------------------------- native attention backend (needed by the probe and key patch)
if hasattr(pipe.transformer, "set_attention_backend"):
    try:
        pipe.transformer.set_attention_backend("native")
        print("[q8-config] attention backend set to native (probe-compatible)")
    except Exception as _e:
        print(f"[q8-config][warn] could not set attention backend: {_e}")

# ------------------------------------------------- scenario scope
DEEP_PAIRS = {(p, s) for p, s in CFG.pilot_pairs}
if Q8.scenario_scope == "pilot":
    Q8_SCENARIOS = [(scenario_id(p, s), p, s, PROMPTS[p]) for p, s in CFG.pilot_pairs]
elif Q8.scenario_scope == "prompts20":
    _s0 = CFG.seeds[0]
    Q8_SCENARIOS = [(scenario_id(p, _s0), p, _s0, PROMPTS[p]) for p in sorted(PROMPTS)]
elif Q8.scenario_scope == "full":
    Q8_SCENARIOS = list(SCENARIOS)
else:
    raise ValueError(f"unknown scenario_scope {Q8.scenario_scope!r}")

# ------------------------------------------------- condition grammar
LESION_KINDS = ("cstar_zero", "vstar_destroy", "vstar_random_dir", "reg_remove")
RESIDUAL_RESCUES = ("vstar", "cstar", "norm", "full", "chan_c2", "chan_ctrl")
ATTN_RESCUES = ("key", "value", "key_value")
_COND_RE = re.compile(
    r"^(cstar_zero|vstar_destroy|vstar_random_dir|reg_remove)(_maint)?"
    r"(?:__(vstar|cstar|norm|full|chan_c2|chan_ctrl|key_value|key|value)"
    r"(?:_f([0-9]*\.?[0-9]+))?(_maint|_single)?(_at_random|_wrong)?)?$")

def parse_condition(name):
    """Condition name -> definition dict (block windows resolved)."""
    base = dict(name=name, family="none", lesion_kind="none", lesion_lo=-1, lesion_hi=-1,
                rescue_kind="none", rescue_lo=-1, rescue_hi=-1, frac=None, channel_role=None,
                position="target", source="target", is_sham=False)
    if name == "baseline":
        return base
    if name == "sham":
        return dict(base, family="sham", lesion_kind="sham", lesion_lo=B_LESION, lesion_hi=B_RESCUE - 1,
                    rescue_kind="sham", rescue_lo=B_RESCUE, rescue_hi=B_MAINT_END, is_sham=True)
    m = _COND_RE.match(name)
    if not m:
        raise ValueError(f"condition {name!r} does not match the grammar")
    les, les_maint, res, frac, ext, pos = m.groups()
    d = dict(base, family=les, lesion_kind=les, lesion_lo=B_LESION,
             lesion_hi=(B_MAINT_END if les_maint else B_RESCUE - 1))
    if res is None:
        if frac or ext or pos:
            raise ValueError(f"{name}: rescue options without a rescue")
        return d
    d["rescue_kind"] = res
    if res in RESIDUAL_RESCUES:
        if ext == "_single":
            raise ValueError(f"{name}: _single applies to attention rescues only")
        d["rescue_lo"], d["rescue_hi"] = B_RESCUE, (B_MAINT_END if ext == "_maint" else B_RESCUE)
    else:
        if ext == "_maint":
            raise ValueError(f"{name}: attention rescues are maintained over the rescue range by default; use _single for one block")
        d["rescue_lo"], d["rescue_hi"] = B_RESCUE, (B_RESCUE if ext == "_single" else B_MAINT_END)
    if frac is not None:
        if res != "vstar":
            raise ValueError(f"{name}: a fraction applies to the vstar rescue only")
        d["frac"] = float(frac)
        assert 0.0 < d["frac"] <= 1.0, name
    elif res == "vstar":
        d["frac"] = 1.0
    if res.startswith("chan_"):
        d["channel_role"] = res.split("_", 1)[1]        # "c2" | "ctrl"
    if pos in ("_at_random", "_wrong") and (ext is not None or frac is not None):
        raise ValueError(f"{name}: position and wrong-key controls cannot be combined with _maint, _single or a fraction")
    if pos == "_at_random":
        d["position"] = "random"
    elif pos == "_wrong":
        if res not in ATTN_RESCUES:
            raise ValueError(f"{name}: _wrong applies to attention rescues only")
        d["source"] = "random"
    if les_maint and d["rescue_kind"] in RESIDUAL_RESCUES and d["rescue_hi"] >= d["lesion_lo"]:
        # a maintained lesion overlaps a residual rescue. The hook applies
        # the lesion first and then the rescue.
        pass
    return d

CONDITIONS = [parse_condition(n) for n in Q8.conditions]
COND_BY_NAME = {c["name"]: c for c in CONDITIONS}
COND_NAMES = [c["name"] for c in CONDITIONS]
FAMILIES = [f for f in LESION_KINDS if any(c["family"] == f for c in CONDITIONS)]

# ------------------------------------------------- human-readable labels
_ROLE_COLORS = dict(black="#000000", verm="#D55E00", blue="#0072B2", orange="#E69F00", green="#009E73",
                    purple="#CC79A7", sky="#56B4E9", yellow="#F0E442")      # Okabe-Ito

def _fam_words(fam, c1, c2, ctrl):
    return {"cstar_zero": f"ch. {c1} of the register set to 0",
            "vstar_destroy": "v* component removed (norm kept)",
            "vstar_random_dir": "register direction randomized (norm kept)",
            "reg_remove": "register replaced by the mean ordinary token"}[fam]

def condition_label(cond, c1, c2, ctrl, short=False):
    """The long label reads "lesion; rescue". The short label names a single
    operation (lesions by name, rescues starting with '+') and is used in
    legends shared across families."""
    n = cond["name"]
    if n == "baseline":
        return "baseline" if short else "baseline (no intervention)"
    if n == "sham":
        return "sham" if short else "sham (stored clean register, key and value written back)"
    les = _fam_words(cond["family"], c1, c2, ctrl)
    if cond["lesion_hi"] > B_RESCUE - 1:
        les += f" through block {cond['lesion_hi']}"
    rk = cond["rescue_kind"]
    if rk == "none":
        return ("lesion: " + les) if short else les
    res = {"vstar": "projection on v* restored", "cstar": f"ch. {c1} restored",
           "norm": "norm restored (direction kept)", "full": "full clean state restored",
           "chan_c2": f"ch. {c2} restored", "chan_ctrl": f"control ch. {ctrl} restored",
           "key": "clean key restored in attention", "value": "clean value restored in attention",
           "key_value": "clean key and value restored"}[rk]
    if rk == "vstar" and cond["frac"] is not None and cond["frac"] < 1.0:
        res = f"{cond['frac']:g} of the v* projection restored"
    if rk in ATTN_RESCUES:
        res += (" at one block" if cond["rescue_hi"] == cond["rescue_lo"] else f" (blocks {cond['rescue_lo']} to {cond['rescue_hi']})")
    elif cond["rescue_hi"] > cond["rescue_lo"]:
        res += f" at every block {cond['rescue_lo']} to {cond['rescue_hi']}"
    if cond["position"] == "random":
        res += " at a random ordinary token"
    if cond["source"] == "random":
        res = res.replace("clean", "random token's clean")
    if short and cond["lesion_hi"] > B_RESCUE - 1:
        res += " (lesion maintained)"
    return ("+ " + res) if short else (les + "; " + res)

# ------------------------------------------------- condition styles, one per operation
# Colour identifies the operation. Lesions are reds, one per family. Each
# rescue has its own colour, for example blue for the v* projection (lighter
# blues with markers for partial doses), purple for c*, greens for the key
# variants, pink for the value, orange for the norm, grey for the full state,
# olives for the control channels and browns with hollow markers for the
# position and wrong-key controls. Maintained variants get their own shade,
# a dash-dot-dot line and an 'x' marker. Dash pattern and marker give a
# second cue that doesn't rely on colour. Thick solid lines sit at the
# bottom and thin, dashed or marked lines on top, so coincident lines both
# stay visible. Below, the palette is checked for CIEDE2000 separation
# within each family, and every figure is checked again after drawing
# (check_fig_styles).
_LESION_COLORS = {"cstar_zero": "#D55E00", "vstar_destroy": "#B2182B", "reg_remove": "#E7298A", "vstar_random_dir": "#7F0000"}
_LESION_MAINT_COLORS = {"cstar_zero": "#E8A673", "vstar_destroy": "#6B0E1A", "reg_remove": "#8B1953", "vstar_random_dir": "#4C0000"}
_RESCUE_COLORS = {"vstar": "#0072B2", "vstar_maint": "#023E73", "cstar": "#984EA3", "cstar_maint": "#4A1486",
                  "key": "#009E73", "key_single": "#005C43", "key_value": "#6ACC9A", "key_value_single": "#20B2AA",
                  "value": "#F781BF", "value_single": "#A0306A",
                  "norm": "#E69F00", "norm_maint": "#A66E00", "full": "#595959", "full_maint": "#00838F",
                  "chan_c2": "#808000", "chan_ctrl": "#BCBD22", "chan_c2_maint": "#4B4B00", "chan_ctrl_maint": "#7A7A16",
                  "vstar_at_random": "#8C510A", "cstar_at_random": "#BF812D", "key_at_random": "#9C755F",
                  "key_wrong": "#543005", "value_at_random": "#C08A6E", "value_wrong": "#6E3B1E",
                  "key_value_at_random": "#7A6A52", "key_value_wrong": "#3D2B1F", "chan_c2_at_random": "#A08B3A",
                  "chan_ctrl_at_random": "#D0C060", "norm_at_random": "#B5891A", "full_at_random": "#6E7FA3"}
_AT_RANDOM_MARKERS = {"vstar": "o", "cstar": "D", "key": "s", "value": "P", "key_value": "*", "norm": "v", "full": "^",
                      "chan_c2": "<", "chan_ctrl": ">"}
_DOSE_COLORS = {0.75: "#3F93CC", 0.5: "#74B3E0", 0.25: "#A8CFEC"}
_DOSE_MARKERS = {0.75: "D", 0.5: "s", 0.25: "^"}
_DASHES = {"solid": (), "long": (6, 2.2), "dashdot": (5, 1.5, 1.2, 1.5), "dashdotdot": (5, 1.3, 1, 1.3, 1, 1.3),
           "short": (3, 1.6), "dotted": (1.0, 1.4), "fine": (0.8, 1.6)}

def _hex_mix(h1, h2, f):
    a = np.array([int(h1.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4)], float)
    b = np.array([int(h2.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4)], float)
    c = np.clip((1 - f) * a + f * b, 0, 255)
    return "#%02X%02X%02X" % tuple(int(round(x)) for x in c)

def _relative_luminance(hexcol):
    c = np.array([int(hexcol.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4)], float) / 255.0
    lin = np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
    return float(0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2])

def condition_style(cond):
    """Style dict of a condition with keys color, dash (a key of _DASHES),
    lw, marker, ms, mfc (None means filled with the line colour) and zorder.
    A rescue on top of a maintained lesion keeps the rescue's dash but gets
    its colour mixed half with mid grey and an 'x' marker (hollow-marker
    controls get their marker filled instead), so it never looks like the
    same rescue after the one-block lesion."""
    st = _base_style(cond)
    rk = cond["rescue_kind"]
    if rk != "none" and cond["lesion_hi"] > B_RESCUE - 1:
        if st["mfc"] is not None:                     # hollow control marker, keep the shape and fill it
            st = dict(st, color=_hex_mix(st["color"], "#808080", 0.5), mfc=None, ms=max(st["ms"], 3.6))
        else:
            st = dict(st, color=_hex_mix(st["color"], "#808080", 0.5), marker=("+" if st["marker"] == "x" else "x"),
                      ms=max(st["ms"], 3.6))
    return st

def _base_style(cond):
    n, rk, fam = cond["name"], cond["rescue_kind"], cond["family"]
    st = dict(color="#7F7F7F", dash="solid", lw=1.4, marker=None, ms=3.0, mfc=None, zorder=5)
    if n == "baseline":
        return dict(st, color="#000000", lw=3.0, zorder=1)
    if n == "sham":
        return dict(st, color="#8C8C8C", dash="fine", lw=1.2, zorder=8)
    maint_les = cond["lesion_hi"] > B_RESCUE - 1
    if rk == "none":
        if maint_les:
            return dict(st, color=_LESION_MAINT_COLORS[fam], dash="dashdotdot", lw=2.0, marker="x", ms=3.5, zorder=2)
        return dict(st, color=_LESION_COLORS[fam], lw=2.4, zorder=2)
    maint_res = rk in RESIDUAL_RESCUES and cond["rescue_hi"] > cond["rescue_lo"]
    single = rk in ATTN_RESCUES and cond["rescue_hi"] == cond["rescue_lo"]
    if cond["position"] == "random":
        return dict(st, color=_RESCUE_COLORS.get(f"{rk}_at_random", "#8C510A"), dash="short", lw=1.4,
                    marker=_AT_RANDOM_MARKERS.get(rk, "o"), ms=3.6, mfc="white", zorder=7)
    if cond["source"] == "random":
        return dict(st, color=_RESCUE_COLORS.get(f"{rk}_wrong", "#543005"), dash="dashdot", lw=1.4,
                    marker=_AT_RANDOM_MARKERS.get(rk, "s"), ms=3.4, mfc="white", zorder=7)
    if rk == "vstar":
        f = cond["frac"] if cond["frac"] is not None else 1.0
        if f < 1.0:
            col = _DOSE_COLORS.get(f, _hex_mix(_RESCUE_COLORS["vstar"], "#FFFFFF", 0.7 * (1.0 - f)))
            if maint_res:
                return dict(st, color=_hex_mix(col, "#000000", 0.35), dash="dashdotdot", lw=1.4, marker=_DOSE_MARKERS.get(f, "p"), ms=3.6, mfc="white", zorder=6)
            return dict(st, color=col, lw=1.4, marker=_DOSE_MARKERS.get(f, "p"), ms=3.6, zorder=5)
        if maint_res:
            return dict(st, color=_RESCUE_COLORS["vstar_maint"], dash="dashdotdot", lw=1.6, marker="x", ms=3.5, zorder=6)
        return dict(st, color=_RESCUE_COLORS["vstar"], lw=1.5, marker="o", ms=3.0, zorder=5)
    if rk == "cstar":
        if maint_res:
            return dict(st, color=_RESCUE_COLORS["cstar_maint"], dash="dashdotdot", lw=1.6, marker="x", ms=3.5, zorder=6)
        return dict(st, color=_RESCUE_COLORS["cstar"], lw=1.5, marker="o", ms=3.0, zorder=5)
    if rk == "key":
        if single:
            return dict(st, color=_RESCUE_COLORS["key_single"], dash="short", lw=1.5, marker="+", ms=4.2, zorder=6)
        return dict(st, color=_RESCUE_COLORS["key"], lw=1.5, marker="o", ms=3.0, zorder=5)
    if rk == "key_value":
        if single:
            return dict(st, color=_RESCUE_COLORS["key_value_single"], dash="short", lw=1.5, marker="*", ms=4.4, zorder=6)
        return dict(st, color=_RESCUE_COLORS["key_value"], lw=1.5, marker="*", ms=4.4, zorder=6)
    if rk == "value":
        if single:
            return dict(st, color=_RESCUE_COLORS["value_single"], dash="short", lw=1.5, marker="P", ms=3.4, zorder=6)
        return dict(st, color=_RESCUE_COLORS["value"], dash="dashdot", lw=1.5, marker="P", ms=3.4, zorder=6)
    if rk == "norm":
        if maint_res:
            return dict(st, color=_RESCUE_COLORS["norm_maint"], dash="dashdotdot", lw=1.8, marker="x", ms=3.5, zorder=4)
        return dict(st, color=_RESCUE_COLORS["norm"], dash="dashdot", lw=1.8, zorder=4)
    if rk == "full":
        if maint_res:
            return dict(st, color=_RESCUE_COLORS["full_maint"], dash="dashdotdot", lw=1.8, marker="x", ms=3.5, zorder=4)
        return dict(st, color=_RESCUE_COLORS["full"], dash="long", lw=2.2, zorder=3)
    if rk in ("chan_c2", "chan_ctrl"):
        if maint_res:
            return dict(st, color=_RESCUE_COLORS[f"{rk}_maint"], dash="dashdotdot", lw=1.8, marker="x", ms=3.5, zorder=6)
        return dict(st, color=_RESCUE_COLORS[rk], dash="dotted", lw=1.8, marker=("v" if rk == "chan_c2" else "^"), ms=3.4, zorder=6)
    return st

def line_kwargs(cond, lw_scale=1.0):
    """Keyword arguments for ax.plot / Line2D from a condition's style."""
    s = condition_style(cond)
    kw = dict(color=s["color"], lw=s["lw"] * lw_scale, zorder=s["zorder"], solid_capstyle="round")
    d = _DASHES[s["dash"]]
    kw["ls"] = "-" if not d else (0, d)
    if s["marker"]:
        kw.update(marker=s["marker"], ms=s["ms"], mfc=(s["mfc"] or s["color"]), mec=s["color"], mew=(0.9 if s["mfc"] else 0.6))
    return kw

def legend_handle(cond):
    kw = line_kwargs(cond)
    kw["lw"] = min(kw["lw"], 2.2)
    return plt.Line2D([], [], **kw)

def op_sort_key(cond):
    n, rk = cond["name"], cond["rescue_kind"]
    if n == "baseline":
        return (0, 0.0)
    if n == "sham":
        return (1, 0.0)
    fi = LESION_KINDS.index(cond["family"]) if cond["family"] in LESION_KINDS else 9
    if rk == "none":
        return (2, fi + (0.5 if cond["lesion_hi"] > B_RESCUE - 1 else 0.0))
    order = {"vstar": 10, "cstar": 12, "key": 14, "key_value": 16, "value": 17, "norm": 20, "full": 21, "chan_c2": 30, "chan_ctrl": 31}
    k = order.get(rk, 50) + 0.0
    if rk == "vstar" and cond["frac"] is not None:
        k += 1.0 - cond["frac"]                       # full projection first, then decreasing doses
    if rk in RESIDUAL_RESCUES and cond["rescue_hi"] > cond["rescue_lo"]:
        k += 0.05
    if rk in ATTN_RESCUES and cond["rescue_hi"] == cond["rescue_lo"]:
        k += 0.5
    if cond["position"] == "random":
        k += 40
    if cond["source"] == "random":
        k += 41
    return (3, k)

def op_legend(conds, include_baseline=True):
    """One legend handle per operation label in `conds` (plus the baseline),
    in a fixed order. A rescue used in several families appears once."""
    seen = {}
    for c in ([COND_BY_NAME["baseline"]] if include_baseline else []) + list(conds):
        lab = COND_LABELS_SHORT[c["name"]]
        if lab not in seen:
            seen[lab] = (op_sort_key(c), legend_handle(c))
    items = sorted(seen.items(), key=lambda kv: kv[1][0])
    return [h for _, (_, h) in items], [lab for lab, _ in items]

# ---- palette check (CIEDE2000, normal vision) among styles that can share a panel
def _srgb_to_lab(hexcol):
    c = np.array([int(hexcol.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4)], float) / 255.0
    lin = np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
    xyz = np.array([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.0721750], [0.0193339, 0.1191920, 0.9503041]]) @ lin
    xyz = xyz / np.array([0.95047, 1.0, 1.08883])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16.0 / 116.0)
    return np.array([116.0 * f[1] - 16.0, 500.0 * (f[0] - f[1]), 200.0 * (f[1] - f[2])])

def ciede2000_hex(h1, h2):
    import math as _m
    L1, a1, b1 = _srgb_to_lab(h1); L2, a2, b2 = _srgb_to_lab(h2)
    C1, C2 = _m.hypot(a1, b1), _m.hypot(a2, b2); Cb = 0.5 * (C1 + C2)
    G = 0.5 * (1 - _m.sqrt(Cb ** 7 / (Cb ** 7 + 25.0 ** 7)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p, C2p = _m.hypot(a1p, b1), _m.hypot(a2p, b2)
    h1p = _m.degrees(_m.atan2(b1, a1p)) % 360 if (a1p or b1) else 0.0
    h2p = _m.degrees(_m.atan2(b2, a2p)) % 360 if (a2p or b2) else 0.0
    dLp, dCp = L2 - L1, C2p - C1p
    if C1p * C2p == 0:
        dhp = 0.0
    elif abs(h2p - h1p) <= 180:
        dhp = h2p - h1p
    else:
        dhp = h2p - h1p - 360 if h2p - h1p > 180 else h2p - h1p + 360
    dHp = 2 * _m.sqrt(C1p * C2p) * _m.sin(_m.radians(dhp / 2))
    Lbp, Cbp = 0.5 * (L1 + L2), 0.5 * (C1p + C2p)
    if C1p * C2p == 0:
        hbp = h1p + h2p
    elif abs(h1p - h2p) <= 180:
        hbp = 0.5 * (h1p + h2p)
    else:
        hbp = 0.5 * (h1p + h2p + 360) if h1p + h2p < 360 else 0.5 * (h1p + h2p - 360)
    T = (1 - 0.17 * _m.cos(_m.radians(hbp - 30)) + 0.24 * _m.cos(_m.radians(2 * hbp))
         + 0.32 * _m.cos(_m.radians(3 * hbp + 6)) - 0.20 * _m.cos(_m.radians(4 * hbp - 63)))
    dth = 30 * _m.exp(-((hbp - 275) / 25) ** 2)
    RC = 2 * _m.sqrt(Cbp ** 7 / (Cbp ** 7 + 25.0 ** 7))
    SL = 1 + 0.015 * (Lbp - 50) ** 2 / _m.sqrt(20 + (Lbp - 50) ** 2)
    SC, SH = 1 + 0.045 * Cbp, 1 + 0.015 * Cbp * T
    RT = -_m.sin(_m.radians(2 * dth)) * RC
    return _m.sqrt((dLp / SL) ** 2 + (dCp / SC) ** 2 + (dHp / SH) ** 2 + RT * (dCp / SC) * (dHp / SH))

STYLE_MIN_DE = 8.0          # colours closer than this (CIEDE2000) must differ in dash pattern or marker
_ALL_OP_STYLES = {}
for _fam_ in LESION_KINDS:
    for _lp_ in (_fam_, _fam_ + "_maint"):
        _names_ = [_lp_]
        for _r_ in RESIDUAL_RESCUES + ATTN_RESCUES:
            for _fr_ in ("", "_f0.5"):
                for _ext_ in ("", "_maint", "_single"):
                    for _pos_ in ("", "_at_random", "_wrong"):
                        _names_.append(f"{_lp_}__{_r_}{_fr_}{_ext_}{_pos_}")
        for _nm_ in _names_:
            try:
                _c_ = parse_condition(_nm_)
            except (ValueError, AssertionError):
                continue
            _s_ = condition_style(_c_)
            _ALL_OP_STYLES[_nm_] = (_s_["color"].upper(), _s_["dash"], _s_["marker"], _s_["mfc"] is None)
_style_clashes = []
for _fam_ in LESION_KINDS:
    _in_fam_ = {k: v for k, v in _ALL_OP_STYLES.items() if k == _fam_ or k.startswith(_fam_ + "_")}
    _keys_ = list(_in_fam_)
    for _i_ in range(len(_keys_)):
        for _j_ in range(_i_ + 1, len(_keys_)):
            (c1, d1, m1, f1), (c2, d2, m2, f2) = _in_fam_[_keys_[_i_]], _in_fam_[_keys_[_j_]]
            if c1 == c2 or (ciede2000_hex(c1, c2) < STYLE_MIN_DE and (d1, m1, f1) == (d2, m2, f2)):
                _style_clashes.append((_keys_[_i_], _keys_[_j_], c1, c2))
assert not _style_clashes, f"condition styles clash within a family panel: {_style_clashes[:5]} ({len(_style_clashes)} total)"
_n_cols_ = len({v[0] for v in _ALL_OP_STYLES.values()})
print(f"[q8-config] condition styles: {_n_cols_} distinct colours over {len(_ALL_OP_STYLES)} possible operations; "
      f"no two operations of one family share a colour, and colours closer than CIEDE2000 {STYLE_MIN_DE:g} differ in dash or marker")

# ------------------------------------------------- comparability manifest (absolute and relative depths)
_nb, _nt = int(CFG.n_blocks_expected), int(CFG.num_steps)
COMPARABILITY = {
    "model_family": MODEL_FAMILY, "model_id": CFG.model_id,
    "n_blocks": _nb, "n_steps": _nt, "d_model": int(CFG.d_model), "n_heads": int(N_HEADS_EXPECTED),
    "n_img_tokens": int(N_IMG), "text_tokens_in_sequence": int(N_TXT_IN_SEQ),
    "guidance": ("embedded guidance, batch 1" if MODEL_FAMILY == "flux"
                 else f"classifier-free guidance, batch {EXP_BATCH}, conditional row {SELECT_ROW} measured, operators applied to all rows"),
    "precision_mode": PRECISION_MODE,
    "band": [_BAND_LO, _BAND_HI], "band_rel": [round(_BAND_LO / (_nb - 1), 3), round(_BAND_HI / (_nb - 1), 3)],
    "l_star": L_STAR, "t_star": T_STAR,
    "lesion_block": B_LESION, "lesion_block_rel": round(B_LESION / (_nb - 1), 3),
    "lesion_block_in_band": round((B_LESION - _BAND_LO) / max(_BAND_HI - _BAND_LO, 1), 3),
    "rescue_block": B_RESCUE, "primary_probe": PRIMARY_PROBE, "maintain_end": B_MAINT_END,
    "probe_blocks": PROBE_BLOCKS_L, "probe_roles": list(PROBE_ROLES),
    "probe_blocks_rel": [round(b / (_nb - 1), 3) for b in PROBE_BLOCKS_L],
    "intervention_steps": [int(s) for s in INT_STEPS],
    "intervention_steps_rel": [round(s / (_nt - 1), 3) for s in INT_STEPS],
    "state_ref_blocks": [int(b) for b in STATE_REF_BLOCKS],
    "register_identity": f"baseline argmax-norm image token at block {B_LESION} at the intervened step"
                         + (f"; expected at token {REG_EXPECT_TOKEN}" if REG_EXPECT_TOKEN is not None else "; position varies"),
    "rules_identical_across_models": {
        "lesion_window": f"blocks {B_LESION}..{B_RESCUE - 1} (maintained variants through {B_MAINT_END})",
        "residual_rescue": f"block {B_RESCUE} (maintained variants through {B_MAINT_END})",
        "attention_rescue": f"blocks {B_RESCUE}..{B_MAINT_END} (single variants at {B_RESCUE})",
        "targets": f"{Q8.n_targets} token(s); extra targets need norm >= {Q8.target_norm_mult:g} x median",
        "ordinary_tokens": f"norm < {Q8.ordinary_norm_mult:g} x median at the lesion block",
        "vstar": Q8.vstar_source,
        "primary_endpoint": f"log10 incoming attention share of the register (x uniform) at block {PRIMARY_PROBE}, step t*",
        "rescue_fraction": f"(m_cond - m_lesion) / (m_clean - m_lesion) per scenario, gap floor {Q8.rf_gap_floor_log10:g} log10",
        "statistics": "paired per scenario; exact sign test, Wilcoxon, prompt-level sign test, prompt-clustered bootstrap CI, Holm; pre-declared equivalence margins",
        "conditions": COND_NAMES,
    },
    "note": ("Depth shifts and block positions are reported in blocks and in relative depth (block / (n_blocks - 1)); "
             "one PixArt block spans about twice the relative depth of one FLUX block."),
}
save_json_atomic(COMPARABILITY, os.path.join(Q8_PATHS["analysis"], "comparability.json"))

print(f"[q8-config] hash {Q8_HASH} | {len(CONDITIONS)} conditions: {COND_NAMES}")
print(f"[q8-config] lesion block {B_LESION} (window {B_LESION}..{B_RESCUE - 1}) | rescue block {B_RESCUE} | "
      f"maintained end {B_MAINT_END} | primary probe {PRIMARY_PROBE} | state refs {STATE_REF_BLOCKS} | "
      f"steps {INT_STEPS} (t* = {T_STAR})")
print(f"[q8-config] scope={Q8.scenario_scope}: {len(Q8_SCENARIOS)} scenarios x {len(CONDITIONS)} conditions x "
      f"{len(INT_STEPS)} step(s) = {len(Q8_SCENARIOS) * ((len(CONDITIONS) - 1) * len(INT_STEPS) + 1)} generations")
for _k, _v in asdict(Q8).items():
    print(f"    {_k:24s} = {_v}")

# =====================================================================
# Figure finishing
#
# The legend goes in a strip above the panels and the title above
# everything at a fixed gap, so it can be cropped off. Each figure is saved
# with and without the title (vector PDF and 600 dpi PNG) plus a caption
# draft. After drawing, every pair of overlapping text elements is printed
# and recorded.
# =====================================================================
TITLE_PT = _BASE_PT + 0.5
TITLE_GAP_PT = 8.0
LEGEND_PT = _BASE_PT - 1.5
FIG_CHECK_PATH = os.path.join(Q8_PATHS["analysis"], "figure_checks.json")
_FIG_CHECKS = {}

def _artist_boxes(fig):
    renderer = fig.canvas.get_renderer()
    boxes = []
    def _add(art, bb=None):
        try:
            if bb is None:
                if art is None or not art.get_visible():
                    return
                if hasattr(art, "get_text") and not str(art.get_text()).strip():
                    return
                bb = art.get_window_extent(renderer)
            if bb is not None and bb.width > 0 and bb.height > 0 and np.isfinite([bb.x0, bb.x1, bb.y0, bb.y1]).all():
                boxes.append(bb)
        except Exception:
            pass
    for ax in fig.axes:
        if not ax.get_visible():
            continue
        _add(ax)
        for axis in (ax.xaxis, ax.yaxis):
            try:
                _add(None, axis.get_tightbbox(renderer))
            except Exception:
                pass
        for art in (ax.title, getattr(ax, "_left_title", None), getattr(ax, "_right_title", None),
                    ax.xaxis.label, ax.yaxis.label, ax.get_legend()):
            _add(art)
        for t in list(ax.texts):
            _add(t)
    for leg in fig.legends:
        _add(leg)
    for t in fig.texts:
        _add(t)
    return boxes

def _content_extent_display(fig):
    boxes = _artist_boxes(fig)
    if not boxes:
        return fig.bbox.x0, fig.bbox.x1, fig.bbox.y1
    return (min(b.x0 for b in boxes), max(b.x1 for b in boxes), max(b.y1 for b in boxes))

def add_fig_legend(fig, ax, ncol=4, title=None, handles=None, labels=None):
    if handles is None:
        handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return None
    return fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.0),
                      ncol=ncol, frameon=False, fontsize=LEGEND_PT,
                      columnspacing=1.0, handletextpad=0.4, title=title,
                      title_fontsize=LEGEND_PT)

def add_fig_title(fig, title, gap_pt=TITLE_GAP_PT, fontsize=TITLE_PT):
    fig.canvas.draw()
    x0, x1, top = _content_extent_display(fig)
    inv = fig.transFigure.inverted()
    cx = inv.transform((0.5 * (x0 + x1), 0.0))[0]
    top_fig = inv.transform((0.0, top))[1]
    gap_fig = (gap_pt / 72.0) / fig.get_figheight()
    max_chars = max(24, int((x1 - x0) / fig.dpi * 72.0 / (0.55 * fontsize)))
    txt = None
    for _ in range(6):
        if txt is not None:
            txt.remove()
        text = "\n".join(_textwrap.wrap(title, width=max_chars, break_long_words=False, break_on_hyphens=False))
        txt = fig.text(cx, top_fig + gap_fig, text, ha="center", va="bottom",
                       fontsize=fontsize, fontweight="semibold", linespacing=1.15)
        fig.canvas.draw()
        bb = txt.get_window_extent(fig.canvas.get_renderer())
        if bb.width <= (x1 - x0) * 1.005 or max_chars <= 24:
            break
        max_chars = max(24, int(max_chars * min(0.92, (x1 - x0) / bb.width)))
    return txt

def _text_artists(fig):
    """(label, artist) for every visible, non-empty text element (titles,
    axis and tick labels, annotations, legend entries, figure texts)."""
    out = []
    for ai, ax in enumerate(fig.axes):
        if not ax.get_visible():
            continue
        for nm, art in (("title", ax.title), ("xlabel", ax.xaxis.label), ("ylabel", ax.yaxis.label)):
            out.append((f"ax{ai}.{nm}", art))
        for kind, axis in (("xtick", ax.xaxis), ("ytick", ax.yaxis)):
            lo, hi = sorted(axis.get_view_interval())
            span = max(hi - lo, 1e-12)
            for ti, tick in enumerate(list(axis.get_major_ticks()) + list(axis.get_minor_ticks())):
                loc = tick.get_loc()
                if not (lo - 1e-9 * span <= loc <= hi + 1e-9 * span):
                    continue                      # tick outside the view, its label is clipped and not drawn
                for lab in (tick.label1, tick.label2):
                    if lab.get_visible():
                        out.append((f"ax{ai}.{kind}{ti}", lab))
        for ti, t in enumerate(ax.texts):
            out.append((f"ax{ai}.text{ti}", t))
        leg = ax.get_legend()
        if leg is not None:
            for ti, t in enumerate(leg.get_texts()):
                out.append((f"ax{ai}.legend{ti}", t))
            if leg.get_title() is not None:
                out.append((f"ax{ai}.legendtitle", leg.get_title()))
    for li, leg in enumerate(fig.legends):
        for ti, t in enumerate(leg.get_texts()):
            out.append((f"fig.legend{li}.{ti}", t))
    for ti, t in enumerate(fig.texts):
        out.append((f"fig.text{ti}", t))
    keep = []
    for nm, art in out:
        try:
            if art.get_visible() and str(art.get_text()).strip():
                keep.append((nm, art))
        except Exception:
            pass
    return keep

def check_fig_overlaps(fig, stem_name, tol_pt=0.5):
    """Pairwise overlap test of all text bounding boxes, in display units
    and shrunk by tol_pt on every side. Returns the overlapping pairs and
    records them under the figure name for the manifest."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    tol = tol_pt * fig.dpi / 72.0
    items = []
    for nm, art in _text_artists(fig):
        try:
            bb = art.get_window_extent(r)
        except Exception:
            continue
        if bb.width <= 0 or bb.height <= 0:
            continue
        items.append((nm, bb.x0 + tol, bb.x1 - tol, bb.y0 + tol, bb.y1 - tol))
    bad = []
    for i in range(len(items)):
        n1, ax0, ax1, ay0, ay1 = items[i]
        for j in range(i + 1, len(items)):
            n2, bx0, bx1, by0, by1 = items[j]
            if ax0 < bx1 and bx0 < ax1 and ay0 < by1 and by0 < ay1:
                bad.append((n1, n2))
    rec = _FIG_CHECKS.setdefault(stem_name, {})
    rec.update({"n_text": len(items), "overlaps": bad})
    save_json_atomic(_FIG_CHECKS, FIG_CHECK_PATH)
    if bad:
        print(f"[fig-check][WARN] {stem_name}: {len(bad)} overlapping text pair(s): {bad[:6]}")
    return bad

def _line_style_key(line):
    import matplotlib.colors as _mc
    col = _mc.to_hex(line.get_color()).upper()
    ls = line.get_linestyle()
    pat = getattr(line, "_unscaled_dash_pattern", None) or getattr(line, "_dash_pattern", None) or (0, None)
    dash = tuple(np.round(np.asarray(pat[1], float), 2)) if pat[1] is not None else ()
    mk = line.get_marker()
    mk = None if mk in (None, "None", "", " ") else str(mk)
    filled = True
    if mk is not None:
        try:
            filled = _mc.to_hex(line.get_markerfacecolor()).upper() == col
        except (ValueError, TypeError):
            filled = False
    return col, (str(ls), dash), (mk, filled)

def check_fig_styles(fig, stem_name):
    """In each axes, two labelled data series must not share a colour, and
    two series closer than STYLE_MIN_DE (CIEDE2000) must differ in dash or
    marker. Only Line2D objects in data coordinates with at least one point
    count, so guide lines and empty legend proxies are skipped. Returns the
    clashes and records them for the manifest."""
    bad = []
    for ai, ax in enumerate(fig.axes):
        if not ax.get_visible():
            continue
        lines = [l for l in ax.get_lines() if not str(l.get_label()).startswith("_") and len(l.get_xdata()) > 0
                 and l.get_transform() == ax.transData and l.get_visible()]
        for i in range(len(lines)):
            for j in range(i + 1, len(lines)):
                a, b = lines[i], lines[j]
                if a.get_label() == b.get_label():
                    continue                                     # same series drawn twice, e.g. a legend duplicate
                (ca, da, ma), (cb, db, mb) = _line_style_key(a), _line_style_key(b)
                de = ciede2000_hex(ca, cb)
                if ca == cb:
                    bad.append((f"ax{ai}", str(a.get_label()), str(b.get_label()), "same colour"))
                elif de < STYLE_MIN_DE and (da, ma) == (db, mb):
                    bad.append((f"ax{ai}", str(a.get_label()), str(b.get_label()), f"dE {de:.1f}, same dash and marker"))
    rec = _FIG_CHECKS.setdefault(stem_name, {})
    rec["style_clashes"] = bad
    save_json_atomic(_FIG_CHECKS, FIG_CHECK_PATH)
    if bad:
        print(f"[fig-check][WARN] {stem_name}: {len(bad)} series style clash(es): {bad[:4]}")
    return bad

def save_paper_fig(fig, stem, title, caption=None, legend_ax=None, ncol=4,
                   legend_title=None, show=True, legend_handles=None, legend_labels=None):
    """Finish and save one figure with legend strip, title-free and titled
    copies, caption draft, and the overlap and style checks on both."""
    if legend_ax is not None or legend_handles is not None:
        add_fig_legend(fig, legend_ax, ncol=ncol, title=legend_title,
                       handles=legend_handles, labels=legend_labels)
    save_fig(fig, stem + "_notitle", caption=None, show=False)
    check_fig_overlaps(fig, os.path.basename(stem) + "_notitle")
    check_fig_styles(fig, os.path.basename(stem) + "_notitle")
    add_fig_title(fig, title)
    save_fig(fig, stem, caption=caption, show=show)
    check_fig_overlaps(fig, os.path.basename(stem))
    check_fig_styles(fig, os.path.basename(stem))

def panel_grid(n, max_cols=4, panel_h=1.9, sharey=True, sharex=True):
    ncols = min(max_cols, n)
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_PAGE, panel_h * nrows),
                             sharex=sharex, sharey=sharey, squeeze=False)
    flat = list(axes.flat)
    for ax in flat[n:]:
        ax.set_visible(False)
    return fig, flat[:n]

def wrap_label(text, width=22):
    """Word wrap that never splits a word. A word longer than the width
    gets its own line."""
    return "\n".join(_textwrap.wrap(str(text), width=max(1, int(width)), break_long_words=False, break_on_hyphens=False))

def grid_labels(axs, max_cols, xlabel, ylabel, fontsize=None):
    fs = LEGEND_PT if fontsize is None else fontsize
    n = len(axs); ncols = min(max_cols, n)
    for col in range(ncols):
        col_axes = [axs[i] for i in range(col, n, ncols)]
        col_axes[-1].set_xlabel(xlabel, fontsize=fs)
        col_axes[-1].tick_params(labelbottom=True)
        for ax in col_axes[:-1]:
            ax.tick_params(labelbottom=False)
    for i in range(0, n, ncols):
        axs[i].set_ylabel(ylabel, fontsize=fs)

def log_floor(values, frac=1e-3, at_least=1e-2):
    v = np.asarray(values, np.float64)
    v = v[np.isfinite(v) & (v > 0)]
    return max(at_least, float(v.min()) * frac) if v.size else at_least

def mark_blocks(ax, first=True):
    """Mark the high-norm band, the lesion block and the rescue block. Only
    the first panel of a figure gets labels."""
    ax.axvspan(_BAND_LO, _BAND_HI, color=OKABE_ITO[1], alpha=0.10, lw=0, label="high-norm band" if first else None)
    ax.axvline(B_LESION, color="0.25", ls=":", lw=0.9, label=f"lesion block {B_LESION}" if first else None)
    ax.axvline(B_RESCUE, color="0.25", ls="-.", lw=0.7, label=f"rescue block {B_RESCUE}" if first else None)

print(f"[fig-finish] titled + title-free saves, title gap {TITLE_GAP_PT:g} pt, overlap audit -> {os.path.basename(FIG_CHECK_PATH)}")


## R2. Channel identities, v\* and the frozen decision

The channel roles come from rules fixed before the pilot. c\* is the largest
channel at the analysis block. The massive-channel control c2 is the largest
other channel at the band end, which is the catching-up channel. The
ordinary control channel is taken from the earlier dissolution experiment's
decision file when it exists, otherwise from a rule at the lesion block.
v\* is loaded from that experiment's frozen file when it exists, otherwise
refitted from the recurrence run's full tensors with the same definition.

Everything is frozen to `meta/q8_decision.json` and `meta/vstar_q8.npz`. In
later sessions the stored files are checked against a recomputation, and
the cell stops on a mismatch.


In [ ]:
# =====================================================================
# R2. Channel identities, v* and the frozen decision
#
# These rules were fixed before the pilot. c* is the channel with the
# largest median mean|x| at the analysis block l* (pilot, step t*). c2 is
# the largest other channel at the band end, the catching-up channel. The
# ordinary control channel comes from the earlier dissolution experiment's
# decision file when it's on Drive, so both experiments use the same
# control. Otherwise it's the largest channel at the lesion block that stays
# below ctrl_exclude_mult x the median channel, with c* and c2 excluded.
#
# v* is that experiment's frozen vector when its file exists (same
# definition, identical bytes). Otherwise it's refitted from the recurrence
# run's full tensors at (l*, t*). Everything is frozen to
# meta/q8_decision.json and meta/vstar_q8.npz and checked against a
# recomputation in every later session.
# =====================================================================
_zs = {s: np.load(os.path.join(PATHS["pilot"], f"{s}.npz"), allow_pickle=False) for s in PILOT_SIDS}
_nb = CFG.n_blocks_expected
MED_PROF = np.zeros((_nb, CFG.d_model), np.float64)
for _b in range(_nb):
    MED_PROF[_b] = np.median(np.stack([_z[f"b{_b:02d}__mean_abs"][T_STAR].astype(np.float64)
                                       for _z in _zs.values()]), axis=0)
for _z in _zs.values():
    _z.close()

CH1 = int(np.argmax(MED_PROF[L_STAR]))
_end_vals = MED_PROF[_BAND_HI].copy(); _end_vals[CH1] = -np.inf
CH2 = int(np.argmax(_end_vals))

_q6dec = None
if os.path.exists(Q6_DEC_PATH):
    with open(Q6_DEC_PATH) as _f:
        _q6dec = json.load(_f)
if _q6dec is not None and "ch_ctrl" in _q6dec:
    CH_CTRL = int(_q6dec["ch_ctrl"]); CTRL_SOURCE = "q6_decision"
else:
    _v = MED_PROF[B_LESION].copy()
    _med_scalar = max(float(np.median(_v)), 1e-12)
    _eligible = _v < Q8.ctrl_exclude_mult * _med_scalar
    _eligible[[CH1, CH2]] = False
    if not _eligible.any():
        raise RuntimeError("no eligible control channel below the outlier threshold")
    CH_CTRL = int(np.argmax(np.where(_eligible, _v, -np.inf))); CTRL_SOURCE = "q8_rule_at_lesion_block"

_mismatch = []
if CH1 != int(Q8.expected_ch1):
    _mismatch.append(f"c* computed {CH1}, expected {Q8.expected_ch1}")
if CH2 != int(Q8.expected_ch2):
    _mismatch.append(f"c2 computed {CH2}, expected {Q8.expected_ch2}")
if _q6dec is not None and (int(_q6dec.get("ch1", CH1)) != CH1 or int(_q6dec.get("ch2", CH2)) != CH2):
    _mismatch.append(f"Q6 decision has ch1={_q6dec.get('ch1')} ch2={_q6dec.get('ch2')}")
if _mismatch and not Q8.allow_ch_mismatch:
    raise RuntimeError("channel-identity tripwire: " + "; ".join(_mismatch) +
                       ". Inspect MED_PROF; if the computed identities are right, set allow_ch_mismatch=True.")
elif _mismatch:
    print(f"[q8-decision][OVERRIDE] {_mismatch} -- proceeding with computed identities.")
if B_LESION < L_STAR:
    raise RuntimeError(f"lesion block {B_LESION} precedes the analysis block {L_STAR}: the register "
                       f"identity rule (argmax at the lesion block) is only validated from l* on")

# ------------------------------------------------- v* (earlier frozen vector or refit)
def fit_vstar(vectors, ch1=None):
    """Fit v*. Unit-normalize the rows, take the top
    right-singular vector without centering, and orient it so the mean
    projection is positive."""
    U = np.asarray(vectors, np.float64)
    assert U.ndim == 2 and U.shape[0] >= 2, f"need >= 2 rows, got {U.shape}"
    nrm = np.linalg.norm(U, axis=1, keepdims=True)
    keep = nrm[:, 0] > 0
    U = U[keep] / nrm[keep]
    _, s, Vt = np.linalg.svd(U, full_matrices=False)
    v = Vt[0]
    if float((U @ v).mean()) < 0:
        v = -v
    info = {"n_rows": int(U.shape[0]),
            "top_singular_energy": float(s[0] ** 2 / max((s ** 2).sum(), 1e-12)),
            "mean_cos_rows": float((U @ v).mean()), "min_cos_rows": float((U @ v).min())}
    if ch1 is not None:
        info["ch1"] = int(ch1)
        info["energy_on_ch1"] = float(v[int(ch1)] ** 2)
        info["abs_cos_with_e_ch1"] = float(abs(v[int(ch1)]))
    return v.astype(np.float32), info

def _load_vstar_file(path):
    with np.load(path, allow_pickle=False) as z:
        return z["vstar"].astype(np.float32), json.loads(str(z["info_json"].item()))

def _refit_vstar_from_v4():
    rows, used, missing = [], [], []
    for sid, pid, seed, text in SCENARIOS:
        tp = scenario_paths(PATHS["main"], sid, True, L_STAR, T_STAR)["tensor"]
        if not os.path.exists(tp):
            missing.append(sid); continue
        X = np.load(tp).astype(np.float32)
        if X.shape != (N_IMG, CFG.d_model):
            missing.append(sid); continue
        nrm = np.linalg.norm(X, axis=1)
        med = max(float(np.median(nrm)), 1e-9)
        cand = np.where(nrm >= Q8.vstar_norm_mult * med)[0]
        if len(cand) == 0:
            continue
        cand = cand[np.argsort(-nrm[cand])][:int(Q8.vstar_max_per_scenario)]
        rows.append(X[cand]); used.append(sid)
        del X
    if len(used) < int(Q8.vstar_min_scenarios):
        raise RuntimeError(f"v* refit needs >= {Q8.vstar_min_scenarios} scenarios with a stored v4 full "
                           f"tensor and a high-norm token; found {len(used)} (missing {len(missing)}). "
                           f"Either complete the v4 main run with save_full_tensor=True or copy the Q6 "
                           f"file {os.path.basename(Q6_VSTAR_PATH)} into meta/.")
    v, info = fit_vstar(np.vstack(rows), ch1=CH1)
    h = len(rows) // 2
    va, _ = fit_vstar(np.vstack(rows[:h]), ch1=CH1)
    vb, _ = fit_vstar(np.vstack(rows[h:]), ch1=CH1)
    info.update({"n_scenarios": len(used), "n_missing": len(missing), "l_star": L_STAR, "t_star": T_STAR,
                 "ch1": CH1, "norm_mult": Q8.vstar_norm_mult, "max_per_scenario": Q8.vstar_max_per_scenario,
                 "split_half_abs_cos": round(float(abs(va @ vb)), 6), "source": "refit from v4 main full tensors at (l*, t*)"})
    return v, info

if os.path.exists(Q8_VSTAR_PATH) and not Q8.force_refit_vstar:
    VSTAR, VSTAR_INFO = _load_vstar_file(Q8_VSTAR_PATH)
    assert VSTAR_INFO.get("ch1") == CH1 and VSTAR_INFO.get("l_star") == L_STAR and VSTAR_INFO.get("t_star") == T_STAR, \
        "stored v* was fitted under a different decision; set force_refit_vstar=True"
    print(f"[vstar] loaded {Q8_VSTAR_PATH} (sha {VSTAR_INFO['vstar_sha']}; {VSTAR_INFO.get('source')})")
else:
    if os.path.exists(Q6_VSTAR_PATH) and Q8.vstar_source == "q6_frozen_or_refit":
        VSTAR, VSTAR_INFO = _load_vstar_file(Q6_VSTAR_PATH)
        assert VSTAR_INFO.get("ch1") == CH1 and VSTAR_INFO.get("l_star") == L_STAR \
            and VSTAR_INFO.get("t_star") == T_STAR, "Q6 v* was fitted under a different decision"
        VSTAR_INFO = dict(VSTAR_INFO, source=f"Q6 frozen direction ({os.path.basename(Q6_VSTAR_PATH)}, sha {VSTAR_INFO.get('vstar_sha')})")
        print(f"[vstar] reusing the Q6 frozen direction ({VSTAR_INFO['source']})")
    else:
        VSTAR, VSTAR_INFO = _refit_vstar_from_v4()
        print(f"[vstar] refitted from {VSTAR_INFO['n_rows']} register rows of {VSTAR_INFO['n_scenarios']} scenarios")
    VSTAR = VSTAR.astype(np.float32)
    VSTAR_INFO["energy_on_ch2"] = float(VSTAR[CH2] ** 2)
    VSTAR_INFO["top5_channels_by_energy"] = [int(c) for c in np.argsort(-VSTAR ** 2)[:5]]
    VSTAR_INFO["top5_energy"] = [round(float(VSTAR[c] ** 2), 5) for c in np.argsort(-VSTAR ** 2)[:5]]
    VSTAR_INFO["vstar_sha"] = hashlib.sha256(VSTAR.tobytes()).hexdigest()[:16]
    VSTAR_INFO["config_hash"] = CONFIG_HASH
    VSTAR_INFO["timestamp"] = datetime.datetime.now().isoformat(timespec="seconds")
    save_npz_atomic(Q8_VSTAR_PATH, vstar=VSTAR, info_json=np.array(json.dumps(VSTAR_INFO)))
    print(f"[vstar] frozen -> {Q8_VSTAR_PATH} (sha {VSTAR_INFO['vstar_sha']})")
VSTAR_SHA = VSTAR_INFO["vstar_sha"]
assert VSTAR.shape == (CFG.d_model,) and abs(float(np.linalg.norm(VSTAR)) - 1.0) < 1e-4, "v* must be a unit vector"
print(f"[vstar] |cos(v*, e_{CH1})| = {VSTAR_INFO['abs_cos_with_e_ch1']:.4f} (energy on c* {VSTAR_INFO['energy_on_ch1']:.4f}, "
      f"on c2 {VSTAR_INFO['energy_on_ch2']:.5f}) | top singular energy {VSTAR_INFO['top_singular_energy']:.4f}")
if VSTAR_INFO["abs_cos_with_e_ch1"] < 0.9:
    print(f"[vstar][WARN] v* is not close to the channel-{CH1} axis; inspect top5_channels_by_energy.")

# ------------------------------------------------- freeze the decision, or verify the stored one
_dec_core = {"ch1": CH1, "ch2": CH2, "ch_ctrl": CH_CTRL, "ctrl_source": CTRL_SOURCE,
             "l_star": L_STAR, "t_star": T_STAR, "band": [_BAND_LO, _BAND_HI],
             "b_lesion": B_LESION, "b_rescue": B_RESCUE, "b_maint_end": B_MAINT_END,
             "primary_probe": PRIMARY_PROBE, "probe_blocks": PROBE_BLOCKS_L,
             "state_ref_blocks": [int(b) for b in STATE_REF_BLOCKS],
             "intervention_steps": [int(s) for s in INT_STEPS], "n_targets": int(Q8.n_targets),
             "vstar_sha": VSTAR_SHA}
Q8_DEC_SHA = sha256_of_text(json.dumps(_dec_core, sort_keys=True))
if os.path.exists(Q8_DEC_PATH) and not Q8.force_redecide:
    with open(Q8_DEC_PATH) as _f:
        _prev = json.load(_f)
    if _prev.get("dec_sha") != Q8_DEC_SHA:
        raise RuntimeError(f"{os.path.basename(Q8_DEC_PATH)} exists but disagrees with the recomputed decision "
                           f"(stored {_prev.get('dec_sha')}, recomputed {Q8_DEC_SHA}). Set force_redecide=True "
                           f"to overwrite; finished runs are then invalidated and redone.")
    Q8_DEC = _prev
    print(f"[q8-decision] decision file verified against recomputation (sha {Q8_DEC_SHA})")
else:
    Q8_DEC = dict(_dec_core)
    Q8_DEC.update({"dec_sha": Q8_DEC_SHA,
                   "rules": {"c_star": "largest median mean|x| at l* (pilot, t*)",
                             "c2": "largest non-c* channel at the band end",
                             "ctrl": ("Q6 control channel" if CTRL_SOURCE == "q6_decision" else
                                      f"largest channel at the lesion block below {Q8.ctrl_exclude_mult:g} x median, c*/c2 excluded"),
                             "vstar": VSTAR_INFO.get("source")},
                   "ch1_curve": [round(float(x), 4) for x in MED_PROF[:, CH1]],
                   "ch2_curve": [round(float(x), 4) for x in MED_PROF[:, CH2]],
                   "ctrl_curve": [round(float(x), 4) for x in MED_PROF[:, CH_CTRL]],
                   "pilot_sids": PILOT_SIDS, "pilot_config_hash": PILOT_DECISION.get("config_hash"),
                   "precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH, "q8_hash": Q8_HASH,
                   "timestamp": datetime.datetime.now().isoformat(timespec="seconds")})
    save_json_atomic(Q8_DEC, Q8_DEC_PATH)
    print(f"[q8-decision] decision frozen -> {Q8_DEC_PATH} (sha {Q8_DEC_SHA})")

TRACKED_CH = [CH1, CH2, CH_CTRL]
TRACKED_CH_NAMES = ["cstar", "c2", "ctrl"]
TRACKED_CH_LABELS = {"cstar": f"dominant ch. {CH1} (c*)", "c2": f"massive-channel control ch. {CH2}",
                     "ctrl": f"ordinary control ch. {CH_CTRL}"}
COND_LABELS = {c["name"]: condition_label(c, CH1, CH2, CH_CTRL) for c in CONDITIONS}
COND_LABELS_SHORT = {c["name"]: condition_label(c, CH1, CH2, CH_CTRL, short=True) for c in CONDITIONS}
FAMILY_LABELS = {f: _fam_words(f, CH1, CH2, CH_CTRL) for f in LESION_KINDS}
FAMILY_TITLES = {"cstar_zero": f"c* suppression (ch. {CH1} set to 0)", "vstar_destroy": "v* destruction (norm kept)",
                 "vstar_random_dir": "direction randomization (norm kept)", "reg_remove": "register removal (mean ordinary token)"}
VSTAR_T = torch.tensor(VSTAR, dtype=torch.float32)

# ------------------------------------------------- decision figure
fig, ax = plt.subplots(figsize=(FIG_COL, 2.3))
_x = np.arange(_nb)
mark_blocks(ax, first=True)
ax.semilogy(_x, MED_PROF[:, CH1], color=OKABE_ITO[0], lw=1.2, label=f"dominant ch. {CH1} (c*)")
ax.semilogy(_x, MED_PROF[:, CH2], color=OKABE_ITO[3], lw=1.1, label=f"massive-channel control ch. {CH2}")
ax.semilogy(_x, MED_PROF[:, CH_CTRL], color=OKABE_ITO[2], lw=1.0, ls="--", label=f"ordinary control ch. {CH_CTRL}")
ax.semilogy(_x, np.median(MED_PROF, axis=1), color="0.6", lw=0.8, label="median channel")
ax.set_xlabel("transformer block")
ax.set_ylabel("mean |activation| over image tokens")
save_paper_fig(fig, q8_fig_stem("R2_channels_and_blocks"), legend_ax=ax, ncol=2,
    title=(f"Depth profiles of the dominant channel {CH1} and the control channels {CH2} and {CH_CTRL}, with the pre-registered "
           f"lesion block {B_LESION} and rescue block {B_RESCUE} ({MODEL_LABEL}, pilot median, step {T_STAR})"),
    caption=(f"Depth profile of the channels used in the Q8 design for {MODEL_LABEL} (median over the five pilot "
             f"scenarios of the per-block mean absolute activation over image tokens, step {T_STAR}): the dominant "
             f"channel {CH1} that carries v* ({VSTAR_INFO['energy_on_ch1']:.3f} of its energy), the massive-channel "
             f"control {CH2}, the ordinary control channel {CH_CTRL}, and the median channel. The shaded region is "
             f"the high-norm band, the dotted line the lesion block {B_LESION}, the dash-dotted line the rescue "
             f"block {B_RESCUE}; the first attention call that sees a rescued residual is block {PRIMARY_PROBE}."))
print("=" * 70)
print("Q8 DECISION SUMMARY")
print("=" * 70)
print(f"  c* = {CH1} | massive-channel control c2 = {CH2} | ordinary control = {CH_CTRL} ({CTRL_SOURCE})")
print(f"  lesion block {B_LESION} -> rescue block {B_RESCUE} -> primary probe {PRIMARY_PROBE}; maintained windows end at {B_MAINT_END}")
print(f"  intervention steps {INT_STEPS} (t* = {T_STAR}) | v* sha {VSTAR_SHA} ({VSTAR_INFO.get('source')})")
print("=" * 70)


## R3. Operators, attention probe, replay wrapper and unit tests

The residual operators edit chosen tokens of the image-stream block output.
The lesions zero c\*, remove v\* at constant norm, randomize the direction,
or replace the register with the mean ordinary token. The rescues restore
the v\* projection, one channel, the norm or the full clean vector, and the
sham writes the clean vector back. Each operator checks its own result in
the model dtype.

The attention probe wraps the scaled-dot-product-attention call. It can
swap the register's key or value for the clean one before the kernel runs,
it captures keys and values, and it computes per-head routing statistics
over image queries and image keys (key rank, incoming share, top sinks and
distance to the clean routing). The replay wrapper returns the baseline's
cached transformer outputs for the steps before the intervened step.

The unit tests run on small CPU tensors and must pass before any model
compute. They cover both attention layouts, bf16 values at register
magnitude, a brute-force attention reference and replay exactness.


In [ ]:
# =====================================================================
# R3. Operators, attention probe, replay wrapper and unit tests
#
# The residual operators edit chosen image tokens of a block output
# [B, N, D] (every batch row) and return the modified clone plus a
# violation count. Each operator re-checks its own result in the model
# dtype (bf16 tolerance), and the count has to be zero.
#
#   cstar_zero         y[t, c*] = 0
#   vstar_destroy      y = ||x|| (x - <x,v*> v*) / ||x - <x,v*> v*||
#   vstar_random_dir   y = ||x|| g, with g a unit direction orthogonal to v*
#   reg_remove         y[t] = mean of the ordinary tokens (norm < mult x median)
#   vstar (frac f)     y = x - <x,v*> v* + f alpha_clean v*
#   cstar, chan        y[t, c] = clean value of channel c
#   norm               y = x ||x_clean|| / ||x||
#   full               y[t] = x_clean
#   sham               y = stored clean x, must equal x bit for bit
#
# The attention probe wraps torch.nn.functional.scaled_dot_product_attention.
# For the active block it can replace the key and/or value rows of chosen
# image tokens before the kernel runs (the natural-key rescue), captures the
# keys and values of the tracked tokens, and computes per-head statistics
# over image queries and image keys only. These are each key's rank and
# cosine against the mean image query, the incoming attention per key
# relative to uniform (softmax summed over queries, so 1 means a uniform
# share), the top sinks, and the total-variation and Jensen-Shannon
# distance to the clean run's incoming attention at the same block.
#
# The replay wrapper replaces the transformer's forward. Steps before the
# intervened step return the baseline's cached output (bit-exact
# trajectory, no block compute), and later steps run normally. It also
# numbers the steps for the recorder and stores the input and output at
# the intervened step.
# =====================================================================
import math, types
import numpy as np
import torch
import torch.nn.functional as F

try:
    from diffusers.models.modeling_outputs import Transformer2DModelOutput as _T2DOut
except Exception:                       # older diffusers, fall back to SimpleNamespace
    _T2DOut = None

# ---------------------------------------------------------------- helpers
def _rel_tol(t, rel=1e-2, abs_=1e-3):
    return t.abs() * rel + abs_

def modify_block_output(output, n_img, n_txt, fn):
    """Apply fn to the image-stream region of a block output and rebuild the
    output structure. The image stream is found by sequence length only.
    Returns (new_output, found)."""
    if torch.is_tensor(output) and output.dim() == 3:
        L = output.shape[1]
        if L == n_img:
            return fn(output), True
        if n_txt and L == n_img + n_txt:
            y = output.clone()
            y[:, n_txt:, :] = fn(output[:, n_txt:, :])
            return y, True
        return output, False
    if isinstance(output, (tuple, list)):
        out = list(output)
        for j, o in enumerate(out):
            if o is not None and torch.is_tensor(o) and o.dim() == 3 and o.shape[1] == n_img:
                out[j] = fn(o)
                return (tuple(out) if isinstance(output, tuple) else out), True
        return output, False
    return output, False

def extract_image_region(output, n_img, n_txt):
    """The image-stream region [B, n_img, D] of a block output (a view)."""
    if torch.is_tensor(output) and output.dim() == 3:
        if output.shape[1] == n_img:
            return output
        if n_txt and output.shape[1] == n_img + n_txt:
            return output[:, n_txt:, :]
    if isinstance(output, (tuple, list)):
        for o in output:
            if o is not None and torch.is_tensor(o) and o.dim() == 3 and o.shape[1] == n_img:
                return o
    raise RuntimeError("image stream not found in block output")

def _rows_of(vals, B):
    """Clean-value arrays carry a batch axis. Broadcast a single row to B."""
    v = vals
    if v.shape[0] == B:
        return v
    if v.shape[0] == 1:
        return v.expand(B, *v.shape[1:])
    raise RuntimeError(f"clean values have batch {v.shape[0]}, region has {B}")

# ------------------------------------------------------ residual operators
def apply_residual_op(region, toks, kind, vstar=None, clean_vec=None, clean_scalar=None,
                      channel=None, frac=1.0, rand_dirs=None, ordinary_mask=None):
    """region [B, N, D], toks LongTensor [K] on region.device.
    clean_vec [B or 1, K, D] for full, clean_scalar [B or 1, K] for alpha,
    a channel value or the norm.
    Returns (modified clone, violation count as a 0-dim tensor)."""
    y = region.clone()
    B, N, D = region.shape
    x = region[:, toks, :].to(torch.float32)                         # [B, K, D]
    K = int(toks.numel())
    if kind == "sham":
        # exact-rewrite sham. The stored clean vector (a float32 copy of the
        # model-dtype values) is written back and must reproduce the current
        # values bit for bit, which shows the clean values used by the rescues
        # are stored without loss. Without them it falls back to a self-copy.
        if clean_vec is not None:
            xc = _rows_of(clean_vec, B)
            y[:, toks, :] = xc.to(y.dtype)
        else:
            y[:, toks, :] = region[:, toks, :]
        viol = (y != region).sum()
    elif kind == "cstar_zero":
        y[:, toks, channel] = 0
        viol = (y[:, toks, channel] != 0).sum()
    elif kind in ("vstar_destroy", "vstar_random_dir"):
        v = vstar.to(torch.float32)
        n = x.norm(dim=-1, keepdim=True)                             # [B, K, 1]
        if kind == "vstar_destroy":
            xp = x - (x @ v)[..., None] * v
            d = xp / torch.clamp(xp.norm(dim=-1, keepdim=True), min=1e-12)
        else:
            g = rand_dirs.to(torch.float32)                          # [K, D] unit, orthogonal to v*
            d = g[None].expand(B, K, D)
        z = n * d
        y[:, toks, :] = z.to(y.dtype)
        zz = y[:, toks, :].to(torch.float32)
        # allow |cos(y, v*)| up to 1e-2. Rounding a register-sized vector to
        # the model dtype (bf16 spacing is 256 near 5e4) limits how exactly
        # the projection can vanish.
        viol = (((zz @ v).abs() > 1e-2 * zz.norm(dim=-1) + 1e-3).sum()
                + ((zz.norm(dim=-1, keepdim=True) - n).abs() > _rel_tol(n)).sum())
    elif kind == "reg_remove":
        m_all = []
        for b in range(B):
            om = ordinary_mask[b] if ordinary_mask.dim() == 2 else ordinary_mask
            m_all.append(region[b][om].to(torch.float32).mean(dim=0))
        m = torch.stack(m_all)                                       # [B, D]
        y[:, toks, :] = m[:, None, :].expand(B, K, D).to(y.dtype)
        zz = y[:, toks, :].to(torch.float32)
        viol = ((zz - m[:, None, :]).abs() > _rel_tol(m[:, None, :])).sum()
    elif kind == "vstar":
        v = vstar.to(torch.float32)
        a_clean = _rows_of(clean_scalar.to(torch.float32), B)       # [B, K]
        a = (x @ v)                                                  # [B, K]
        z = x - a[..., None] * v + (float(frac) * a_clean)[..., None] * v
        y[:, toks, :] = z.to(y.dtype)
        zz = y[:, toks, :].to(torch.float32)
        a_new = zz @ v
        target = float(frac) * a_clean
        viol = ((a_new - target).abs() > _rel_tol(target)).sum()
        xp_old = x - a[..., None] * v
        xp_new = zz - a_new[..., None] * v
        # x_perp unchanged up to model-dtype rounding of the rebuilt vector (5 percent of ||x_perp||)
        viol = viol + ((xp_new - xp_old).norm(dim=-1) > 5e-2 * xp_old.norm(dim=-1) + 1e-2).sum()
    elif kind in ("cstar", "chan"):
        val = _rows_of(clean_scalar.to(torch.float32), B)           # [B, K]
        y[:, toks, channel] = val.to(y.dtype)
        viol = ((y[:, toks, channel].to(torch.float32) - val).abs() > _rel_tol(val)).sum()
    elif kind == "norm":
        n_clean = _rows_of(clean_scalar.to(torch.float32), B)       # [B, K]
        n = x.norm(dim=-1)
        z = x * (n_clean / torch.clamp(n, min=1e-12))[..., None]
        y[:, toks, :] = z.to(y.dtype)
        zz = y[:, toks, :].to(torch.float32)
        cos = (zz * x).sum(-1) / (zz.norm(dim=-1) * x.norm(dim=-1) + 1e-12)
        viol = ((cos < 1.0 - 1e-4).sum() + ((zz.norm(dim=-1) - n_clean).abs() > _rel_tol(n_clean)).sum())
    elif kind == "full":
        xc = _rows_of(clean_vec.to(torch.float32), B)               # [B, K, D]
        y[:, toks, :] = xc.to(y.dtype)
        viol = ((y[:, toks, :].to(torch.float32) - xc).abs() > _rel_tol(xc)).sum()
    else:
        raise ValueError(f"unknown residual operator {kind!r}")
    return y, viol

def orthogonal_random_dirs(vstar_np, k, seed):
    """k unit directions orthogonal to v* (seeded)."""
    rng = np.random.default_rng(int(seed))
    v = np.asarray(vstar_np, np.float64); v = v / np.linalg.norm(v)
    out = np.zeros((k, v.size), np.float32)
    for i in range(k):
        g = rng.normal(size=v.size)
        g = g - (g @ v) * v
        out[i] = (g / np.linalg.norm(g)).astype(np.float32)
    return out

# ---------------------------------------------------------- geometry readout
GEOM_TOKEN_FIELDS = ("alpha", "cos", "xperp", "norm")
GEOM_POP_FIELDS = ("med_cos_all", "p99_cos_all", "max_cos_all", "argmax_cos_tok", "n_cos_ge_thr",
                   "med_norm", "amax_tok", "amax_norm")

def geometry_readout(X, vstar, toks, thr=0.9):
    """X [N, D] fp32 (one row), vstar [D] unit, toks LongTensor [K].
    Returns tok_fields [K, 4] and pop_fields [8] as float32 tensors."""
    proj = X @ vstar
    norms = X.norm(dim=1)
    cos = proj / torch.clamp(norms, min=1e-12)
    k99 = max(1, int(math.ceil(0.99 * X.shape[0])))
    xr = X[toks]                                                     # [K, D]
    a = proj[toks]
    xp = (xr - a[:, None] * vstar[None, :]).norm(dim=1)
    tok = torch.stack([a, cos[toks], xp, norms[toks]], dim=1)
    amax = torch.argmax(norms)
    pop = torch.stack([cos.median(), cos.kthvalue(k99).values, cos.max(), cos.argmax().to(torch.float32),
                       (cos >= thr).sum().to(torch.float32), norms.median(), amax.to(torch.float32), norms[amax]])
    return tok.to(torch.float32), pop.to(torch.float32)

# ------------------------------------------------------------ attention probe
PROBE_TOKEN_FIELDS = ("key_rank", "key_cos", "sink_rel", "is_top_sink")
PROBE_HEAD_FIELDS = ("med_key_cos", "top1_idx", "top1_rel", "top2_idx", "top2_rel", "top3_idx", "top3_rel",
                     "tv_to_clean", "js_to_clean")

def routing_distances(rel, rel_clean):
    """rel, rel_clean [H, N] (sum N per head) -> (tv [H], js [H]) in [0, 1]."""
    p = rel / torch.clamp(rel.sum(dim=1, keepdim=True), min=1e-12)
    q = rel_clean / torch.clamp(rel_clean.sum(dim=1, keepdim=True), min=1e-12)
    tv = 0.5 * (p - q).abs().sum(dim=1)
    m = 0.5 * (p + q)
    def _kl(a, b):
        mask = a > 0
        return torch.where(mask, a * (torch.log2(torch.clamp(a, min=1e-30)) - torch.log2(torch.clamp(b, min=1e-30))),
                           torch.zeros_like(a)).sum(dim=1)
    js = 0.5 * _kl(p, m) + 0.5 * _kl(q, m)
    return tv, js.clamp(min=0.0)

class AttentionProbe:
    """Wraps the attention call (see the cell header). `active` is the block
    whose image self-attention call is captured, `patch_spec` says which
    key/value rows to replace before the kernel runs, `capture_toks` are the
    tokens whose keys and values are recorded, and `clean_rel` [H, N] is the
    routing reference."""

    def __init__(self, n_img, n_txt, select_row=0, chunk=256, n_heads_expected=None):
        self.n_img, self.n_txt = int(n_img), int(n_txt)
        self.select_row, self.chunk = int(select_row), int(chunk)
        self.n_heads_expected = n_heads_expected
        self.active = None
        self.patch_spec = None            # dict(kind, pos [K], k_src [B,H,K,Dh] or None, v_src ...)
        self.capture_toks = None          # LongTensor [K] or None
        self.clean_rel = None             # [H, N] or None
        self.full = True                  # compute the full routing statistics
        self.stash = None
        self.fires = 0; self.ignored = 0; self.errors = []
        self._orig = None

    def patch(self):
        assert self._orig is None, "probe already patched"
        self._orig = F.scaled_dot_product_attention
        probe = self

        def wrapped(*args, **kwargs):
            if probe.active is None:
                return probe._orig(*args, **kwargs)
            try:
                q = kwargs["query"] if "query" in kwargs else args[0]
                k = kwargs["key"] if "key" in kwargs else args[1]
                v = kwargs["value"] if "value" in kwargs else args[2]
                if not probe._is_image_self_attention(q, k):
                    probe.ignored += 1
                    return probe._orig(*args, **kwargs)
                k2, v2 = probe._apply_patch(k, v)
                if k2 is not k or v2 is not v:
                    args = list(args); kw = dict(kwargs)
                    if "key" in kw: kw["key"] = k2
                    else: args[1] = k2
                    if "value" in kw: kw["value"] = v2
                    else: args[2] = v2
                    args = tuple(args); kwargs = kw
                out = probe._orig(*args, **kwargs)
                probe.record(q, k2, v2, kwargs.get("scale", None))
                return out
            except Exception as e:
                probe.errors.append(f"probe block {probe.active}: {type(e).__name__}: {e}")
                return probe._orig(*args, **kwargs)
        F.scaled_dot_product_attention = wrapped
        return self

    def unpatch(self):
        if self._orig is not None:
            F.scaled_dot_product_attention = self._orig
            self._orig = None

    def _is_image_self_attention(self, q, k):
        if not (torch.is_tensor(q) and torch.is_tensor(k) and q.dim() == 4 and k.dim() == 4):
            return False
        S = k.shape[2]
        if S == self.n_img and q.shape[2] == S:
            return True
        return bool(self.n_txt and S == self.n_img + self.n_txt and q.shape[2] == S)

    def _lo(self, k):
        return 0 if k.shape[2] == self.n_img else self.n_txt

    def _apply_patch(self, k, v):
        spec = self.patch_spec
        if spec is None:
            return k, v
        lo = self._lo(k)
        pos = spec["pos"].to(k.device)
        kind = spec["kind"]
        k2, v2 = k, v
        if kind == "sham":
            if spec.get("k_src") is None:
                k2 = k.clone(); k2[:, :, lo + pos, :] = k[:, :, lo + pos, :]
                return k2, v
            kind = "key_value"          # exact-rewrite sham, writes the stored clean key and value back
        if kind in ("key", "key_value"):
            src = spec["k_src"].to(k.device, k.dtype)                 # [B or 1, H, K, Dh]
            k2 = k.clone(); k2[:, :, lo + pos, :] = _rows_of(src, k.shape[0])
        if kind in ("value", "key_value"):
            src = spec["v_src"].to(v.device, v.dtype)
            v2 = v.clone(); v2[:, :, lo + pos, :] = _rows_of(src, v.shape[0])
        return k2, v2

    @torch.no_grad()
    def record(self, q, k, v, scale=None):
        lo = self._lo(k)
        st = {"H": int(q.shape[1]), "N": self.n_img}
        if self.capture_toks is not None:
            toks = self.capture_toks.to(k.device)
            st["k_cap"] = k[:, :, lo + toks, :].to(torch.float32)     # [B, H, K, Dh]
            st["v_cap"] = v[:, :, lo + toks, :].to(torch.float32)
        if self.full:
            row = self.select_row if q.shape[0] > self.select_row else 0
            qi = q[row, :, lo:, :].to(torch.float32)                 # [H, N, Dh]
            ki = k[row, :, lo:, :].to(torch.float32)
            H, N, Dh = qi.shape
            sc = float(scale) if scale is not None else Dh ** -0.5
            qbar = qi.mean(dim=1)                                    # [H, Dh]
            s = torch.einsum("hnd,hd->hn", ki, qbar)
            cos_k = s / (ki.norm(dim=-1) * qbar.norm(dim=-1, keepdim=True) + 1e-12)
            rel = torch.zeros(H, N, dtype=torch.float32, device=qi.device)
            for a in range(0, N, self.chunk):
                logits = torch.einsum("hcd,hnd->hcn", qi[:, a:a + self.chunk, :], ki) * sc
                rel += torch.softmax(logits, dim=-1).sum(dim=1)
            st.update(s=s, cos_k=cos_k, rel=rel)
        self.stash = st
        self.fires += 1

    def consume(self, toks):
        """toks LongTensor [K] -> dict with tok [H, K, 4], head [H, 9],
        rel [H, N] or None, k_cap and v_cap. Clears the stash."""
        st = self.stash
        self.stash = None
        if st is None:
            return None
        out = {"H": st["H"], "k_cap": st.get("k_cap"), "v_cap": st.get("v_cap"),
               "tok": None, "head": None, "rel": None}
        if "rel" in st:
            s, cos_k, rel = st["s"], st["cos_k"], st["rel"]
            H, N = rel.shape
            toks = toks.to(rel.device)
            top_rel, top_idx = torch.topk(rel, 3, dim=1)                 # [H, 3]
            rank = 1 + (s[:, :, None] > s[:, toks][:, None, :]).sum(dim=1).to(torch.float32)   # [H, K]
            is_top = (top_idx[:, :1] == toks[None, :]).to(torch.float32)
            tok = torch.stack([rank, cos_k[:, toks], rel[:, toks], is_top], dim=2)          # [H, K, 4]
            if self.clean_rel is not None:
                tv, js = routing_distances(rel, self.clean_rel.to(rel.device, torch.float32))
            else:
                tv = torch.full((H,), float("nan"), device=rel.device); js = tv.clone()
            head = torch.stack([cos_k.median(dim=1).values,
                                top_idx[:, 0].to(torch.float32), top_rel[:, 0],
                                top_idx[:, 1].to(torch.float32), top_rel[:, 1],
                                top_idx[:, 2].to(torch.float32), top_rel[:, 2], tv, js], dim=1)
            out.update(tok=tok.to(torch.float32), head=head.to(torch.float32), rel=rel)
        return out

# ------------------------------------------------------------- replay wrapper
class TransformerReplay:
    """Wraps transformer.forward. Numbers the steps, replays cached outputs
    for steps < n_replay, and records the input and output at chosen steps."""

    def __init__(self, transformer, cache=None, n_replay=0, record_steps=(), on_step=None,
                 cache_steps=None):
        self.tr = transformer
        self.cache = cache                    # list/array of tensors [n_replay, ...] (model dtype)
        self.n_replay = int(n_replay)
        self.record_steps = set(int(s) for s in record_steps)
        self.cache_steps = None if cache_steps is None else set(int(s) for s in cache_steps)
        self.on_step = on_step
        self.step = 0
        self.inputs, self.outputs, self.cached = {}, {}, {}
        self._orig = None; self._had_attr = False

    def install(self):
        assert self._orig is None
        self._had_attr = "forward" in self.tr.__dict__
        self._orig = self.tr.forward
        wrapper = self

        def forward(*args, **kwargs):
            step = wrapper.step
            wrapper.step += 1
            if wrapper.on_step is not None:
                wrapper.on_step(step)
            return_dict = kwargs.get("return_dict", True)
            if step < wrapper.n_replay:
                if wrapper.cache is None or step >= len(wrapper.cache):
                    raise RuntimeError(f"replay requested at step {step} but no cached output")
                x = kwargs["hidden_states"] if "hidden_states" in kwargs else args[0]
                out = wrapper.cache[step].to(x.device)
                if return_dict:
                    return _T2DOut(sample=out) if _T2DOut is not None else types.SimpleNamespace(sample=out)
                return (out,)
            out = wrapper._orig(*args, **kwargs)
            o = out[0] if isinstance(out, (tuple, list)) else getattr(out, "sample", out)
            if wrapper.cache_steps is not None and step in wrapper.cache_steps:
                wrapper.cached[step] = o.detach().clone()
            if step in wrapper.record_steps:
                x = kwargs["hidden_states"] if "hidden_states" in kwargs else args[0]
                wrapper.inputs[step] = x.detach().to("cpu", torch.float32).clone()
                wrapper.outputs[step] = o.detach().to("cpu", torch.float32).clone()
            return out
        self.tr.forward = forward
        return self

    def uninstall(self):
        if self._orig is None:
            return
        if self._had_attr:
            self.tr.forward = self._orig
        else:
            try:
                del self.tr.forward
            except AttributeError:
                pass
        self._orig = None

def tensor_to_bits(t):
    """Exact storage of bf16/fp16 tensors as int16 bit patterns (numpy)."""
    if t.dtype in (torch.bfloat16, torch.float16):
        return t.detach().contiguous().view(torch.int16).cpu().numpy(), str(t.dtype).replace("torch.", "")
    return t.detach().to("cpu", torch.float32).numpy(), "float32"

def bits_to_tensor(a, dtype_name):
    t = torch.from_numpy(np.ascontiguousarray(a))
    if dtype_name in ("bfloat16", "float16"):
        return t.view(getattr(torch, dtype_name))
    return t

# ------------------------------------------------------------ unit tests
def _run_q8_unit_tests():
    g = torch.Generator().manual_seed(0)
    B, N, NT, D = 2, 12, 5, 8
    vs = torch.zeros(D); vs[3] = 0.995; vs[1] = math.sqrt(1 - 0.995 ** 2)
    x = torch.randn(B, N, D, generator=g)
    toks = torch.tensor([7, 2])
    others = [t for t in range(N) if t not in toks.tolist()]
    x[:, 7, :] += 50.0 * vs
    x[:, 2, :] += 30.0 * vs

    # lesions
    y, v = apply_residual_op(x, toks, "cstar_zero", channel=3)
    assert v.item() == 0 and (y[:, toks, 3] == 0).all() and torch.equal(y[:, others], x[:, others])
    assert not (x[:, toks, 3] == 0).all(), "operator must not modify its input"
    y, v = apply_residual_op(x, toks, "vstar_destroy", vstar=vs)
    assert v.item() == 0
    assert ((y[:, toks, :] @ vs).abs() < 1e-3).all() and torch.allclose(y[:, toks, :].norm(dim=-1), x[:, toks, :].norm(dim=-1), rtol=1e-5)
    yb, vb = apply_residual_op(x.to(torch.bfloat16), toks, "vstar_destroy", vstar=vs)
    assert vb.item() == 0 and yb.dtype == torch.bfloat16
    rd = torch.tensor(orthogonal_random_dirs(vs.numpy(), 2, seed=1))
    assert (rd @ vs).abs().max() < 1e-5 and torch.allclose(rd.norm(dim=1), torch.ones(2))
    y, v = apply_residual_op(x, toks, "vstar_random_dir", vstar=vs, rand_dirs=rd)
    assert v.item() == 0 and torch.allclose(y[:, toks, :].norm(dim=-1), x[:, toks, :].norm(dim=-1), rtol=1e-5)
    om = torch.ones(N, dtype=torch.bool); om[toks] = False
    y, v = apply_residual_op(x, toks, "reg_remove", ordinary_mask=om)
    assert v.item() == 0
    for b in range(B):
        assert torch.allclose(y[b, 7], x[b][om].mean(0), atol=1e-5) and torch.allclose(y[b, 2], x[b][om].mean(0), atol=1e-5)

    # rescues, with clean values taken from a "clean" copy
    xc = x.clone(); xl, _ = apply_residual_op(x, toks, "cstar_zero", channel=3)
    a_clean = xc[:, toks, :] @ vs                                                # [B, K]
    y, v = apply_residual_op(xl, toks, "vstar", vstar=vs, clean_scalar=a_clean, frac=1.0)
    assert v.item() == 0 and torch.allclose(y[:, toks, :] @ vs, a_clean, atol=1e-4)
    xp_l = xl[:, toks, :] - (xl[:, toks, :] @ vs)[..., None] * vs
    xp_y = y[:, toks, :] - (y[:, toks, :] @ vs)[..., None] * vs
    assert torch.allclose(xp_l, xp_y, atol=1e-4), "x_perp must be unchanged by the v* rescue"
    y, v = apply_residual_op(xl, toks, "vstar", vstar=vs, clean_scalar=a_clean, frac=0.5)
    assert v.item() == 0 and torch.allclose(y[:, toks, :] @ vs, 0.5 * a_clean, atol=1e-4)
    y, v = apply_residual_op(xl, toks, "cstar", clean_scalar=xc[:, toks, 3], channel=3)
    assert v.item() == 0 and torch.allclose(y[:, toks, 3], xc[:, toks, 3])
    y, v = apply_residual_op(xl, toks, "norm", clean_scalar=xc[:, toks, :].norm(dim=-1))
    assert v.item() == 0 and torch.allclose(y[:, toks, :].norm(dim=-1), xc[:, toks, :].norm(dim=-1), rtol=1e-5)
    c = (y[:, toks, :] * xl[:, toks, :]).sum(-1) / (y[:, toks, :].norm(dim=-1) * xl[:, toks, :].norm(dim=-1))
    assert (c > 1 - 1e-6).all()
    y, v = apply_residual_op(xl, toks, "full", clean_vec=xc[:, toks, :])
    assert v.item() == 0 and torch.allclose(y[:, toks, :], xc[:, toks, :]) and torch.equal(y[:, others], xl[:, others])
    y, v = apply_residual_op(xl, toks, "full", clean_vec=xc[:1, toks, :])      # single-row clean values broadcast
    assert v.item() == 0 and torch.allclose(y[1, toks, :], xc[0, toks, :])
    y, v = apply_residual_op(x, toks, "sham")
    assert v.item() == 0 and torch.equal(y, x) and y.data_ptr() != x.data_ptr()
    yb, vb = apply_residual_op(xl.to(torch.bfloat16), toks, "vstar", vstar=vs, clean_scalar=a_clean, frac=1.0)
    assert vb.item() == 0
    yb, vb = apply_residual_op(xl.to(torch.bfloat16), toks, "full", clean_vec=xc[:, toks, :])
    assert vb.item() == 0

    # bf16 check at register magnitude (registers can reach about 5e4 on c*, where bf16 spacing is 256)
    xb = torch.randn(B, N, D, generator=g) * 300.0
    xb[:, 7, :] += 47000.0 * vs; xb[:, 2, :] += 30000.0 * vs
    xb16 = xb.to(torch.bfloat16)
    for kind, kw in (("vstar_destroy", dict(vstar=vs)), ("cstar_zero", dict(channel=3)),
                     ("norm", dict(clean_scalar=xb[:, toks, :].norm(dim=-1))), ("full", dict(clean_vec=xb[:, toks, :])),
                     ("cstar", dict(clean_scalar=xb[:, toks, 3], channel=3))):
        _, vb = apply_residual_op(xb16, toks, kind, **kw)
        assert vb.item() == 0, f"{kind}: bf16 postcondition failed at register magnitude"
    xl16, _ = apply_residual_op(xb16, toks, "cstar_zero", channel=3)
    _, vb = apply_residual_op(xl16, toks, "vstar", vstar=vs, clean_scalar=(xb[:, toks, :] @ vs), frac=1.0)
    assert vb.item() == 0, "vstar rescue: bf16 postcondition failed at register magnitude"

    # structure helpers
    enc = torch.randn(B, NT, D, generator=g)
    new, found = modify_block_output((enc, x), N, NT, lambda img: apply_residual_op(img, toks, "cstar_zero", channel=3)[0])
    assert found and new[0] is enc and (new[1][:, toks, 3] == 0).all()
    cat = torch.cat([enc, x], dim=1)
    new, found = modify_block_output(cat, N, NT, lambda img: apply_residual_op(img, toks, "cstar_zero", channel=3)[0])
    assert found and torch.equal(new[:, :NT], enc) and (new[:, NT:][:, toks, 3] == 0).all()
    assert extract_image_region((enc, x), N, NT) is x and torch.equal(extract_image_region(cat, N, NT), x)

    # geometry readout
    tokf, pop = geometry_readout(x[0], vs, toks)
    assert tokf.shape == (2, 4) and pop.shape == (8,)
    assert abs(tokf[0, 0].item() - float(x[0, 7] @ vs)) < 1e-4 and tokf[0, 1] > 0.99
    assert int(pop[6].item()) == 7 and int(pop[3].item()) == 7

    # routing distances
    rel = torch.rand(3, N) * 5; relc = rel.clone()
    tv, js = routing_distances(rel, relc)
    assert tv.abs().max() < 1e-6 and js.abs().max() < 1e-6
    rel2 = torch.zeros(3, N); rel2[:, 0] = N
    tv, js = routing_distances(rel2, torch.full((3, N), 1.0))
    assert torch.allclose(tv, torch.full((3,), 1.0 - 1.0 / N), atol=1e-5) and (js > 0.5).all()

    # attention probe against a brute-force reference, both layouts, kwargs and positional calls, scale, patches, capture
    H, Dh = 3, 4
    for (n_txt, S) in ((0, N), (NT, NT + N)):
        pr = AttentionProbe(n_img=N, n_txt=n_txt, select_row=1, chunk=5)
        q = torch.randn(B, H, S, Dh, generator=g)
        k = torch.randn(B, H, S, Dh, generator=g)
        vv = torch.randn(B, H, S, Dh, generator=g)
        tok = 7
        k[:, :, n_txt + tok, :] = q[:, :, n_txt:, :].mean(dim=2) * 40.0        # strong key at token 7
        qi = q[1, :, n_txt:, :]; ki = k[1, :, n_txt:, :]
        A = torch.softmax(torch.einsum("hqd,hkd->hqk", qi, ki) * Dh ** -0.5, dim=-1)
        rel_ref = A.sum(dim=1)
        pr.patch()
        try:
            pr.active = 0; pr.clean_rel = rel_ref.clone(); pr.capture_toks = torch.tensor([tok, 2])
            out = F.scaled_dot_product_attention(query=q, key=k, value=vv)
            pr.active = None
            _ = F.scaled_dot_product_attention(q, k[:, :, :3, :], vv[:, :, :3, :])   # cross-attention shape, left alone
        finally:
            pr.unpatch()
        assert F.scaled_dot_product_attention is pr._orig or pr._orig is None
        assert pr.fires == 1 and pr.ignored == 0 and not pr.errors, (pr.fires, pr.ignored, pr.errors)
        assert torch.allclose(out, torch.nn.functional.scaled_dot_product_attention(q, k, vv))
        res = pr.consume(torch.tensor([tok, 2]))
        assert res["tok"].shape == (H, 2, 4) and res["head"].shape == (H, 9) and res["rel"].shape == (H, N)
        assert torch.allclose(res["rel"], rel_ref, atol=1e-5) and torch.allclose(res["rel"].sum(1), torch.full((H,), float(N)), atol=1e-4)
        assert (res["tok"][:, 0, 0] == 1).all() and (res["tok"][:, 0, 3] == 1).all() and (res["tok"][:, 1, 3] == 0).all()
        assert torch.allclose(res["tok"][:, 0, 2], rel_ref[:, tok], atol=1e-5)
        assert (res["head"][:, 1] == tok).all() and res["head"][:, 7].abs().max() < 1e-6 and res["head"][:, 8].abs().max() < 1e-6
        assert torch.allclose(res["k_cap"][:, :, 0, :], k[:, :, n_txt + tok, :]) and torch.allclose(res["v_cap"][:, :, 1, :], vv[:, :, n_txt + 2, :])
        # key patch. Replacing token 7's key by an ordinary key removes the sink, and the output matches brute force
        k_src = k[:, :, n_txt + 5, :].clone()[:, :, None, :]                       # [B, H, 1, Dh]
        pr.patch()
        try:
            pr.active = 0; pr.clean_rel = rel_ref.clone(); pr.capture_toks = torch.tensor([tok])
            pr.patch_spec = {"kind": "key", "pos": torch.tensor([tok]), "k_src": k_src, "v_src": None}
            out = F.scaled_dot_product_attention(q, k, vv, scale=0.1)
        finally:
            pr.unpatch(); pr.patch_spec = None
        k_p = k.clone(); k_p[:, :, n_txt + tok, :] = k[:, :, n_txt + 5, :]
        assert torch.allclose(out, torch.nn.functional.scaled_dot_product_attention(q, k_p, vv, scale=0.1))
        res = pr.consume(torch.tensor([tok]))
        assert torch.allclose(res["k_cap"][:, :, 0, :], k[:, :, n_txt + 5, :]), "captured key must be the patched key"
        assert (res["tok"][:, 0, 3] == 0).all() and (res["head"][:, 7] > 0.1).all()
        assert (res["tok"][:, 0, 0] > 1).all(), "patched ordinary key must not rank first"
        # value patch. Routing is unchanged and the output matches brute force with the patched value
        v_src = torch.randn(B, H, 1, Dh, generator=g)
        pr.patch()
        try:
            pr.active = 0; pr.clean_rel = rel_ref.clone(); pr.capture_toks = None
            pr.patch_spec = {"kind": "value", "pos": torch.tensor([tok]), "k_src": None, "v_src": v_src}
            out = F.scaled_dot_product_attention(query=q, key=k, value=vv)
        finally:
            pr.unpatch(); pr.patch_spec = None
        v_p = vv.clone(); v_p[:, :, n_txt + tok, :] = v_src[:, :, 0, :]
        assert torch.allclose(out, torch.nn.functional.scaled_dot_product_attention(q, k, v_p))
        res = pr.consume(torch.tensor([tok]))
        assert torch.allclose(res["rel"], rel_ref, atol=1e-5) and res["head"][:, 7].abs().max() < 1e-6
        # key and value with single-row sources broadcast to both rows. The sham patch is a no-op
        pr.patch()
        try:
            pr.active = 0; pr.clean_rel = None
            pr.patch_spec = {"kind": "key_value", "pos": torch.tensor([tok]), "k_src": k_src[:1], "v_src": v_src[:1]}
            out = F.scaled_dot_product_attention(q, k, vv)
            pr.consume(torch.tensor([tok]))
            pr.patch_spec = {"kind": "sham", "pos": torch.tensor([tok]), "k_src": None, "v_src": None}
            out2 = F.scaled_dot_product_attention(q, k, vv)
        finally:
            pr.unpatch(); pr.patch_spec = None
        k_p = k.clone(); k_p[:, :, n_txt + tok, :] = k_src[0, :, 0, :]; v_p = vv.clone(); v_p[:, :, n_txt + tok, :] = v_src[0, :, 0, :]
        assert torch.allclose(out, torch.nn.functional.scaled_dot_product_attention(q, k_p, v_p))
        assert torch.equal(out2, torch.nn.functional.scaled_dot_product_attention(q, k, vv))
        res = pr.consume(torch.tensor([tok]))
        assert math.isnan(res["head"][0, 7].item())
        pr.stash = None; pr.full = False
        pr.patch()
        try:
            pr.active = 0; pr.capture_toks = torch.tensor([2])
            _ = F.scaled_dot_product_attention(q, k, vv)
        finally:
            pr.unpatch()
        res = pr.consume(torch.tensor([2]))
        assert res["rel"] is None and res["k_cap"].shape == (B, H, 1, Dh)

    # replay wrapper on a tiny module
    class _M(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.lin = torch.nn.Linear(4, 4); self.calls = 0
        def forward(self, hidden_states, return_dict=True):
            self.calls += 1
            o = self.lin(hidden_states)
            return types.SimpleNamespace(sample=o) if return_dict else (o,)
    m = _M()
    xs = [torch.randn(1, 4, generator=g) for _ in range(4)]
    seen = []
    w = TransformerReplay(m, cache=None, n_replay=0, record_steps=(2,), on_step=seen.append, cache_steps=(0, 1)).install()
    outs = [m(x_, return_dict=False)[0] for x_ in xs]
    w.uninstall()
    assert seen == [0, 1, 2, 3] and m.calls == 4 and sorted(w.cached) == [0, 1] and 2 in w.outputs
    assert not ("forward" in m.__dict__), "uninstall must restore the class forward"
    cache = [w.cached[0], w.cached[1]]
    m.calls = 0; seen2 = []
    w2 = TransformerReplay(m, cache=cache, n_replay=2, record_steps=(2,), on_step=seen2.append).install()
    outs2 = [m(x_, return_dict=False)[0] for x_ in xs]
    o_rd = m(xs[0], return_dict=True)
    w2.uninstall()
    assert m.calls == 3 and seen2 == [0, 1, 2, 3, 4]
    assert all(torch.equal(a, b) for a, b in zip(outs, outs2)) and torch.equal(o_rd.sample, m(xs[0]).sample)
    assert torch.equal(w2.inputs[2][0], xs[2][0]) and torch.equal(w2.outputs[2], outs[2])
    bits, dn = tensor_to_bits(torch.randn(3, generator=g).to(torch.bfloat16))
    assert bits.dtype == np.int16 and dn == "bfloat16"
    t = torch.randn(5, generator=g).to(torch.bfloat16)
    assert torch.equal(bits_to_tensor(*tensor_to_bits(t)), t)
    print("[q8-operators] all unit tests PASSED (lesions, rescues, structure helpers, geometry, routing "
          "distances, attention probe with key/value patching and capture in both layouts, replay wrapper)")

_run_q8_unit_tests()


## R4. Recorder and runner

Each block gets a forward pre-hook and a forward hook. At the intervened
step the pre-hook turns on the attention probe and, inside the attention
rescue window, installs the key or value patch. The hook applies the lesion
and the residual rescue to the block output (all batch rows) inside their
windows, then measures the conditional row. It records token norms, the
tracked tokens' channel values, vectors and v\* geometry, the probe
readouts, the attention distributions at the probe blocks and the distance
to the clean state at the reference blocks.

A baseline run computes every step, picks the register (the largest-norm
token at the lesion block) and a random ordinary token, and stores the
clean values, routing and state references and the replay cache. A
condition run replays the clean prefix and reads these from the baseline.
Runs are resumable, and each stored run carries its condition, run hash,
decision hash, v\* hash and step so it can be checked on reload.


In [ ]:
# =====================================================================
# R4. Rescue recorder and runner
#
# This is where interventions and measurements happen, for both the smoke
# test and the capture loop. Each block gets one forward pre-hook and one
# forward hook. The replay wrapper around the transformer's forward numbers
# the denoising steps and replays the clean prefix in condition runs.
#
# At the intervened step the pre-hook turns the attention probe on for this
# block (routing statistics, key and value capture at the tracked tokens,
# distance to the clean routing) and installs the key/value patch if the
# condition's attention rescue covers this block.
#
# The hook, at the same step, first applies the lesion and the residual
# rescue inside their windows to the image-stream block output (all batch
# rows). It then measures the conditional row of the result: every token's
# norm, the tracked tokens' channel values, v* geometry and full vectors
# (all rows, fp32, which in a baseline run are the clean values), the
# population cosine statistics, the probe fields, the attention
# distributions at the probe blocks and the per-token distance to the clean
# state at the reference blocks. At later steps it records only token norms
# and the tracked tokens' geometry.
#
# A baseline run picks the register as the largest-norm token of the
# conditional row at the lesion block (plus more high-norm tokens when
# n_targets > 1) and a seeded random ordinary token, at each intervention
# step. Condition runs take these from the baseline.
# =====================================================================
import gc, time, datetime
import numpy as np
import torch

K_CH = len(TRACKED_CH)
N_GT, N_GP = len(GEOM_TOKEN_FIELDS), len(GEOM_POP_FIELDS)
N_PT, N_PH = len(PROBE_TOKEN_FIELDS), len(PROBE_HEAD_FIELDS)
N_TOK = int(Q8.n_targets) + 1          # targets plus the random ordinary token, which is always last
RAND_SLOT = N_TOK - 1

def _select_targets(norms_row, n_targets, mult):
    """norms_row [N] fp32 (conditional row) -> LongTensor [n_targets].
    The argmax token comes first, then further tokens with norm >= mult x
    median in descending order, padded with -1 when fewer qualify."""
    med = torch.clamp(norms_row.median(), min=1e-9)
    order = torch.argsort(norms_row, descending=True)
    out = torch.full((n_targets,), -1, dtype=torch.long, device=norms_row.device)
    out[0] = order[0]
    j = 1
    for t in order[1:]:
        if j >= n_targets or norms_row[t] < mult * med:
            break
        out[j] = t; j += 1
    return out

def _pick_random_ordinary(norms_row, exclude, mult, seed):
    med = max(float(norms_row.median().item()), 1e-9)
    ordinary = torch.where(norms_row < mult * med)[0].cpu().numpy()
    ordinary = np.array([t for t in ordinary if t not in set(int(e) for e in exclude)], np.int64)
    if len(ordinary) == 0:
        ordinary = np.array([t for t in range(norms_row.numel()) if t not in set(int(e) for e in exclude)], np.int64)
    rng = np.random.default_rng(int(seed) + int(Q8.random_token_seed))
    return int(rng.choice(ordinary))

class RescueRecorder:
    def __init__(self, transformer, n_steps, n_img, n_txt, d_model, full_steps, cond=None, aux=None,
                 seed=0, expected_batch=1, select_row=0, vstar=None, probe_blocks=(), state_ref_blocks=(),
                 store_tn_later=True, store_probe_rel=True, store_kv=True, store_state_ref=False,
                 store_rel_all=False, probe_chunk=256, geom_thr=0.9, n_heads=None, rand_dirs=None):
        self.all_blocks = enumerate_flux_blocks(transformer) if MODEL_FAMILY == "flux" else enumerate_model_blocks(transformer)
        self.L = len(self.all_blocks)
        self.n_steps, self.n_img, self.n_txt, self.d_model = int(n_steps), int(n_img), int(n_txt), int(d_model)
        self.full_steps = [int(s) for s in full_steps]           # steps with full readouts
        self.full_pos = {s: i for i, s in enumerate(self.full_steps)}
        self.cond = cond                                         # None = baseline
        self.aux = aux                                           # clean values per full step, condition runs only
        self.seed = int(seed)
        self.expected_batch, self.select_row = int(expected_batch), int(select_row)
        self.vstar_np = np.asarray(vstar, np.float32)
        self.probe_blocks = [int(b) for b in probe_blocks]
        self.probe_pos = {b: i for i, b in enumerate(self.probe_blocks)}
        self.state_ref_blocks = [int(b) for b in state_ref_blocks]
        self.state_pos = {b: i for i, b in enumerate(self.state_ref_blocks)}
        self.store_tn_later, self.store_probe_rel = bool(store_tn_later), bool(store_probe_rel)
        self.store_kv, self.store_state_ref = bool(store_kv), bool(store_state_ref)
        self.store_rel_all = bool(store_rel_all)
        self.geom_thr = float(geom_thr)
        self.n_heads = int(n_heads)
        self.rand_dirs = None if rand_dirs is None else np.asarray(rand_dirs, np.float32)
        self.probe = AttentionProbe(n_img=self.n_img, n_txt=self.n_txt, select_row=self.select_row,
                                    chunk=probe_chunk, n_heads_expected=self.n_heads)
        self._handles = []
        self.step = -1
        self._cache = {}
        self.reset()

    # ------------------------------------------------------------ storage
    def reset(self):
        L, T, N, D, S = self.L, self.n_steps, self.n_img, self.d_model, len(self.full_steps)
        H, K, P, R = self.n_heads, N_TOK, len(self.probe_blocks), len(self.state_ref_blocks)
        first = min(self.full_steps)
        self.tn_steps = list(range(T)) if self.cond is None else list(range(first, T))
        self.tn_pos = {s: i for i, s in enumerate(self.tn_steps)}
        self.tn_all = np.zeros((len(self.tn_steps), L, N), np.float16)
        self.geom_tok_steps = np.full((len(self.tn_steps), L, K, N_GT), np.nan, np.float32)
        self.tracked_toks = np.full((S, K), -1, np.int64)            # per full step
        self.tracked_vals = np.full((S, L, K, K_CH + 1), np.nan, np.float32)
        self.geom_tok = np.full((S, L, K, N_GT), np.nan, np.float32)
        self.geom_pop = np.full((S, L, N_GP), np.nan, np.float32)
        self.reg_vec = np.full((S, L, self.expected_batch, K, D), np.nan, np.float32)
        self.probe_tok = np.full((S, L, H, K, N_PT), np.nan, np.float32)
        self.probe_head = np.full((S, L, H, N_PH), np.nan, np.float32)
        self.probe_rel = np.zeros((S, P, H, N), np.float16) if self.store_probe_rel else None
        self.rel_all = np.zeros((S, L, H, N), np.float16) if self.store_rel_all else None
        self.probe_fires = np.zeros((S, L), np.int32)
        Dh = None
        self.k_cap = None; self.v_cap = None                         # allocated on the first capture, Dh isn't known yet
        self.state_dist = np.full((S, R, N), np.nan, np.float32)
        self.state_ref = (np.zeros((S, R, N, D), np.float16) if self.store_state_ref else None)
        self.viol = np.zeros((S, L), np.float32)
        self.intv_fires = np.zeros((S, L), np.int32)
        self.counters = {b: 0 for b in range(L)}
        self.errors = []
        self._clamped = 0
        self.probe.fires = 0; self.probe.ignored = 0; self.probe.errors = []
        self.probe.stash = None; self.probe.active = None; self.probe.patch_spec = None
        self.probe.capture_toks = None; self.probe.clean_rel = None

    def set_step(self, step):
        self.step = int(step)

    # ------------------------------------------------------------ helpers
    def _dev(self, dev):
        key = str(dev)
        ent = self._cache.get(key)
        if ent is None:
            ent = {"vstar": torch.tensor(self.vstar_np, dtype=torch.float32, device=dev),
                   "tracked_ch": torch.tensor(TRACKED_CH, dtype=torch.long, device=dev)}
            self._cache[key] = ent
        return ent

    def _pick_row(self, t3):
        b = t3.shape[0]
        if b == self.expected_batch:
            return t3[self.select_row if b > 1 else 0]
        if self.expected_batch == 2 and b == 1:
            return t3[0]
        raise RuntimeError(f"unexpected batch size {b} (expected {self.expected_batch})")

    def _toks_for(self, si):
        """Tracked tokens of full step si as a LongTensor (targets first, then
        the random token), or None."""
        t = self.tracked_toks[si]
        if (t < 0).all():
            return None
        return torch.tensor(t, dtype=torch.long)

    def _aux_step(self):
        return None if self.aux is None else self.aux[self.step]

    def _clean(self, key):
        a = self._aux_step()
        return None if a is None else a.get(key)

    def _target_slice(self, position):
        """Token indices (LongTensor) that receive the write, either the
        targets or the random token, and the matching source slots."""
        toks = self._toks_for(self.full_pos[self.step])
        tg = toks[:RAND_SLOT]
        tg = tg[tg >= 0]
        if position == "random":
            return toks[RAND_SLOT:RAND_SLOT + 1], torch.arange(1)           # write at the random token, using source slot 0 (the primary target)
        return tg, torch.arange(len(tg))

    # ------------------------------------------------------------ hooks
    def _make_pre_hook(self, bidx):
        def pre_hook(module, args, kwargs):
            step = self.step
            self.probe.active = None; self.probe.patch_spec = None
            self.probe.capture_toks = None; self.probe.clean_rel = None; self.probe.stash = None
            if step not in self.full_pos or step >= self.n_steps:
                return None
            si = self.full_pos[step]
            self.probe.active = bidx
            self.probe.full = True
            toks = self._toks_for(si)
            if toks is not None and self.store_kv:
                self.probe.capture_toks = toks
            relc = self._clean("rel_clean")
            if relc is not None:
                self.probe.clean_rel = relc[bidx]
            c = self.cond
            if c is not None and c["rescue_kind"] in (ATTN_RESCUES + ("sham",)) and c["rescue_lo"] <= bidx <= c["rescue_hi"]:
                try:
                    pos, src = self._target_slice(c["position"])
                    src_slot = (torch.full_like(src, RAND_SLOT) if c["source"] == "random" else src)
                    kc, vc = self._clean("k_clean"), self._clean("v_clean")     # [L, B, H, K, Dh] torch
                    ks, vs = kc[bidx][:, :, src_slot, :], vc[bidx][:, :, src_slot, :]
                    if c["rescue_kind"] == "sham" and not (torch.isfinite(ks).all() and torch.isfinite(vs).all()):
                        ks = vs = None                                  # no stored keys at this block, fall back to a self-copy sham
                    self.probe.patch_spec = {"kind": c["rescue_kind"], "pos": pos, "k_src": ks, "v_src": vs}
                except Exception as e:
                    self.errors.append(f"block {bidx} step {step} [attn-rescue]: {type(e).__name__}: {e}")
            return None
        return pre_hook

    def _apply_residual(self, region, bidx, si):
        """Lesion then residual rescue on the image region (all rows)."""
        c = self.cond
        dev = region.device
        ent = self._dev(dev)
        viol = torch.zeros((), device=dev)
        n_ops = 0
        toks_all = self._toks_for(si).to(dev)
        targets = toks_all[:RAND_SLOT]; targets = targets[targets >= 0]
        y = region
        if c["lesion_lo"] <= bidx <= c["lesion_hi"]:
            kind = c["lesion_kind"]
            kw = {}
            if kind == "cstar_zero":
                kw["channel"] = int(CH1)
            elif kind in ("vstar_destroy", "vstar_random_dir"):
                kw["vstar"] = ent["vstar"]
                if kind == "vstar_random_dir":
                    kw["rand_dirs"] = torch.tensor(self.rand_dirs[:len(targets)], device=dev)
            elif kind == "reg_remove":
                Xr = self._pick_row(region).to(torch.float32)
                nr = Xr.norm(dim=1)
                om = nr < float(Q8.ordinary_norm_mult) * torch.clamp(nr.median(), min=1e-9)
                om[toks_all[toks_all >= 0]] = False
                kw["ordinary_mask"] = om
            elif kind == "sham":
                rv0 = self._clean("reg_vec_clean")
                xc0 = rv0[bidx][:, torch.arange(len(targets)), :].to(dev)
                if torch.isfinite(xc0).all():
                    kw["clean_vec"] = xc0
            y, v = apply_residual_op(y, targets, kind, **kw)
            viol = viol + v; n_ops += 1
        rk = c["rescue_kind"]
        if rk in (RESIDUAL_RESCUES + ("sham",)) and c["rescue_lo"] <= bidx <= c["rescue_hi"]:
            pos, src = self._target_slice(c["position"])
            pos = pos.to(dev)
            rv = self._clean("reg_vec_clean")                      # [L, B, K, D] torch fp32
            xc = rv[bidx][:, src, :].to(dev)                        # [B, K', D]
            kw = {}
            if rk == "sham":
                kind = "sham"; kw = dict(clean_vec=xc)
            elif rk == "vstar":
                kind = "vstar"; kw = dict(vstar=ent["vstar"], clean_scalar=(xc @ ent["vstar"]), frac=float(c["frac"]))
            elif rk == "cstar":
                kind = "cstar"; kw = dict(channel=int(CH1), clean_scalar=xc[:, :, int(CH1)])
            elif rk.startswith("chan_"):
                ch = int(CH2) if c["channel_role"] == "c2" else int(CH_CTRL)
                kind = "chan"; kw = dict(channel=ch, clean_scalar=xc[:, :, ch])
            elif rk == "norm":
                kind = "norm"; kw = dict(clean_scalar=xc.norm(dim=-1))
            elif rk == "full":
                kind = "full"; kw = dict(clean_vec=xc)
            else:
                raise RuntimeError(rk)
            y, v = apply_residual_op(y, pos, kind, **kw)
            viol = viol + v; n_ops += 1
        return y, viol, n_ops

    def _make_hook(self, bidx):
        def hook(module, args, kwargs, output):
            step = self.step
            self.counters[bidx] += 1
            self.probe.active = None
            self.probe.patch_spec = None
            if step >= self.n_steps or step < 0:
                self.probe.stash = None
                return None
            new_output = None
            si = self.full_pos.get(step)
            # ---- interventions at the intervened step
            if si is not None and self.cond is not None:
                try:
                    holder = {}
                    def _fn(region):
                        y, v, n = self._apply_residual(region, bidx, si)
                        holder["viol"], holder["n"] = v, n
                        return y
                    new_output, found = modify_block_output(output, self.n_img, self.n_txt, _fn)
                    if not found:
                        new_output = None
                        raise RuntimeError(f"image stream not found in output at block {bidx}")
                    if holder.get("n", 0) == 0:
                        new_output = None
                    else:
                        self.intv_fires[si, bidx] += holder["n"]
                        self.viol[si, bidx] = float(holder["viol"].item())
                except Exception as e:
                    self.errors.append(f"block {bidx} step {step} [intervene]: {type(e).__name__}: {e}")
                    new_output = None
            # ---- measurement
            try:
                src = new_output if new_output is not None else output
                region = extract_image_region(src, self.n_img, self.n_txt)            # [B, N, D]
                X = self._pick_row(region).to(torch.float32)                          # [N, D]
                if X.shape[0] != self.n_img or X.shape[1] != self.d_model:
                    raise RuntimeError(f"measured stream shape {tuple(X.shape)}")
                ent = self._dev(X.device)
                norms = X.norm(dim=1)
                if step in self.tn_pos:
                    self._clamped += int((norms > 65504.0).sum().item())
                    self.tn_all[self.tn_pos[step], bidx] = norms.clamp(max=65504.0).to("cpu", torch.float16).numpy()
                if si is None:
                    # later step. Only the tracked-token geometry, using the tokens of the nearest earlier full step
                    ref_si = max(i for i, s in enumerate(self.full_steps) if s < step) if any(s < step for s in self.full_steps) else None
                    if ref_si is not None and step in self.tn_pos:
                        toks = self._toks_for(ref_si)
                        if toks is not None:
                            toks = toks.to(X.device); ok = toks >= 0
                            tg, _ = geometry_readout(X, ent["vstar"], torch.where(ok, toks, torch.zeros_like(toks)), thr=self.geom_thr)
                            tg = tg.to("cpu").numpy(); tg[~ok.cpu().numpy()] = np.nan
                            self.geom_tok_steps[self.tn_pos[step], bidx] = tg
                    self.probe.stash = None
                    return new_output
                # ---- full readouts at the intervened step
                if self.cond is None and bidx == B_LESION and (self.tracked_toks[si] < 0).all():
                    tg = _select_targets(norms, int(Q8.n_targets), float(Q8.target_norm_mult))
                    excl = [int(t) for t in tg.tolist() if t >= 0]
                    rt = _pick_random_ordinary(norms, excl, float(Q8.ordinary_norm_mult), self.seed + 1000 * si)
                    self.tracked_toks[si, :RAND_SLOT] = tg.cpu().numpy()
                    self.tracked_toks[si, RAND_SLOT] = rt
                toks = self._toks_for(si)
                if toks is not None:
                    toks = toks.to(X.device); ok = toks >= 0
                    safe = torch.where(ok, toks, torch.zeros_like(toks))
                    okn = ok.cpu().numpy()
                    vals = torch.cat([X[safe][:, ent["tracked_ch"]], norms[safe][:, None]], dim=1)   # [K, K_CH+1]
                    tg, pop = geometry_readout(X, ent["vstar"], safe, thr=self.geom_thr)
                    pack = torch.cat([vals.flatten(), tg.flatten(), pop]).to("cpu", torch.float32).numpy()
                    o = 0
                    v_ = pack[o:o + N_TOK * (K_CH + 1)].reshape(N_TOK, K_CH + 1); o += N_TOK * (K_CH + 1)
                    g_ = pack[o:o + N_TOK * N_GT].reshape(N_TOK, N_GT); o += N_TOK * N_GT
                    v_[~okn] = np.nan; g_[~okn] = np.nan
                    self.tracked_vals[si, bidx] = v_
                    self.geom_tok[si, bidx] = g_
                    self.geom_tok_steps[self.tn_pos[step], bidx] = g_
                    self.geom_pop[si, bidx] = pack[o:o + N_GP]
                    rv = region[:, safe, :].to("cpu", torch.float32).numpy()                       # [B, K, D]
                    rv[:, ~okn] = np.nan
                    self.reg_vec[si, bidx, :rv.shape[0]] = rv
                else:
                    _, pop = geometry_readout(X, ent["vstar"], torch.zeros(1, dtype=torch.long, device=X.device), thr=self.geom_thr)
                    self.geom_pop[si, bidx] = pop.to("cpu").numpy()
                # ---- probe
                res = self.probe.consume(toks if toks is not None else torch.zeros(1, dtype=torch.long, device=X.device))
                if res is None:
                    self.errors.append(f"block {bidx} step {step} [probe]: no attention call captured")
                else:
                    H = res["H"]
                    if H != self.n_heads:
                        raise RuntimeError(f"probe saw {H} heads, expected {self.n_heads}")
                    self.probe_fires[si, bidx] += 1
                    if res["tok"] is not None and toks is not None:
                        pt = res["tok"].to("cpu").numpy(); pt[:, ~okn] = np.nan
                        self.probe_tok[si, bidx] = pt
                    if res["head"] is not None:
                        self.probe_head[si, bidx] = res["head"].to("cpu").numpy()
                    if res["rel"] is not None:
                        rel16 = res["rel"].clamp(max=65504.0).to("cpu", torch.float16).numpy()
                        if self.probe_rel is not None and bidx in self.probe_pos:
                            self.probe_rel[si, self.probe_pos[bidx]] = rel16
                        if self.rel_all is not None:
                            self.rel_all[si, bidx] = rel16
                    if res["k_cap"] is not None:
                        kc = res["k_cap"]
                        if self.k_cap is None:
                            S, L, B, K, Dh = len(self.full_steps), self.L, self.expected_batch, N_TOK, kc.shape[-1]
                            self.k_cap = np.full((S, L, B, H, K, Dh), np.nan, np.float32)
                            self.v_cap = np.full((S, L, B, H, K, Dh), np.nan, np.float32)
                        kk = kc.to("cpu").numpy(); vv = res["v_cap"].to("cpu").numpy()
                        kk[:, :, ~okn] = np.nan; vv[:, :, ~okn] = np.nan
                        self.k_cap[si, bidx, :kk.shape[0]] = kk; self.v_cap[si, bidx, :vv.shape[0]] = vv
                # ---- state references and distances
                if bidx in self.state_pos:
                    ri = self.state_pos[bidx]
                    if self.store_state_ref and self.state_ref is not None:
                        self.state_ref[si, ri] = X.clamp(-65504.0, 65504.0).to("cpu", torch.float16).numpy()
                    ref = self._clean("state_ref")
                    if ref is not None:
                        Xc = ref[ri].to(X.device, torch.float32)
                        d = (X - Xc).norm(dim=1) / torch.clamp(Xc.norm(dim=1), min=1e-9)
                        self.state_dist[si, ri] = d.to("cpu").numpy()
                    elif self.cond is None:
                        self.state_dist[si, ri] = 0.0
            except Exception as e:
                self.errors.append(f"block {bidx} step {step} [measure]: {type(e).__name__}: {e}")
            return new_output
        return hook

    def attach(self):
        assert not self._handles, "recorder already attached"
        self.probe.patch()
        for i, mod, _kind in self.all_blocks:
            self._handles.append(mod.register_forward_pre_hook(self._make_pre_hook(i), with_kwargs=True))
            self._handles.append(mod.register_forward_hook(self._make_hook(i), with_kwargs=True))
        return self

    def detach(self):
        for h in self._handles:
            h.remove()
        self._handles = []
        self.probe.unpatch()

    def finalize(self, computed_steps):
        exp = int(computed_steps)
        fires_ok = all(self.counters[b] == exp for b in range(self.L))
        intv_ok, exp_ops = True, {}
        if self.cond is not None:
            c = self.cond
            for si in range(len(self.full_steps)):
                for b in range(self.L):
                    n = int(c["lesion_lo"] <= b <= c["lesion_hi"]) + int(
                        c["rescue_kind"] in (RESIDUAL_RESCUES + ("sham",)) and c["rescue_lo"] <= b <= c["rescue_hi"])
                    if int(self.intv_fires[si, b]) != n:
                        intv_ok = False
        probe_ok = bool((self.probe_fires == 1).all()) and not self.probe.errors
        nan_flag = not (np.isfinite(self.tn_all.astype(np.float32)).all() and np.isfinite(self.geom_pop).all())
        return {"fires_ok": bool(fires_ok), "intv_fires_ok": bool(intv_ok), "probe_ok": bool(probe_ok),
                "probe_fires": int(self.probe_fires.sum()), "probe_expected": int(self.probe_fires.size),
                "probe_ignored": int(self.probe.ignored), "nan_flag": bool(nan_flag),
                "hook_errors": len(self.errors) + len(self.probe.errors),
                "errors": list(self.errors) + list(self.probe.errors),
                "counters": {int(b): int(c) for b, c in self.counters.items()},
                "viol_total": float(self.viol.sum()), "fp16_clamped": int(self._clamped),
                "targets_found": bool((self.tracked_toks[:, 0] >= 0).all())}

    def arrays(self):
        out = {"tn_all": self.tn_all, "tn_steps": np.asarray(self.tn_steps, np.int32),
               "full_steps": np.asarray(self.full_steps, np.int32),
               "tracked_toks": self.tracked_toks.astype(np.int32), "tracked_vals": self.tracked_vals,
               "geom_tok": self.geom_tok, "geom_pop": self.geom_pop, "geom_tok_steps": self.geom_tok_steps,
               "reg_vec": self.reg_vec, "probe_tok": self.probe_tok, "probe_head": self.probe_head,
               "probe_fires": self.probe_fires, "state_dist": self.state_dist,
               "viol": self.viol, "intv_fires": self.intv_fires}
        if self.probe_rel is not None:
            out["probe_rel"] = self.probe_rel
        if self.rel_all is not None:
            out["rel_all"] = self.rel_all
        if self.k_cap is not None:
            out["k_cap"] = self.k_cap.astype(np.float32 if self.cond is None else np.float16)
            out["v_cap"] = self.v_cap.astype(np.float32 if self.cond is None else np.float16)
        if self.state_ref is not None:
            out["state_ref"] = self.state_ref
        return out

# ------------------------------------------------------------ baseline auxiliaries
def _rid(cond_name, int_step, sid_base):
    return f"baseline__{sid_base}" if cond_name == "baseline" else f"{cond_name}__t{int(int_step):02d}__{sid_base}"

def _replay_path(sid_base, out_dir):
    return os.path.join(out_dir, f"baseline__{sid_base}__replay.npz")

def load_baseline_aux(sid_base, out_dir=None, device="cpu"):
    """Clean values of every intervention step from a finished baseline run,
    as {step: dict(tracked_toks, reg_vec_clean [L,B,K,D] torch fp32, k_clean,
    v_clean [L,B,H,K,Dh], rel_clean [L,H,N] torch fp32, state_ref [R,N,D]
    torch fp16, out_clean, in_clean, tn_lesion)}, plus the replay cache."""
    out_dir = out_dir or Q8_PATHS["runs"]
    bp = scenario_paths(out_dir, f"baseline__{sid_base}", False)
    aux = {}
    with np.load(bp["npz"], allow_pickle=False) as z:
        steps = [int(s) for s in z["full_steps"]]
        rel_all = z["rel_all"] if "rel_all" in z.files else None
        for si, s in enumerate(steps):
            a = {"tracked_toks": z["tracked_toks"][si].astype(np.int64),
                 "reg_vec_clean": torch.from_numpy(z["reg_vec"][si].astype(np.float32)),
                 "k_clean": torch.from_numpy(z["k_cap"][si].astype(np.float32)),
                 "v_clean": torch.from_numpy(z["v_cap"][si].astype(np.float32)),
                 "rel_clean": (torch.from_numpy(rel_all[si].astype(np.float32)) if rel_all is not None else None),
                 "state_ref": (torch.from_numpy(z["state_ref"][si]) if "state_ref" in z.files else None),
                 "out_clean": z[f"out_t{s:02d}"].astype(np.float32),
                 "in_clean": z[f"in_t{s:02d}"].astype(np.float32),
                 "tn_lesion": z["tn_all"][list(z["tn_steps"]).index(s), B_LESION].astype(np.float32)}
            aux[s] = a
    rp = _replay_path(sid_base, out_dir)
    with np.load(rp, allow_pickle=False) as z:
        aux["replay_bits"] = z["bits"]
        aux["replay_dtype"] = str(z["dtype"].item())
        aux["replay_steps"] = int(z["n_cached"])
    return aux

def stored_run_healthy(paths):
    try:
        with np.load(paths["npz"], allow_pickle=False) as z:
            m = meta_from_npz(z)
            return bool(m["fires_ok"] and m["intv_fires_ok"] and m["probe_ok"] and m["hook_errors"] == 0
                        and not m["nan_flag"] and m["viol_total"] == 0.0 and m["targets_found"])
    except Exception:
        return False

# ------------------------------------------------------------------ runner
def run_rescue_run(pipe, cond, int_step, sid_base, pid, prompt_text, seed, baseline_aux=None,
                   out_dir=None, n_steps=None, tag="q8", force=False, full_steps=None):
    """One generation under one condition at one intervention step. The
    baseline instead records every intervention step. Resumable, and all
    saves are atomic."""
    out_dir = out_dir or Q8_PATHS["runs"]
    n_steps = int(n_steps or CFG.num_steps)
    is_base = cond["name"] == "baseline"
    full_steps = list(full_steps) if full_steps is not None else (list(INT_STEPS) if is_base else [int(int_step)])
    full_steps = [s for s in full_steps if s < n_steps]
    assert full_steps, "no intervention step inside the run"
    rid = _rid(cond["name"], int_step, sid_base)
    paths = scenario_paths(out_dir, rid, False)
    expect = {"condition": cond["name"], "q8_hash": Q8_HASH, "dec_sha": Q8_DEC_SHA, "vstar_sha": VSTAR_SHA,
              "num_steps": n_steps, "int_step": (None if is_base else int(int_step))}
    if is_base:
        expect["full_steps"] = [int(s) for s in full_steps]     # a baseline must cover every intervention step
    if (not force and is_scenario_complete(paths, need_tensor=False, expect=expect) and stored_run_healthy(paths)
            and (not is_base or os.path.exists(_replay_path(sid_base, out_dir)))):
        return {"tag": tag, "sid": rid, "prompt_id": pid, "seed": seed, "status": "skipped", "precision_mode": PRECISION_MODE}
    if not is_base and baseline_aux is None:
        raise RuntimeError(f"{rid}: a condition run needs the baseline auxiliaries")

    aux = None; rand_dirs = None
    if not is_base:
        aux = {int(int_step): baseline_aux[int(int_step)]}
        rand_dirs = orthogonal_random_dirs(VSTAR, max(1, int(Q8.n_targets)), seed=int(seed) + 4242)
    rec = RescueRecorder(pipe.transformer, n_steps=n_steps, n_img=N_IMG, n_txt=N_TXT_IN_SEQ, d_model=CFG.d_model,
                         full_steps=full_steps, cond=(None if is_base else cond), aux=aux, seed=seed,
                         expected_batch=EXP_BATCH, select_row=SELECT_ROW, vstar=VSTAR,
                         probe_blocks=PROBE_BLOCKS_L, state_ref_blocks=STATE_REF_BLOCKS,
                         store_tn_later=Q8.store_tn_later_steps, store_probe_rel=(Q8.store_probe_rel or is_base),
                         store_kv=(is_base or Q8.store_kv_conditions or cond["rescue_kind"] in ATTN_RESCUES),
                         store_state_ref=(is_base and Q8.store_state_ref),
                         store_rel_all=is_base, probe_chunk=Q8.probe_chunk, geom_thr=Q8.geom_cos_thr, n_heads=N_HEADS_EXPECTED, rand_dirs=rand_dirs)
    if not is_base:
        rec.tracked_toks[0] = aux[int(int_step)]["tracked_toks"]
    # ---- replay wrapper
    cache = None; n_replay = 0
    if not is_base and Q8.replay_prefix:
        n_replay = int(int_step)
        bits = baseline_aux["replay_bits"]
        assert baseline_aux["replay_steps"] >= n_replay, "replay cache shorter than the intervention step"
        cache = [bits_to_tensor(bits[s], baseline_aux["replay_dtype"]) for s in range(n_replay)]
    wrap = TransformerReplay(pipe.transformer, cache=cache, n_replay=n_replay, record_steps=full_steps,
                             on_step=rec.set_step, cache_steps=(range(max(full_steps)) if is_base else None))
    torch.manual_seed(seed)
    gen = torch.Generator("cpu").manual_seed(seed)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    rec.attach(); wrap.install()
    try:
        with torch.inference_mode():
            if MODEL_FAMILY == "flux":
                pe, pp = EMBEDS[pid]
                out = pipe(prompt_embeds=pe.to(DEVICE, DTYPE), pooled_prompt_embeds=pp.to(DEVICE, DTYPE),
                           height=CFG.height, width=CFG.width, num_inference_steps=n_steps,
                           guidance_scale=CFG.guidance_scale, generator=gen, output_type="pil")
            else:
                E = EMBEDS[pid]
                out = pipe(prompt_embeds=E["pe"].to(DEVICE, DTYPE), prompt_attention_mask=E["pm"].to(DEVICE),
                           negative_prompt_embeds=E["ne"].to(DEVICE, DTYPE), negative_prompt_attention_mask=E["nm"].to(DEVICE),
                           negative_prompt=None, height=CFG.height, width=CFG.width, num_inference_steps=n_steps,
                           guidance_scale=CFG.guidance_scale, generator=gen, output_type="pil", use_resolution_binning=False)
    finally:
        wrap.uninstall(); rec.detach()
    wall = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    image = out.images[0]
    computed = n_steps - n_replay
    fin = rec.finalize(computed_steps=computed)
    if wrap.step != n_steps:
        fin["errors"].append(f"transformer called {wrap.step} times, expected {n_steps}")
        fin["hook_errors"] += 1; fin["fires_ok"] = False
    if fin["errors"]:
        append_error_log(ERROR_LOG_PATH, rid, "hook errors:\n  " + "\n  ".join(fin["errors"][:20]))

    arrays = rec.arrays()
    for s in full_steps:
        arrays[f"out_t{s:02d}"] = wrap.outputs[s].numpy() if s in wrap.outputs else np.zeros(0, np.float32)
        arrays[f"in_t{s:02d}"] = wrap.inputs[s].numpy() if s in wrap.inputs else np.zeros(0, np.float32)
    traj = None
    for s in full_steps:
        if s in wrap.outputs:
            traj = trajectory_vector(wrap.outputs[s]).numpy()
            arrays[f"traj_t{s:02d}"] = traj
    arrays["sigmas"] = (pipe.scheduler.sigmas.detach().float().cpu().numpy() if hasattr(pipe.scheduler, "sigmas") else np.zeros(0, np.float32))
    arrays["timesteps"] = (pipe.scheduler.timesteps.detach().float().cpu().numpy() if hasattr(pipe.scheduler, "timesteps") else np.zeros(0, np.float32))

    meta = {"sid": rid, "sid_base": sid_base, "prompt_id": pid, "seed": int(seed), "prompt": prompt_text, "tag": tag,
            "model_id": CFG.model_id, "condition": cond["name"], "cond_def": {k: cond[k] for k in cond},
            "int_step": (None if is_base else int(int_step)), "full_steps": [int(s) for s in full_steps],
            "n_replay": int(n_replay), "computed_steps": int(computed),
            "ch1": int(CH1), "ch2": int(CH2), "ch_ctrl": int(CH_CTRL), "tracked_ch_names": list(TRACKED_CH_NAMES),
            "b_lesion": B_LESION, "b_rescue": B_RESCUE, "b_maint_end": B_MAINT_END,
            "q8_hash": Q8_HASH, "dec_sha": Q8_DEC_SHA, "vstar_sha": VSTAR_SHA,
            "geom_token_fields": list(GEOM_TOKEN_FIELDS), "geom_pop_fields": list(GEOM_POP_FIELDS),
            "probe_token_fields": list(PROBE_TOKEN_FIELDS), "probe_head_fields": list(PROBE_HEAD_FIELDS),
            "probe_blocks": PROBE_BLOCKS_L, "state_ref_blocks": [int(b) for b in STATE_REF_BLOCKS],
            "n_heads": int(N_HEADS_EXPECTED), "n_targets": int(Q8.n_targets), "n_tracked": int(N_TOK),
            "height": CFG.height, "width": CFG.width, "num_steps": n_steps, "guidance_scale": CFG.guidance_scale,
            "n_img": int(N_IMG), "n_txt": int(N_TXT_IN_SEQ), "d_model": CFG.d_model,
            "expected_batch": EXP_BATCH, "select_row": SELECT_ROW, "applied_rows": "all",
            "counters": fin["counters"], "fires_ok": fin["fires_ok"], "intv_fires_ok": fin["intv_fires_ok"],
            "probe_ok": fin["probe_ok"], "probe_fires": fin["probe_fires"], "probe_expected": fin["probe_expected"],
            "probe_ignored": fin["probe_ignored"], "nan_flag": fin["nan_flag"], "hook_errors": fin["hook_errors"],
            "hook_error_msgs": fin["errors"][:10], "viol_total": fin["viol_total"], "fp16_clamped": fin["fp16_clamped"],
            "targets_found": fin["targets_found"], "tracked_toks": rec.tracked_toks.tolist(),
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2), "precision_mode": PRECISION_MODE,
            "config_hash": CONFIG_HASH, "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}
    arrays["meta_json"] = meta_to_npz_field(meta)

    save_png_atomic(image, paths["png"])
    if is_base:
        n_cached = max(full_steps)
        bits, dn = None, "float32"
        if n_cached > 0:
            parts = [tensor_to_bits(wrap.cached[s]) for s in range(n_cached)]
            bits = np.stack([p[0] for p in parts]); dn = parts[0][1]
        else:
            bits = np.zeros((0,), np.int16)
        save_npz_atomic(_replay_path(sid_base, out_dir), bits=bits, dtype=np.array(dn), n_cached=np.array(n_cached))
    save_npz_atomic(paths["npz"], **arrays)          # written last, marks the run as complete
    ok = (fin["fires_ok"] and fin["intv_fires_ok"] and fin["probe_ok"] and not fin["nan_flag"]
          and fin["hook_errors"] == 0 and fin["viol_total"] == 0.0 and fin["targets_found"])
    return {"tag": f"{tag}:{cond['name']}", "sid": rid, "prompt_id": pid, "seed": seed,
            "status": "ok" if ok else "warn", "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "fires_ok": fin["fires_ok"], "nan_flag": fin["nan_flag"], "hook_errors": fin["hook_errors"],
            "precision_mode": PRECISION_MODE}

print(f"[q8-runner] ready | c* {CH1} | tracked tokens per run {N_TOK} ({Q8.n_targets} target(s) + random) | "
      f"lesion {B_LESION}..{B_RESCUE - 1} | rescue {B_RESCUE} (maintained to {B_MAINT_END}) | steps {INT_STEPS}")
for _c in CONDITIONS:
    print(f"    {_c['name']:34s} lesion={_c['lesion_kind']:16s} [{_c['lesion_lo']:>2},{_c['lesion_hi']:>2}]  "
          f"rescue={_c['rescue_kind']:9s} [{_c['rescue_lo']:>2},{_c['rescue_hi']:>2}]"
          + (f" frac={_c['frac']:g}" if _c['frac'] not in (None, 1.0) else "")
          + (f" at {_c['position']}" if _c['position'] != 'target' else "")
          + (f" source={_c['source']}" if _c['source'] != 'target' else ""))


## R5. Smoke test

This runs a few short generations with the intervention at the last step and
must pass before the capture starts.

The baseline run checks that the targets are found, the geometry is finite,
the attention probe fires once per block and the keys and replay cache are
stored. The sham run writes the stored clean register vector, key and value
back inside their windows. Its transformer input at the intervened step has
to match the baseline bit for bit, and its image has to reach at least 40 dB
PSNR against the baseline image. This tests the hooks and the stored clean
values that every rescue relies on.

Every other condition then goes through the same postcondition checker that
the audit uses later. Any failure stops everything here, before the
expensive runs.

In [ ]:
# =====================================================================
# R5. Smoke test
#
# Short generations (CFG.smoke_steps) with the intervention at the last
# step. The baseline has to find the register as the max-norm token at the
# lesion block, and the probe has to fire once per block with sink shares
# summing to N. The sham run has to reproduce the baseline input exactly,
# keep the token norms within tolerance and reach 40 dB PSNR. Every other
# condition goes through check_run_postconditions, which the audit reuses.
# Any failure stops here, before the long runs.
# =====================================================================
print("=" * 70)
print("Q8 SMOKE TEST")
print("=" * 70)
_SS = int(CFG.smoke_steps)
_SMOKE_STEP = _SS - 1
_GT = {f: i for i, f in enumerate(GEOM_TOKEN_FIELDS)}
_GP = {f: i for i, f in enumerate(GEOM_POP_FIELDS)}
_PT = {f: i for i, f in enumerate(PROBE_TOKEN_FIELDS)}
_PH = {f: i for i, f in enumerate(PROBE_HEAD_FIELDS)}
_CI = {n: i for i, n in enumerate(TRACKED_CH_NAMES)}
for _f in glob.glob(os.path.join(Q8_PATHS["smoke"], "*")):
    os.remove(_f)                                   # always a fresh smoke run
_smk = dict(sid_base="p00_s0", pid="p00", prompt_text=PROMPTS["p00"], seed=0, out_dir=Q8_PATHS["smoke"],
            n_steps=_SS, tag="q8_smoke", force=True)

def _smoke_load(cond_name, step):
    p = scenario_paths(Q8_PATHS["smoke"], _rid(cond_name, step, "p00_s0"), False)
    z = np.load(p["npz"], allow_pickle=False)
    return z, meta_from_npz(z), p

def _close(a, b, rel=2e-2, abs_=1e-2):
    a = np.asarray(a, np.float64); b = np.asarray(b, np.float64)
    return bool(np.all(np.abs(a - b) <= rel * np.abs(b) + abs_))

def check_run_postconditions(z, m, zb, cond, step, rel=2e-2):
    """Check one run against its baseline for the given condition.

    z and zb are the open npz files of the run and the baseline, m is the run
    meta. Returns a list of problems, empty if the run passes.
    """
    probs = []
    si = 0
    sib = list(int(s) for s in zb["full_steps"]).index(int(step))
    K_t = int(Q8.n_targets)
    tt, ttb = z["tracked_toks"][si], zb["tracked_toks"][sib]
    if not np.array_equal(tt, ttb):
        probs.append("tracked tokens differ from baseline")
        return probs
    slots_t = [k for k in range(K_t) if tt[k] >= 0]
    inb = zb[f"in_t{step:02d}"]; inr = z[f"in_t{step:02d}"]
    if inb.shape != inr.shape or inb.size == 0:
        probs.append("transformer input at the intervened step missing")
    else:
        d = float(np.abs(inb - inr).max())
        if d > 0.0 and m.get("n_replay", 0) > 0:
            probs.append(f"replay not exact: max |input difference| at step {step} = {d:.3e}")
        elif d > 1e-3 * max(float(np.abs(inb).max()), 1e-9):
            probs.append(f"input at step {step} differs from baseline by {d:.3e}")
    if int(z["probe_fires"][si].sum()) != int(z["probe_fires"][si].size):
        probs.append(f"probe fire count {int(z['probe_fires'][si].sum())} of {int(z['probe_fires'][si].size)}")
    if float(z["viol"][si].sum()) != 0.0:
        probs.append(f"operator violations {float(z['viol'][si].sum())}")
    if cond["name"] == "baseline":
        return probs
    gt, gtb = z["geom_tok"][si], zb["geom_tok"][sib]                       # [L, K, 4]
    tv, tvb = z["tracked_vals"][si], zb["tracked_vals"][sib]               # [L, K, K_CH+1]
    rv, rvb = z["reg_vec"][si], zb["reg_vec"][sib]                         # [L, B, K, D]
    pop = z["geom_pop"][si]
    row = int(m["select_row"])
    lk = cond["lesion_kind"]
    rk = cond["rescue_kind"]
    for b in range(cond["lesion_lo"], cond["lesion_hi"] + 1):
        if rk in RESIDUAL_RESCUES and cond["position"] == "target" and cond["rescue_lo"] <= b <= cond["rescue_hi"]:
            continue        # a residual rescue at the target overrides the lesion in the same hook, so only the rescue is checked
        for k in slots_t:
            if lk == "cstar_zero":
                if tv[b, k, _CI["cstar"]] != 0.0:
                    probs.append(f"block {b}: c* of target {k} not exactly zero ({tv[b, k, _CI['cstar']]})")
            elif lk in ("vstar_destroy", "vstar_random_dir"):
                if abs(gt[b, k, _GT["cos"]]) > 2e-2:
                    probs.append(f"block {b}: |cos(target {k}, v*)| = {abs(gt[b, k, _GT['cos']]):.3f} after v* destruction")
                if not _close(gt[b, k, _GT["norm"]], gtb[b, k, _GT["norm"]], rel):
                    probs.append(f"block {b}: norm not preserved by the direction lesion")
            elif lk == "reg_remove":
                if not gt[b, k, _GT["norm"]] < float(Q8.ordinary_norm_mult) * pop[b, _GP["med_norm"]]:
                    probs.append(f"block {b}: removed register still large (norm {gt[b, k, _GT['norm']]:.1f})")
            elif lk == "sham":
                if not _close(rv[b, row, k], rvb[b, row, k], rel):
                    probs.append(f"block {b}: sham changed the register vector")
    rk = cond["rescue_kind"]
    if rk != "none":
        if cond["position"] == "random":
            slots = [RAND_SLOT]; src = [0] * len(slots)
        else:
            slots = slots_t; src = list(slots_t)
        if rk in RESIDUAL_RESCUES or rk == "sham":
            for b in range(cond["rescue_lo"], cond["rescue_hi"] + 1):
                for k, s_ in zip(slots, src):
                    xc = rvb[b, row, s_]
                    if rk == "vstar":
                        a_c = float(xc @ VSTAR) * float(cond["frac"])
                        if not _close(gt[b, k, _GT["alpha"]], a_c, rel, abs_=1e-2 * abs(a_c) + 1e-2):
                            probs.append(f"block {b}: alpha of slot {k} = {gt[b, k, _GT['alpha']]:.3f}, expected {a_c:.3f}")
                    elif rk in ("cstar", "chan_c2", "chan_ctrl"):
                        ch = {"cstar": CH1, "chan_c2": CH2, "chan_ctrl": CH_CTRL}[rk]
                        ci = _CI[{"cstar": "cstar", "chan_c2": "c2", "chan_ctrl": "ctrl"}[rk]]
                        if not _close(tv[b, k, ci], xc[ch], rel):
                            probs.append(f"block {b}: channel {ch} of slot {k} = {tv[b, k, ci]:.3f}, expected {xc[ch]:.3f}")
                    elif rk == "norm":
                        if not _close(gt[b, k, _GT["norm"]], np.linalg.norm(xc), rel):
                            probs.append(f"block {b}: norm of slot {k} = {gt[b, k, _GT['norm']]:.2f}, expected {np.linalg.norm(xc):.2f}")
                    elif rk == "full":
                        if not _close(rv[b, row, k], xc, rel, abs_=1e-2 * float(np.abs(xc).max()) + 1e-3):
                            probs.append(f"block {b}: full restoration of slot {k} does not match the clean vector")
                    elif rk == "sham":
                        if not _close(rv[b, row, k], rvb[b, row, k], rel):
                            probs.append(f"block {b}: sham changed the register vector in the rescue window")
        elif rk in ATTN_RESCUES:
            if "k_cap" in z.files and "k_cap" in zb.files:
                kc, kcb = z["k_cap"][si].astype(np.float32), zb["k_cap"][sib].astype(np.float32)   # [L, B, H, K, Dh]
                vc, vcb = z["v_cap"][si].astype(np.float32), zb["v_cap"][sib].astype(np.float32)
                src_att = ([RAND_SLOT] * len(slots) if cond["source"] == "random" else src)
                for b in range(cond["rescue_lo"], cond["rescue_hi"] + 1):
                    for k, s_ in zip(slots, src_att):
                        if rk in ("key", "key_value"):
                            if not _close(kc[b, :, :, k], kcb[b, :, :, s_], rel, abs_=1e-2 * float(np.abs(kcb[b, :, :, s_]).max()) + 1e-3):
                                probs.append(f"block {b}: key of slot {k} does not equal the clean key of slot {s_}")
                        if rk in ("value", "key_value"):
                            if not _close(vc[b, :, :, k], vcb[b, :, :, s_], rel, abs_=1e-2 * float(np.abs(vcb[b, :, :, s_]).max()) + 1e-3):
                                probs.append(f"block {b}: value of slot {k} does not equal the clean value of slot {s_}")
                if cond["rescue_hi"] + 1 <= LAST_BLOCK and rk in ("key", "key_value") and cond["source"] == "target" and cond["position"] == "target":
                    b = cond["rescue_hi"] + 1
                    # past the patched range the key comes from the lesioned residual again, so it should differ
                    for k in slots:
                        if _close(kc[b, :, :, k], kcb[b, :, :, k], 1e-3, abs_=1e-3 * float(np.abs(kcb[b, :, :, k]).max()) + 1e-4) \
                                and cond["lesion_kind"] != "sham":
                            probs.append(f"block {b}: key of slot {k} still equals the clean key outside the patched range")
            else:
                probs.append("keys/values not stored; attention rescue cannot be verified")
    return probs

# ---- (a) baseline
_d0 = run_rescue_run(pipe, COND_BY_NAME["baseline"], _SMOKE_STEP, full_steps=[_SMOKE_STEP], **_smk)
print(f"[q8-smoke] baseline: {_d0['status']} ({_d0['wall_s']}s)")
assert _d0["status"] == "ok", f"baseline smoke not clean: {_d0}"
_zb, _mb, _pb = _smoke_load("baseline", _SMOKE_STEP)
_L = CFG.n_blocks_expected
assert all(int(c) == _SS for c in _mb["counters"].values()), _mb["counters"]
assert list(_zb["full_steps"]) == [_SMOKE_STEP] and _zb["tn_all"].shape == (_SS, _L, N_IMG)
_TT = _zb["tracked_toks"][0]
assert _TT[0] >= 0 and _TT[RAND_SLOT] >= 0 and _TT[RAND_SLOT] not in set(_TT[:RAND_SLOT].tolist()), _TT
_tn_L = _zb["tn_all"][_SMOKE_STEP, B_LESION].astype(np.float32)
assert int(np.argmax(_tn_L)) == int(_TT[0]), "primary target is not the max-norm token at the lesion block"
assert _tn_L[_TT[RAND_SLOT]] < Q8.ordinary_norm_mult * np.median(_tn_L), "random token is not ordinary"
_g = _zb["geom_tok"][0]
assert np.isfinite(_g[B_LESION:, 0]).all(), "target geometry not finite from the lesion block"
assert np.isnan(_g[:B_LESION, 0]).all(), "baseline target geometry before the lesion block must be NaN"
assert np.isfinite(_zb["geom_pop"][0]).all()
assert _mb["probe_ok"] and int(_zb["probe_fires"][0].sum()) == _L, \
    (f"probe did not fire once per block: {_zb['probe_fires'][0]}; errors {_mb['hook_error_msgs']}. If fires == 0 the "
     f"model's attention does not reach torch.nn.functional.scaled_dot_product_attention (non-native backend).")
_rel = _zb["rel_all"][0].astype(np.float32)                                  # [L, H, N]
assert _rel.shape == (_L, N_HEADS_EXPECTED, N_IMG)
assert np.allclose(_rel.sum(axis=-1), N_IMG, rtol=2e-2), "sink shares do not sum to N"
assert "k_cap" in _zb.files and np.isfinite(_zb["k_cap"][0, B_LESION + 1:, :, :, 0]).all(), "keys not captured after the lesion block"
assert np.isnan(_zb["k_cap"][0, B_LESION, :, :, 0]).all(), "baseline keys at the lesion block are not capturable (identity unknown)"
with np.load(_replay_path("p00_s0", Q8_PATHS["smoke"]), allow_pickle=False) as _zr:
    assert int(_zr["n_cached"]) == _SMOKE_STEP and _zr["bits"].shape[0] == _SMOKE_STEP, "replay cache size"
assert _zb[f"out_t{_SMOKE_STEP:02d}"].size > 0 and _zb[f"in_t{_SMOKE_STEP:02d}"].size > 0
if Q8.store_state_ref:
    assert _zb["state_ref"].shape == (1, len(STATE_REF_BLOCKS), N_IMG, CFG.d_model)
_cos_ref = float(_g[B_LESION, 0, _GT["cos"]]); _str_ref = float(_g[B_LESION, 0, _GT["norm"]] / max(_zb["geom_pop"][0, B_LESION, _GP["med_norm"]], 1e-9))
print(f"[q8-smoke] baseline: target token {_TT[0]} (random token {_TT[RAND_SLOT]}); at block {B_LESION}: cos(v*) = {_cos_ref:.3f}, "
      f"strength {_str_ref:.1f}x median; probe block {PRIMARY_PROBE}: register is top sink in "
      f"{np.nanmean(_zb['probe_tok'][0, PRIMARY_PROBE, :, 0, _PT['is_top_sink']]):.0%} of heads, share "
      f"{np.nanmedian(_zb['probe_tok'][0, PRIMARY_PROBE, :, 0, _PT['sink_rel']]):.1f}x uniform, key rank median "
      f"{np.nanmedian(_zb['probe_tok'][0, PRIMARY_PROBE, :, 0, _PT['key_rank']]):.0f}")
if REG_EXPECT_TOKEN is not None and int(_TT[0]) != int(REG_EXPECT_TOKEN):
    print(f"[q8-smoke][WARN] primary target {_TT[0]} is not the expected register token {REG_EXPECT_TOKEN}")
if _cos_ref < 0.9:
    print("[q8-smoke][WARN] register cosine with v* below 0.9 at the lesion block (short smoke run)")
_AUX = load_baseline_aux("p00_s0", out_dir=Q8_PATHS["smoke"])
assert _AUX["replay_steps"] == _SMOKE_STEP and _SMOKE_STEP in _AUX

# ---- (b) sham with replay, checks the replayed prefix and the hooks
_ds = run_rescue_run(pipe, COND_BY_NAME["sham"], _SMOKE_STEP, baseline_aux=_AUX, **_smk)
print(f"[q8-smoke] sham: {_ds['status']} ({_ds['wall_s']}s; replayed {_SMOKE_STEP} step(s))")
assert _ds["status"] == "ok", f"sham smoke not clean: {_ds}"
_z, _m, _p = _smoke_load("sham", _SMOKE_STEP)
assert _m["n_replay"] == (_SMOKE_STEP if Q8.replay_prefix else 0) and _m["computed_steps"] == _SS - _m["n_replay"]
_din = float(np.abs(_z[f"in_t{_SMOKE_STEP:02d}"] - _zb[f"in_t{_SMOKE_STEP:02d}"]).max())
print(f"[q8-smoke] sham vs baseline: max |transformer input difference| at step {_SMOKE_STEP} = {_din:.3e}")
if Q8.replay_prefix:
    assert _din == 0.0, "replayed prefix is not bit-exact (the trajectory entering the intervened step differs)"
_probs = check_run_postconditions(_z, _m, _zb, COND_BY_NAME["sham"], _SMOKE_STEP)
assert not _probs, f"sham postconditions failed: {_probs}"
_a = _z["tn_all"].astype(np.float32); _b = _zb["tn_all"][_SMOKE_STEP:].astype(np.float32)
_reld = np.abs(_a - _b) / np.maximum(np.abs(_b), 1.0)
print(f"[q8-smoke] sham vs baseline token norms (steps >= {_SMOKE_STEP}): max rel diff {float(_reld.max()):.2e}, median {float(np.median(_reld)):.2e}")
assert float(np.median(_reld)) < 1e-2, "sham run differs from baseline beyond tolerance"
_pd = np.nanmax(np.abs(_z["probe_tok"][0, B_LESION:, :, 0, _PT["sink_rel"]] - _zb["probe_tok"][0, B_LESION:, :, 0, _PT["sink_rel"]]))
print(f"[q8-smoke] sham vs baseline register sink share: max abs diff {_pd:.3e} (x uniform)")
from PIL import Image as _Image
_ia = np.asarray(_Image.open(_p["png"]).convert("RGB")).astype(np.float64)
_ib = np.asarray(_Image.open(_pb["png"]).convert("RGB")).astype(np.float64)
_mse = float(np.mean((_ia - _ib) ** 2))
_psnr = 99.0 if _mse == 0 else 10 * math.log10(255.0 ** 2 / _mse)
print(f"[q8-smoke] sham image vs baseline image: PSNR {_psnr:.1f} dB")
assert _psnr >= 40.0, "sham image differs from the baseline image (hook mechanics or replay are not clean)"

# ---- (c) every other condition
for _cond in CONDITIONS:
    if _cond["name"] in ("baseline", "sham"):
        continue
    _d = run_rescue_run(pipe, _cond, _SMOKE_STEP, baseline_aux=_AUX, **_smk)
    assert _d["status"] == "ok", f"{_cond['name']} smoke not clean: {_d}"
    _z, _m, _p = _smoke_load(_cond["name"], _SMOKE_STEP)
    _probs = check_run_postconditions(_z, _m, _zb, _cond, _SMOKE_STEP)
    assert not _probs, f"{_cond['name']}: postconditions failed: {_probs}"
    _sh = _z["probe_tok"][0, :, :, 0, _PT["sink_rel"]]; _shb = _zb["probe_tok"][0, :, :, 0, _PT["sink_rel"]]
    _msg = ", ".join(f"b{b} {np.nanmedian(_sh[b]):.1f}x (clean {np.nanmedian(_shb[b]):.1f}x)"
                     for b in (B_LESION + 1, PRIMARY_PROBE, B_MAINT_END) if b < _L)
    print(f"[q8-smoke] {_cond['name']:34s} ok ({_d['wall_s']}s) | register sink share: {_msg}")
    _z.close()
_zb.close()
print("Q8 SMOKE TEST PASSED -- safe to start the capture loop.")


## R6. Capture

For each scenario the baseline runs first and saves the clean values and the
replay cache. Every condition at every intervention step then runs from those
files, with the earlier steps replayed from the baseline. Finished runs that
passed their checks are skipped, so after a disconnect you only lose the
generation that was in progress.


In [ ]:
# =====================================================================
# R6. Capture
#
# Runs every scenario x condition x intervention step, and can be resumed.
# For each scenario the baseline computes all steps and saves its clean
# values, routing and state references and the replay cache. Each condition
# then runs from those files with the earlier steps replayed. Finished runs
# that passed are skipped. If the decision, v* or run hash changes, the
# metadata check treats old files as stale and they are redone.
# =====================================================================
from tqdm.auto import tqdm

_n_cond_runs = len(Q8_SCENARIOS) * (len(CONDITIONS) - 1) * len(INT_STEPS)
_eta_s = len(Q8_SCENARIOS) * (22 if MODEL_FAMILY == "flux" else 5) + _n_cond_runs * (
    (9 if Q8.replay_prefix else 24) if MODEL_FAMILY == "flux" else (2.5 if Q8.replay_prefix else 5))
print("=" * 70)
print(f"Q8 CAPTURE: {len(Q8_SCENARIOS)} scenarios x ({len(CONDITIONS) - 1} conditions x {len(INT_STEPS)} step(s) + baseline) = "
      f"{len(Q8_SCENARIOS) + _n_cond_runs} generations (~{_eta_s / 3600:.1f} h estimated on an A100)")
print(f"conditions: {COND_NAMES}")
print(f"lesion {B_LESION}..{B_RESCUE - 1} | rescue {B_RESCUE} (maintained to {B_MAINT_END}) | steps {INT_STEPS} | "
      f"replay prefix: {Q8.replay_prefix}")
print("=" * 70)

_n_ok = _n_skip = _n_warn = _n_err = 0
_pbar = tqdm(Q8_SCENARIOS, desc="scenarios")
for _sid_base, _pid, _seed, _text in _pbar:
    _aux = None
    try:
        _diag = run_rescue_run(pipe, COND_BY_NAME["baseline"], None, _sid_base, _pid, _text, _seed)
    except Exception as _e:
        _n_err += 1
        append_error_log(ERROR_LOG_PATH, f"baseline__{_sid_base}", f"{type(_e).__name__}: {_e}")
        print(f"[capture][ERROR] baseline__{_sid_base}: {type(_e).__name__}: {_e} -- continuing")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue
    if _diag["status"] == "skipped":
        _n_skip += 1
    else:
        append_run_log(RUN_LOG_PATH, _diag)
        if _diag["status"] == "ok":
            _n_ok += 1
        else:
            _n_warn += 1
            append_error_log(ERROR_LOG_PATH, _diag["sid"], f"warn diag: {_diag}")
            print(f"[capture][WARN] baseline for {_sid_base} not clean -- skipping its conditions this pass")
            continue
    try:
        _aux = load_baseline_aux(_sid_base)
    except Exception as _e:
        print(f"[capture][ERROR] cannot read baseline auxiliaries for {_sid_base}: {_e} -- skipping its conditions")
        continue
    for _step in INT_STEPS:
        for _cond in CONDITIONS[1:]:
            try:
                _diag = run_rescue_run(pipe, _cond, _step, _sid_base, _pid, _text, _seed, baseline_aux=_aux)
            except Exception as _e:
                _n_err += 1
                append_error_log(ERROR_LOG_PATH, _rid(_cond["name"], _step, _sid_base), f"{type(_e).__name__}: {_e}")
                print(f"[capture][ERROR] {_rid(_cond['name'], _step, _sid_base)}: {type(_e).__name__}: {_e} -- continuing")
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            if _diag["status"] == "skipped":
                _n_skip += 1
            else:
                append_run_log(RUN_LOG_PATH, _diag)
                if _diag["status"] == "ok":
                    _n_ok += 1
                else:
                    _n_warn += 1
                    append_error_log(ERROR_LOG_PATH, _diag["sid"], f"warn diag: {_diag}")
            _pbar.set_postfix_str(f"{_cond['name']} t{_step} {_diag.get('wall_s', 0)}s ok={_n_ok} skip={_n_skip}")
    del _aux
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f"[capture] done: {_n_ok} ok | {_n_skip} skipped(resume) | {_n_warn} warn | {_n_err} error")
if _n_warn or _n_err:
    print(f"[capture] see {ERROR_LOG_PATH}; incomplete or unhealthy runs are redone automatically on the next run of this cell.")


## R7. Integrity audit

This reopens every expected run file before any analysis. It checks that the
stored metadata matches the current condition definitions, decision and v\*,
that the hooks and probe fired as expected with no errors or operator
violations, that the tracked tokens match the baseline's, and that the replay
was exact. It also reruns the postcondition checks from the smoke test.

A scenario enters the paired analysis set only if its baseline and every
condition at every step pass.


In [ ]:
# =====================================================================
# R7. Integrity audit
#
# Reopens every expected run file and checks the metadata against the
# current condition, run hash, decision, v* and step. It also checks the run
# health flags, operator violations, finite arrays, and the smoke-test
# postconditions (check_run_postconditions), which include the tracked
# tokens and replay exactness. Scenarios where the baseline and every
# condition at every step pass go into CLEAN_Q8, the paired analysis set.
# Failed or stale runs are redone by running R6 again.
# =====================================================================
_clean, _problems = [], []
_reg_pos_hits, _reg_pos_total = 0, 0

def _audit_one(cond, step, sid_base, zb=None):
    rid = _rid(cond["name"], step, sid_base)
    paths = scenario_paths(Q8_PATHS["runs"], rid, False)
    if not (os.path.exists(paths["npz"]) and os.path.exists(paths["png"])):
        return f"{rid}: missing files"
    if cond["name"] == "baseline" and not os.path.exists(_replay_path(sid_base, Q8_PATHS["runs"])):
        return f"{rid}: replay cache missing"
    try:
        z = np.load(paths["npz"], allow_pickle=False)
        m = meta_from_npz(z)
        if (m["condition"] != cond["name"] or m["q8_hash"] != Q8_HASH or m["dec_sha"] != Q8_DEC_SHA
                or m.get("vstar_sha") != VSTAR_SHA or m.get("int_step") != (None if cond["name"] == "baseline" else int(step))):
            z.close(); return f"{rid}: metadata mismatch (stale decision, v*, config or step)"
        cd = m.get("cond_def", {})
        for k in ("lesion_kind", "lesion_lo", "lesion_hi", "rescue_kind", "rescue_lo", "rescue_hi", "frac", "position", "source"):
            if cd.get(k) != cond.get(k):
                z.close(); return f"{rid}: stored condition definition differs from the current one ({k})"
        if not (m["fires_ok"] and m["intv_fires_ok"] and m["probe_ok"] and m["hook_errors"] == 0 and not m["nan_flag"] and m["targets_found"]):
            z.close(); return f"{rid}: unhealthy run diagnostics"
        if m["viol_total"] != 0.0:
            z.close(); return f"{rid}: operator violations ({m['viol_total']})"
        if cond["name"] == "baseline":
            if list(int(s) for s in z["full_steps"]) != list(INT_STEPS):
                z.close(); return f"{rid}: baseline does not carry every intervention step"
            if not (np.isfinite(z["geom_pop"]).all() and np.isfinite(z["tn_all"].astype(np.float32)).all()):
                z.close(); return f"{rid}: non-finite baseline arrays"
            for si, s in enumerate(INT_STEPS):
                if not np.isfinite(z["geom_tok"][si, B_LESION:, 0]).all():
                    z.close(); return f"{rid}: target geometry not finite from the lesion block (step {s})"
                if not np.isfinite(z["rel_all"][si].astype(np.float32)).all():
                    z.close(); return f"{rid}: routing reference not finite (step {s})"
            return ("OK", z, m)
        probs = check_run_postconditions(z, m, zb, cond, int(step))
        z.close()
        if probs:
            return f"{rid}: " + "; ".join(probs[:3])
        return ("OK", None, m)
    except Exception as e:
        return f"{rid}: unreadable ({type(e).__name__}: {e})"

for _sid_base, _pid, _seed, _text in Q8_SCENARIOS:
    _res = _audit_one(COND_BY_NAME["baseline"], None, _sid_base)
    if not isinstance(_res, tuple):
        _problems.append(_res); continue
    _zb = _res[1]
    if REG_EXPECT_TOKEN is not None:
        _reg_pos_hits += int((_zb["tracked_toks"][:, 0] == REG_EXPECT_TOKEN).sum()); _reg_pos_total += len(INT_STEPS)
    _ok_all = True
    for _step in INT_STEPS:
        for _cond in CONDITIONS[1:]:
            _r = _audit_one(_cond, _step, _sid_base, zb=_zb)
            if not isinstance(_r, tuple):
                _ok_all = False; _problems.append(_r)
    _zb.close()
    if _ok_all:
        _clean.append((_sid_base, _pid, _seed, _text))

CLEAN_Q8 = _clean
print(f"[audit] {len(CLEAN_Q8)}/{len(Q8_SCENARIOS)} scenarios fully clean across all {len(CONDITIONS)} conditions and {len(INT_STEPS)} step(s)")
if REG_EXPECT_TOKEN is not None and _reg_pos_total:
    _fr = _reg_pos_hits / _reg_pos_total
    print(f"[audit] primary target at the expected register token {REG_EXPECT_TOKEN}: {_fr:.1%} of (scenario, step) pairs"
          + ("" if _fr >= 0.8 else "  [WARN: below 80 percent, inspect tracked_toks]"))
if _problems:
    print(f"[audit] {len(_problems)} problem(s), first 10:")
    for _p in _problems[:10]:
        print(f"    {_p}")
    print("[audit] incomplete or stale runs are redone by re-running R6.")
save_json_atomic({"clean": [s for s, _, _, _ in CLEAN_Q8], "problems": _problems[:300],
                  "q8_hash": Q8_HASH, "dec_sha": Q8_DEC_SHA, "vstar_sha": VSTAR_SHA,
                  "timestamp": datetime.datetime.now().isoformat(timespec="seconds")},
                 os.path.join(Q8_PATHS["analysis"], "q8_audit.json"))
if not CLEAN_Q8:
    raise RuntimeError("no clean scenarios; run R6 to completion first")
print("[audit] AUDIT PASSED for the clean set -- proceed to the analysis.")


## R8. Mediation endpoints

This is the main analysis. For each scenario, condition and block at the
intervened step we read out the attention the register receives (log10 of the
head-mean share relative to uniform, the primary readout), the fraction of
heads where it is the top sink, its key rank, how far the whole attention
pattern moved from the clean run, and the register geometry.

Lesion effects are paired differences to the clean run. The rescue fraction
of a condition is (rescue minus lesion) divided by (clean minus lesion),
computed per scenario and only where that gap is large enough. Both come with
prompt-clustered bootstrap intervals and sign and Wilcoxon tests. Equivalence
margins fixed in advance decide whether a control shows no rescue and whether
the full-state rescue counts as full. If partial v\* rescues are configured,
a dose-response test runs over them.

The figures are the mediation ladder for each lesion family, an overview of
the primary readout, dot plots of the rescue fractions, the dose-response, and
a heatmap of conditions by block.


In [ ]:
# =====================================================================
# R8. Mediation endpoints
#
# Reads every clean run once into a cache, then computes lesion effects,
# rescue fractions, the control checks, the dose-response and the ladder
# figures. Readouts per scenario, condition and block at step t*:
#
#   lshare   log10 incoming attention share of the register, head mean,
#            relative to uniform (primary)
#   ftop     fraction of heads whose top sink is the register
#   lrank    log10 of the head-median key rank of the register
#   tv, js   head-mean routing distance to the clean run over all keys
#   alpha, cos, xperp, norm, strength (norm / median norm), c* value
#   the same share for the random ordinary token, for the position controls
#
# The lesion effect is m(lesion) - m(clean). For a rescue condition c with
# lesion l, RF = (m_c - m_l) / (m_clean - m_l) per scenario, kept only where
# the clean-lesion gap is at least the metric's floor (rf_gap_floor_log10
# for the share). Tests are paired per scenario. We run an exact sign test,
# a Wilcoxon test and a prompt-level sign test with seeds collapsed. CIs of
# the median come from a prompt-clustered bootstrap that resamples prompts
# with all their seeds. Holm correction runs across the conditions of one
# family. In the lesion and rescue tables the family is all lesion
# conditions (or all rescue conditions) for one metric and block. A rescue condition
# is called "no rescue" when the CI of the median RF lies inside
# +/- rf_equiv_margin, and full when it lies inside 1 +/- rf_full_margin.
# =====================================================================
import math, warnings
from tqdm.auto import tqdm
import pandas as pd
from PIL import Image as _Image

FORCE_REBUILD_CACHE = False
_GT = {f: i for i, f in enumerate(GEOM_TOKEN_FIELDS)}
_GP = {f: i for i, f in enumerate(GEOM_POP_FIELDS)}
_PT = {f: i for i, f in enumerate(PROBE_TOKEN_FIELDS)}
_PH = {f: i for i, f in enumerate(PROBE_HEAD_FIELDS)}
_CI = {n: i for i, n in enumerate(TRACKED_CH_NAMES)}
_S, _NC, _L, _T = len(CLEAN_Q8), len(CONDITIONS), CFG.n_blocks_expected, CFG.num_steps
_H = N_HEADS_EXPECTED
_STEP = T_STAR
_SI_BASE = list(INT_STEPS).index(_STEP)
_R = len(STATE_REF_BLOCKS)
_CACHE = os.path.join(Q8_PATHS["analysis"], "mediation_cache.npz")
_ckey = sha256_of_text(json.dumps({"dec": Q8_DEC_SHA, "q8": Q8_HASH, "vstar": VSTAR_SHA, "step": _STEP,
                                   "conds": COND_NAMES, "sids": sorted(s for s, _, _, _ in CLEAN_Q8)}, sort_keys=True))
SHARE_FLOOR = 1e-3            # floor for log10 shares, a share of 0 means no attention at all

def _psnr(a, b):
    mse = float(np.mean((a - b) ** 2))
    return 99.0 if mse == 0 else 10.0 * math.log10(255.0 ** 2 / mse)

def _blur(img, sigma=8.0):
    from scipy.ndimage import gaussian_filter
    return np.stack([gaussian_filter(img[..., c], sigma) for c in range(img.shape[-1])], axis=-1)

def _build_cache():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    A = {k: np.full((_S, _NC, _L), np.nan, np.float32) for k in
         ("lshare", "ftop", "lrank", "tv", "js", "alpha", "cos", "xperp", "norm", "strength", "cstar", "c2", "ctrl",
          "lshare_rand", "ftop_rand", "cos_rand", "med_cos_all", "share_med")}
    A["top_class"] = np.full((_S, _NC, _L, 3), np.nan, np.float32)        # fraction of heads whose top sink is target / other high-norm / ordinary
    A["traj_dist"] = np.full((_S, _NC), np.nan, np.float32)
    A["state_dist_med"] = np.full((_S, _NC, _R), np.nan, np.float32)
    A["state_dist_reg"] = np.full((_S, _NC, _R), np.nan, np.float32)
    A["psnr"] = np.full((_S, _NC), np.nan, np.float32)
    A["psnr_low"] = np.full((_S, _NC), np.nan, np.float32)
    A["psnr_high"] = np.full((_S, _NC), np.nan, np.float32)
    A["strength_steps"] = np.full((_S, _NC, _T, _L), np.nan, np.float32)  # register strength at every recorded step
    A["target_tok"] = np.full((_S,), -1, np.int64)
    A["rand_tok"] = np.full((_S,), -1, np.int64)
    for si, (sid_base, pid, seed, _text) in enumerate(tqdm(CLEAN_Q8, desc="cache")):
        base_img = None; traj_clean = None; hn_by_block = None
        for ci, cond in enumerate(CONDITIONS):
            rid = _rid(cond["name"], _STEP, sid_base)
            pth = scenario_paths(Q8_PATHS["runs"], rid, False)
            img = np.asarray(_Image.open(pth["png"]).convert("RGB")).astype(np.float64)
            with np.load(pth["npz"], allow_pickle=False) as z:
                fs = [int(s) for s in z["full_steps"]]
                k = fs.index(_STEP)
                tt = z["tracked_toks"][k]
                if ci == 0:
                    A["target_tok"][si], A["rand_tok"][si] = int(tt[0]), int(tt[RAND_SLOT])
                    base_img = img
                    traj_clean = z[f"traj_t{_STEP:02d}"].astype(np.float64)
                    tn_clean = z["tn_all"][list(z["tn_steps"]).index(_STEP)].astype(np.float32)       # [L, N]
                    med = np.median(tn_clean, axis=1, keepdims=True)
                    hn_by_block = tn_clean >= float(Q8.target_norm_mult) * np.maximum(med, 1e-9)       # [L, N] clean high-norm tokens
                    A["psnr"][si, ci] = 99.0; A["psnr_low"][si, ci] = 99.0; A["psnr_high"][si, ci] = 99.0
                    A["traj_dist"][si, ci] = 0.0
                else:
                    A["psnr"][si, ci] = _psnr(img, base_img)
                    bl_c, bl_b = _blur(img), _blur(base_img)
                    A["psnr_low"][si, ci] = _psnr(bl_c, bl_b)
                    A["psnr_high"][si, ci] = _psnr(img - bl_c + 128.0, base_img - bl_b + 128.0)
                    tr = z[f"traj_t{_STEP:02d}"].astype(np.float64)
                    A["traj_dist"][si, ci] = float(np.linalg.norm(tr - traj_clean) / max(np.linalg.norm(traj_clean), 1e-12))
                pt = z["probe_tok"][k]                              # [L, H, K, 4]
                ph = z["probe_head"][k]                             # [L, H, 9]
                gt = z["geom_tok"][k]                               # [L, K, 4]
                gp = z["geom_pop"][k]                               # [L, 8]
                tv_ = z["tracked_vals"][k]                          # [L, K, K_CH+1]
                share = pt[:, :, 0, _PT["sink_rel"]]                # [L, H]
                A["lshare"][si, ci] = np.log10(np.maximum(np.nanmean(share, axis=1), SHARE_FLOOR))
                A["share_med"][si, ci] = np.nanmedian(share, axis=1)
                A["ftop"][si, ci] = np.nanmean(pt[:, :, 0, _PT["is_top_sink"]], axis=1)
                A["lrank"][si, ci] = np.log10(np.nanmedian(pt[:, :, 0, _PT["key_rank"]], axis=1))
                A["lshare_rand"][si, ci] = np.log10(np.maximum(np.nanmean(pt[:, :, RAND_SLOT, _PT["sink_rel"]], axis=1), SHARE_FLOOR))
                A["ftop_rand"][si, ci] = np.nanmean(pt[:, :, RAND_SLOT, _PT["is_top_sink"]], axis=1)
                A["tv"][si, ci] = np.nanmean(ph[:, :, _PH["tv_to_clean"]], axis=1)
                A["js"][si, ci] = np.nanmean(ph[:, :, _PH["js_to_clean"]], axis=1)
                if ci == 0:
                    A["tv"][si, ci] = 0.0; A["js"][si, ci] = 0.0
                for fld in ("alpha", "cos", "xperp", "norm"):
                    A[fld][si, ci] = gt[:, 0, _GT[fld]]
                A["cos_rand"][si, ci] = gt[:, RAND_SLOT, _GT["cos"]]
                A["strength"][si, ci] = gt[:, 0, _GT["norm"]] / np.maximum(gp[:, _GP["med_norm"]], 1e-9)
                A["med_cos_all"][si, ci] = gp[:, _GP["med_cos_all"]]
                A["cstar"][si, ci] = tv_[:, 0, _CI["cstar"]]; A["c2"][si, ci] = tv_[:, 0, _CI["c2"]]; A["ctrl"][si, ci] = tv_[:, 0, _CI["ctrl"]]
                top1 = ph[:, :, _PH["top1_idx"]]                    # [L, H]
                for b in range(_L):
                    t1 = top1[b]
                    ok = np.isfinite(t1)
                    if not ok.any():
                        continue
                    t1i = t1[ok].astype(np.int64)
                    is_t = t1i == int(tt[0])
                    is_hn = hn_by_block[b][t1i] & ~is_t
                    A["top_class"][si, ci, b] = [is_t.mean(), is_hn.mean(), (~is_t & ~is_hn).mean()]
                sd = z["state_dist"][k]                              # [R, N]
                for ri in range(_R):
                    d = sd[ri]
                    if np.isfinite(d).all():
                        A["state_dist_med"][si, ci, ri] = float(np.median(np.delete(d, [int(tt[0]), int(tt[RAND_SLOT])])))
                        A["state_dist_reg"][si, ci, ri] = float(d[int(tt[0])])
                tn = z["tn_all"].astype(np.float32); tsteps = [int(s) for s in z["tn_steps"]]
                gts = z["geom_tok_steps"]                            # [n_steps_stored, L, K, 4]
                for j, s in enumerate(tsteps):
                    if s < _STEP:
                        continue
                    medn = np.median(tn[j], axis=1)
                    A["strength_steps"][si, ci, s] = gts[j, :, 0, _GT["norm"]] / np.maximum(medn, 1e-9)
    np.savez_compressed(_CACHE, cache_key=np.array(_ckey), **A)
    return A

if os.path.exists(_CACHE) and not FORCE_REBUILD_CACHE:
    with np.load(_CACHE, allow_pickle=False) as _z:
        if str(_z["cache_key"]) == _ckey:
            CACHE = {k: _z[k].copy() for k in _z.files if k != "cache_key"}
            print(f"[mediation] cache loaded ({_S} scenarios x {_NC} conditions)")
        else:
            print("[mediation] cache stale -> rebuilding"); CACHE = _build_cache()
else:
    CACHE = _build_cache()

# ------------------------------------------------- statistics helpers
PROMPT_IDS = [pid for _, pid, _, _ in CLEAN_Q8]
_uniq_prompts = sorted(set(PROMPT_IDS))
_prompt_index = {p: [i for i, q in enumerate(PROMPT_IDS) if q == p] for p in _uniq_prompts}
try:
    from scipy.stats import wilcoxon as _wilcoxon, spearmanr as _spearmanr
except Exception:
    _wilcoxon = None; _spearmanr = None

def _sign_test_p(diffs):
    d = np.asarray(diffs, np.float64); d = d[np.isfinite(d) & (d != 0)]
    n = len(d)
    if n == 0:
        return 1.0, 0
    k = int((d > 0).sum())
    lo = sum(math.comb(n, i) for i in range(0, min(k, n - k) + 1)) / 2 ** n
    return min(1.0, 2.0 * lo), n

def cluster_boot_median(values, n_boot=None, seed=0):
    """Prompt-clustered bootstrap 95 percent CI of the median of per-scenario
    values, ignoring NaN. Prompts are resampled with replacement, and each
    sampled prompt brings all its kept scenarios."""
    v = np.asarray(values, np.float64)
    n_boot = int(Q8.n_boot if n_boot is None else n_boot)
    groups = [v[idx][np.isfinite(v[idx])] for p, idx in _prompt_index.items()]
    groups = [g for g in groups if len(g)]
    if not groups:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    P = len(groups)
    sizes = {len(g) for g in groups}
    if len(sizes) == 1:
        G = np.stack(groups)                                              # [P, k]
        draws = rng.integers(0, P, size=(n_boot, P))
        meds = np.median(G[draws].reshape(n_boot, -1), axis=1)
    else:
        meds = np.empty(n_boot)
        for i in range(n_boot):
            meds[i] = np.median(np.concatenate([groups[j] for j in rng.integers(0, P, size=P)]))
    return (float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5)))

def paired_rows(values_SC, ref_col, cols, names, equiv=None, full_at=None, full_margin=None, rel_gap=None):
    """Paired statistics of columns `cols` against column `ref_col`."""
    rows, pvals = [], []
    for ci in cols:
        dlt = values_SC[:, ci] - values_SC[:, ref_col]
        ok = np.isfinite(dlt); d = dlt[ok]
        p_sign, _ = _sign_test_p(d)
        p_wil = np.nan
        if _wilcoxon is not None and (d != 0).any():
            try:
                p_wil = float(_wilcoxon(d, alternative="two-sided").pvalue)
            except Exception:
                pass
        pm = [float(np.median(dlt[idx][np.isfinite(dlt[idx])])) for p, idx in _prompt_index.items() if np.isfinite(dlt[idx]).any()]
        p_prompt, _ = _sign_test_p(pm)
        lo, hi = cluster_boot_median(dlt)
        row = {"condition": names[ci], "n": int(len(d)), "median": float(np.median(d)) if len(d) else np.nan,
               "mean": float(np.mean(d)) if len(d) else np.nan,
               "iqr_lo": float(np.percentile(d, 25)) if len(d) else np.nan, "iqr_hi": float(np.percentile(d, 75)) if len(d) else np.nan,
               "ci_lo": lo, "ci_hi": hi, "n_pos": int((d > 0).sum()), "n_neg": int((d < 0).sum()), "n_zero": int((d == 0).sum()),
               "p_sign": p_sign, "p_wilcoxon": p_wil, "p_sign_prompt_level": p_prompt, "n_prompts": len(pm)}
        if equiv is not None:
            row["equiv_margin"] = float(equiv)
            row["equivalent"] = bool(np.isfinite(lo) and np.isfinite(hi) and lo >= -equiv and hi <= equiv)
        rows.append(row); pvals.append(p_sign)
    holm = _holm(pvals)
    for i, r in enumerate(rows):
        r["p_sign_holm"] = holm[i]
    return rows

def _holm(pvals):
    """Holm step-down adjusted p-values, in the input order."""
    order = np.argsort(pvals); run_max, m = 0.0, len(pvals); holm = {}
    for rank, oi in enumerate(order):
        run_max = max(run_max, (m - rank) * pvals[oi]); holm[oi] = min(1.0, run_max)
    return [holm[i] for i in range(m)]

def rescue_fraction(values_SC, ci, cl, floor):
    """Per-scenario rescue fraction of condition ci relative to its lesion cl."""
    gap = values_SC[:, 0] - values_SC[:, cl]
    rf = (values_SC[:, ci] - values_SC[:, cl]) / np.where(np.abs(gap) >= floor, gap, np.nan)
    return rf, gap

def lesion_of(cond):
    """Name of the lesion-only comparator of a rescue condition (same lesion, same window)."""
    if cond["rescue_kind"] == "none" or cond["family"] in ("none", "sham"):
        return None
    n = cond["lesion_kind"] + ("_maint" if cond["lesion_hi"] > B_RESCUE - 1 else "")
    return n if n in COND_BY_NAME else None

RESCUE_CONDS = [c for c in CONDITIONS if c["rescue_kind"] != "none" and not c["is_sham"] and lesion_of(c) is not None]
LESION_CONDS = [c for c in CONDITIONS if c["rescue_kind"] == "none" and c["family"] in LESION_KINDS]
_missing_les = [c["name"] for c in CONDITIONS if c["rescue_kind"] != "none" and not c["is_sham"] and lesion_of(c) is None]
if _missing_les:
    print(f"[mediation][WARN] no lesion-only comparator for {_missing_les}; their rescue fractions are skipped")

def rf_table(values_SC, block_label, metric_name, floor, higher_is_clean=True):
    """Table rows with the lesion effect of every lesion condition and the
    rescue fraction of every rescue condition, with prompt-clustered CIs."""
    rows = []
    for c in LESION_CONDS:
        ci = COND_NAMES.index(c["name"])
        r = paired_rows(values_SC, 0, [ci], COND_NAMES)[0]
        rows.append({"metric": metric_name, "block": block_label, "kind": "lesion_effect", "family": c["family"],
                     "rf_median": np.nan, "rf_ci_lo": np.nan, "rf_ci_hi": np.nan, "frac_rf_ge_half": np.nan,
                     "n_rf": 0, "verdict": "", "note": "", **{k: r[k] for k in r}})
    for c in RESCUE_CONDS:
        ci, cl = COND_NAMES.index(c["name"]), COND_NAMES.index(lesion_of(c))
        rf, gap = rescue_fraction(values_SC, ci, cl, floor)
        ok = np.isfinite(rf)
        lo, hi = cluster_boot_median(rf)
        r = paired_rows(values_SC, cl, [ci], COND_NAMES)[0]         # condition vs its lesion
        med = float(np.median(rf[ok])) if ok.any() else np.nan
        if not ok.any():
            verdict = "undefined (no lesion gap)"
        elif np.isfinite(lo) and lo >= -Q8.rf_equiv_margin and hi <= Q8.rf_equiv_margin:
            verdict = "no rescue (equivalent to lesion)"
        elif np.isfinite(lo) and lo >= 1.0 - Q8.rf_full_margin and hi <= 1.0 + Q8.rf_full_margin:
            verdict = "full rescue"
        elif np.isfinite(lo) and lo > 0.0:
            verdict = "partial rescue" if hi < 1.0 - Q8.rf_full_margin else "rescue (CI reaches full)"
        else:
            verdict = "inconclusive"
        note = ""
        if c["rescue_kind"] in ("key", "key_value") and isinstance(block_label, (int, np.integer)) \
                and c["rescue_lo"] <= int(block_label) <= c["rescue_hi"] and metric_name in ("lshare", "ftop", "lrank"):
            note = ("key patched inside this block: routing to the register is restored by construction; read the "
                    "routing distance, the downstream endpoints, and the blocks beyond the patched window")
        rows.append({"metric": metric_name, "block": block_label, "kind": "rescue", "family": c["family"],
                     "lesion": lesion_of(c), "rf_median": med, "rf_ci_lo": lo, "rf_ci_hi": hi,
                     "frac_rf_ge_half": float(np.mean(rf[ok] >= 0.5)) if ok.any() else np.nan,
                     "n_rf": int(ok.sum()), "verdict": verdict, "note": note, **{k: r[k] for k in r}})
    # Holm across the lesion conditions and, separately, across the rescue
    # conditions of this table (one metric at one block)
    for kind in ("lesion_effect", "rescue"):
        idx = [i for i, r in enumerate(rows) if r["kind"] == kind]
        for i, adj in zip(idx, _holm([rows[i]["p_sign"] for i in idx])):
            rows[i]["p_sign_holm"] = adj
    return rows

def print_rf_rows(rows, title):
    print("\n" + "=" * 110); print(title); print("=" * 110)
    for r in rows:
        if r["kind"] == "lesion_effect":
            print(f"  LESION  {r['condition']:32s} d vs clean = {r['median']:+7.3f} CI [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]  "
                  f"+/0/- {r['n_pos']}/{r['n_zero']}/{r['n_neg']}  sign p {r['p_sign']:.1e} (Holm {r['p_sign_holm']:.1e}) prompt-level p {r['p_sign_prompt_level']:.1e}")
    for r in rows:
        if r["kind"] == "rescue":
            print(f"  RESCUE  {r['condition']:32s} RF median {r['rf_median']:+6.2f} CI [{r['rf_ci_lo']:+.2f}, {r['rf_ci_hi']:+.2f}] "
                  f"(n {r['n_rf']}, {r['frac_rf_ge_half']:.0%} >= 0.5)  d vs lesion {r['median']:+.3f} sign p {r['p_sign']:.1e} "
                  f"(Holm {r['p_sign_holm']:.1e})  -> {r['verdict']}" + (f"\n          [{r['note']}]" if r.get("note") else ""))

# ------------------------------------------------- primary endpoint and tables
ALL_ROWS = []
PB = PRIMARY_PROBE
_metrics = (("lshare", "log10 register share (head mean, x uniform)", Q8.rf_gap_floor_log10),
            ("ftop", "fraction of heads with the register as top sink", 0.2),
            ("lrank", "log10 register key rank (head median)", 0.3),
            ("cos", "cos(register, v*)", 0.2),
            ("strength", "register strength (norm / median norm)", 2.0))
for mi, (key, label, floor) in enumerate(_metrics):
    for b in PROBE_BLOCKS_L:
        rows = rf_table(CACHE[key][:, :, b], b, key, floor)
        for r in rows:
            r["probe_role"] = PROBE_ROLES[PROBE_BLOCKS_L.index(b)]; r["metric_label"] = label
        ALL_ROWS += rows
        if b == PB:
            print_rf_rows(rows, f"{'PRIMARY' if mi == 0 else 'SECONDARY'} ENDPOINT at block {b} ({PROBE_ROLES[PROBE_BLOCKS_L.index(b)]}), "
                                f"step {_STEP}: {label}; n = {_S} paired scenarios; baseline median {np.nanmedian(CACHE[key][:, 0, b]):.3f}")
# routing distance is 0 for the clean run and lower is cleaner, so the rescue fraction uses -tv
for b in PROBE_BLOCKS_L:
    rows = rf_table(-CACHE["tv"][:, :, b], b, "neg_tv", 0.05)
    for r in rows:
        r["probe_role"] = PROBE_ROLES[PROBE_BLOCKS_L.index(b)]; r["metric_label"] = "routing distance to clean (total variation, head mean), negated"
    ALL_ROWS += rows
    if b == PB:
        print_rf_rows(rows, f"SECONDARY ENDPOINT at block {b}: routing distance to the clean run (negated; 0 = identical routing)")
ENDPOINTS = pd.DataFrame(ALL_ROWS)
ENDPOINTS.to_csv(os.path.join(Q8_PATHS["analysis"], "mediation_endpoints.csv"), index=False)

# position controls compare the random token's share under the _at_random conditions with its clean share
_rand_conds = [c for c in CONDITIONS if c["position"] == "random"]
if _rand_conds:
    rows = paired_rows(CACHE["lshare_rand"][:, :, PB], 0, [COND_NAMES.index(c["name"]) for c in _rand_conds], COND_NAMES,
                       equiv=Q8.share_equiv_log10)
    print("\n  POSITION CONTROLS at block %d: log10 share of the random ordinary token, condition minus clean:" % PB)
    for r in rows:
        print(f"    {r['condition']:32s} d = {r['median']:+.3f} CI [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]  equivalent(+/-{Q8.share_equiv_log10:g}) = {r['equivalent']}"
              f"  (random-token share {10 ** np.nanmedian(CACHE['lshare_rand'][:, COND_NAMES.index(r['condition']), PB]):.2f}x vs register clean "
              f"{10 ** np.nanmedian(CACHE['lshare'][:, 0, PB]):.1f}x)")
    pd.DataFrame(rows).to_csv(os.path.join(Q8_PATHS["analysis"], "position_controls.csv"), index=False)

# the sham should be equivalent to the baseline on the primary metric
_ci_sham = COND_NAMES.index("sham") if "sham" in COND_NAMES else None
if _ci_sham is not None:
    r = paired_rows(CACHE["lshare"][:, :, PB], 0, [_ci_sham], COND_NAMES, equiv=Q8.share_equiv_log10)[0]
    print(f"\n  SHAM at block {PB}: d log10 share = {r['median']:+.4f} CI [{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}], "
          f"equivalent = {r['equivalent']}; median trajectory distance {np.nanmedian(CACHE['traj_dist'][:, _ci_sham]):.2e}, "
          f"median PSNR {np.nanmedian(CACHE['psnr'][:, _ci_sham]):.1f} dB")

# dose-response over the v* fraction (lesion = 0, partial fractions, full v* = 1)
DOSE = {}
for fam in FAMILIES:
    levels = []
    les = next((c for c in LESION_CONDS if c["family"] == fam and c["lesion_hi"] == B_RESCUE - 1), None)
    if les is None:
        continue
    levels.append((0.0, COND_NAMES.index(les["name"])))
    for c in CONDITIONS:
        if c["family"] == fam and c["rescue_kind"] == "vstar" and c["position"] == "target" and c["rescue_hi"] == c["rescue_lo"] and lesion_of(c) == les["name"]:
            levels.append((float(c["frac"]), COND_NAMES.index(c["name"])))
    levels = sorted(set(levels))
    if len(levels) >= 3 and _spearmanr is not None:
        fr = np.array([f for f, _ in levels]); cols = [i for _, i in levels]
        rhos = np.array([_spearmanr(fr, CACHE["lshare"][s, cols, PB]).correlation for s in range(_S)])
        rhos = rhos[np.isfinite(rhos)]
        p_rho, _ = _sign_test_p(rhos)
        DOSE[fam] = {"levels": levels, "rho_median": float(np.median(rhos)), "frac_positive": float(np.mean(rhos > 0)), "p_sign": p_rho}
        print(f"\n  DOSE-RESPONSE ({fam}) over v* fraction {list(fr)}: per-scenario Spearman rho of the register share vs fraction, "
              f"median {np.median(rhos):+.2f}, {np.mean(rhos > 0):.0%} positive, sign test p = {p_rho:.1e}")

# per-scenario table
_per = []
for si, (sid_base, pid, seed, _t) in enumerate(CLEAN_Q8):
    for ci, cn in enumerate(COND_NAMES):
        row = {"sid": sid_base, "prompt_id": pid, "seed": seed, "condition": cn, "target_tok": int(CACHE["target_tok"][si]),
               "traj_dist": float(CACHE["traj_dist"][si, ci]), "psnr_db": float(CACHE["psnr"][si, ci])}
        for key in ("lshare", "ftop", "lrank", "tv", "cos", "strength", "alpha"):
            for b in PROBE_BLOCKS_L:
                row[f"{key}_b{b}"] = float(CACHE[key][si, ci, b])
        _per.append(row)
pd.DataFrame(_per).to_csv(os.path.join(Q8_PATHS["analysis"], "per_scenario_metrics.csv"), index=False)
print(f"[mediation] endpoint and per-scenario tables -> {Q8_PATHS['analysis']}")

# =====================================================================
# Figures
# =====================================================================
_x = np.arange(_L)
_XLAB = "transformer block"

def _fam_conds(fam):
    return [c for c in CONDITIONS if c["family"] == fam]

def collect_handles(axes):
    """Unique (handle, label) pairs over several axes, in order of appearance."""
    hs, ls = [], []
    for ax in axes:
        h, l = ax.get_legend_handles_labels()
        for hh, ll in zip(h, l):
            if ll not in ls:
                hs.append(hh); ls.append(ll)
    return hs, ls

def _plot_conds(ax, key, conds, transform=None, first=False, lw_scale=1.0):
    """Plot the median over scenarios of a per-block readout, baseline first,
    each condition in its own style from line_kwargs. Blocks before the
    lesion block are left out. The register is only identified from the
    lesion block on, and every condition is still clean before it."""
    for c in [COND_BY_NAME["baseline"]] + conds:
        ci = COND_NAMES.index(c["name"])
        v = CACHE[key][:, ci, :]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            med = np.nanmedian(v, axis=0)
        med = np.where(np.arange(len(med)) >= B_LESION, med, np.nan)
        if transform is not None:
            med = transform(med)
        ax.plot(_x, med, label=(COND_LABELS_SHORT[c["name"]] if first else None), **line_kwargs(c, lw_scale=lw_scale))

# ---- F1. mediation ladder per family, four readouts across depth
_MEDIATOR = {"cstar_zero": "cstar_zero__vstar", "vstar_destroy": "vstar_destroy__key", "vstar_random_dir": "vstar_random_dir__key",
             "reg_remove": "reg_remove__cstar"}
_LADDER_PANELS = (("lshare", "attention received by the register", "incoming attention (x uniform share)", lambda m: 10 ** m, True),
                  ("ftop", "register is the top sink", "fraction of heads", None, False),
                  ("lrank", "register-key rank", "median rank among image keys\n(1 = most aligned with queries)", lambda m: 10 ** m, True),
                  ("cos", "alignment with v*", r"$\cos(x_r, v^*)$", None, False))
for fam in FAMILIES:
    conds = _fam_conds(fam)
    _core_names = [fam, _MEDIATOR.get(fam), f"{fam}__norm", f"{fam}__full"]
    _core = [c for c in conds if c["name"] in _core_names]
    _extra = [c for c in conds if c["name"] not in _core_names]
    # with more than six conditions the variants and controls go in a second row, with the baseline and lesion repeated
    rows = [conds] if len(conds) <= 6 else [_core, [c for c in conds if c["name"] == fam] + _extra]
    fig, axes = plt.subplots(len(rows), 4, figsize=(FIG_PAGE, 2.5 * len(rows)), squeeze=False)
    for ri, rconds in enumerate(rows):
        for pi, (key, ptitle, ylab, tf, logy) in enumerate(_LADDER_PANELS):
            ax = axes[ri, pi]
            mark_blocks(ax, first=(ri == 0 and pi == 0))
            _plot_conds(ax, key, rconds, transform=tf, first=(pi == 0))
            if logy:
                ax.set_yscale("log")
            if key == "ftop":
                ax.set_ylim(-0.02, 1.02)
            if key == "cos":
                ax.set_ylim(-0.05, 1.05)
                bg = np.nanmedian(CACHE["med_cos_all"][:, 0, :], axis=0)
                ax.plot(_x, bg, color="0.55", lw=0.8, ls="--", label=("median over all image tokens (baseline)" if ri == 0 else None))
            ax.set_xlim(B_LESION - 0.5, _L - 1 + 0.5)
            ax.set_title(wrap_label(f"({'abcdefgh'[ri * 4 + pi]}) {ptitle}", 24), fontsize=_BASE_PT, pad=3)
            ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.set_ylabel(ylab, fontsize=LEGEND_PT)
            ax.tick_params(labelsize=LEGEND_PT)
    _hm, _lm = axes[0, 0].get_legend_handles_labels()
    _hm, _lm = _hm[:3], _lm[:3]                                            # band, lesion block, rescue block
    _ho, _lo = op_legend(conds)
    _hg = [plt.Line2D([], [], color="0.55", lw=0.8, ls="--")]
    _hh, _ll = _hm + _ho + _hg, _lm + _lo + ["median over all image tokens (baseline)"]
    _row_note = ("" if len(rows) == 1 else " The top row shows the lesion, the mediator rescue and the two bounds; the bottom row the "
                 "dose, position, channel and maintained variants, with the baseline and the lesion repeated for reference.")
    save_paper_fig(fig, q8_fig_stem(f"R8_ladder_{fam}"), legend_handles=_hh, legend_labels=_ll, ncol=3,
        title=(f"Register attention routing and geometry across depth under the {FAMILY_TITLES[fam].split(' (')[0]} lesion "
               f"and its rescues ({MODEL_LABEL}, step {_STEP}, median over {_S} generations)"),
        caption=(f"Mediation ladder for the {FAMILY_TITLES[fam]} family in {MODEL_LABEL} (median over {_S} scenarios, step {_STEP}). "
                 f"The lesion is applied to the register token at the output of block {B_LESION} (dotted line) and each rescue at the "
                 f"output of block {B_RESCUE} (dash-dotted line; attention rescues inside the attention of blocks {B_RESCUE} to {B_MAINT_END}); "
                 f"the first attention call that sees a rescued residual is block {PRIMARY_PROBE}. (a) The register's incoming image-to-image "
                 f"attention relative to the uniform share (mean over heads); (b) the fraction of heads whose strongest sink is the register; "
                 f"(c) the median rank of the register key among image keys by dot product with the mean image query; (d) the cosine of the "
                 f"register token with v* (grey dashed: median over all image tokens in the baseline). The shaded region is the high-norm band."
                 + _row_note))

# ---- F1b. overview of the primary readout, one panel per family
fig, axes = plt.subplots(len(FAMILIES), 1, figsize=(FIG_COL * 1.35, 1.75 * len(FAMILIES)), sharex=True, sharey=True, squeeze=False)
_hand, _labs = [], []
_all_role_conds = []
def _role_conds(fam):
    """Conditions for the overview figures, in the roles lesion only,
    mediator, norm only and full state (those that exist)."""
    names = [fam, _MEDIATOR.get(fam), f"{fam}__norm", f"{fam}__full"]
    return [COND_BY_NAME[n] for n in names if n in COND_BY_NAME]
for fi, fam in enumerate(FAMILIES):
    ax = axes[fi, 0]
    mark_blocks(ax, first=(fi == 0))
    _plot_conds(ax, "lshare", _role_conds(fam), transform=lambda m: 10 ** m, first=(fi == 0))
    _all_role_conds += _role_conds(fam)
    ax.set_yscale("log"); ax.set_xlim(B_LESION - 0.5, _L - 1 + 0.5)
    ax.set_ylabel("incoming attention\n(x uniform share)", fontsize=LEGEND_PT)
    ax.set_title(f"({'abcd'[fi]}) {FAMILY_TITLES[fam]}", fontsize=_BASE_PT, pad=3)
    ax.tick_params(labelsize=LEGEND_PT)
axes[-1, 0].set_xlabel(_XLAB, fontsize=LEGEND_PT)
_h_marks, _l_marks = axes[0, 0].get_legend_handles_labels()
_h_marks, _l_marks = _h_marks[:3], _l_marks[:3]                         # band, lesion block, rescue block
_h_ops, _l_ops = op_legend(_all_role_conds)
save_paper_fig(fig, q8_fig_stem("R8_ladders_overview"), legend_handles=_h_marks + _h_ops, legend_labels=_l_marks + _l_ops, ncol=2,
    title=(f"Register incoming attention across depth under lesion, mediator rescue and bounding rescues for the "
           f"{len(FAMILIES)} lesion-rescue families ({MODEL_LABEL}, step {_STEP}, median over {_S} generations)"),
    caption=(f"Overview of the three lesion-rescue families in {MODEL_LABEL}: the register's incoming image-to-image attention relative "
             f"to the uniform share (mean over heads, median over {_S} scenarios, step {_STEP}) across depth under the baseline, the lesion "
             f"alone, and each rescue. Lesions act at the output of block {B_LESION}, rescues at the output of block {B_RESCUE} "
             f"(attention rescues inside blocks {B_RESCUE} to {B_MAINT_END}). The shaded region is the high-norm band."))

# ---- F2. rescue fractions at the primary probe, dot plots
def condition_tick_label(cond):
    """Compact two-line label of a rescue condition for category axes."""
    rk = cond["rescue_kind"]
    base = {"vstar": "v* projection", "cstar": f"ch. {CH1}", "norm": "norm only", "full": "full state",
            "chan_c2": f"ch. {CH2}", "chan_ctrl": f"ch. {CH_CTRL}", "key": "clean key", "value": "clean value",
            "key_value": "key + value", "none": "lesion only", "sham": "sham"}[rk]
    if rk == "vstar" and cond["frac"] is not None and cond["frac"] < 1.0:
        base += f" x{cond['frac']:g}"
    tags = []
    if rk in ATTN_RESCUES and cond["rescue_hi"] == cond["rescue_lo"]:
        tags.append("1 block")
    if rk in RESIDUAL_RESCUES and cond["rescue_hi"] > cond["rescue_lo"]:
        tags.append("maintained")
    if cond["position"] == "random":
        tags.append("at random token")
    if cond["source"] == "random":
        tags.append("wrong key")
    if cond["lesion_hi"] > B_RESCUE - 1:
        tags.append("lesion maintained" if rk != "none" else "maintained")
    return base + ("\n(" + ", ".join(tags) + ")" if tags else "")

def _chars_per_tick(total_w, ax_width_in, fontsize):
    return (ax_width_in / max(total_w, 1)) * 72.0 / (0.58 * fontsize)        # characters per tick position

def group_positions(groups, titles=None, ax_width_in=None, fontsize=None):
    """Tick positions for grouped category axes.

    Each group is padded to the width its header needs, so that the longest
    word of the short title fits on one line and the short title fits in
    three lines. Returns the positions per group, the (start, end) span per
    group and the total width.
    """
    n_groups = [len(c) for c in groups]
    widths = [max(n, 1) if n else 0 for n in n_groups]
    if titles is not None and ax_width_in is not None:
        for _ in range(3):                                   # repeat, since the padding changes the characters per tick
            cpt = _chars_per_tick(sum(widths), ax_width_in, fontsize)
            for gi, (n, (full, short)) in enumerate(zip(n_groups, titles)):
                if n == 0:
                    continue
                need = max(len(w) for w in str(short).split()) + 1
                lines3 = len(str(short)) / 3.0 + 1
                widths[gi] = max(n, int(math.ceil(max(need, lines3) / max(cpt, 1e-6))))
    pos, spans, positions = 0, [], []
    for n, w in zip(n_groups, widths):
        if n == 0:
            positions.append([]); spans.append(None); continue
        off = (w - n) / 2.0
        positions.append([pos + off + i for i in range(n)])
        spans.append((pos - 0.5, pos + w - 0.5))
        pos += w
    return positions, spans, pos

def group_headers(ax, spans, titles, total_w, ax_width_in, fontsize):
    """Family headers above grouped category axes, wrapped to the group
    width (short title when the full one does not fit in two lines)."""
    cpt = _chars_per_tick(total_w, ax_width_in, fontsize)
    for span, (full, short) in zip(spans, titles):
        if span is None:
            continue
        width = max(4, int((span[1] - span[0]) * cpt))
        lab = wrap_label(full, width)
        if lab.count("\n") >= 2 or max(len(l) for l in lab.split("\n")) > width:
            lab = wrap_label(short, width)
        ax.annotate(lab, xy=(0.5 * (span[0] + span[1]), 1.0), xycoords=("data", "axes fraction"),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=fontsize, color="0.25")

def _rf_dotplot(ax, key, floor, ylab):
    xs, labs = [], []
    rng = np.random.default_rng(0)
    groups = [[c for c in RESCUE_CONDS if c["family"] == fam] for fam in FAMILIES]
    _titles = [(FAMILY_TITLES[f], FAMILY_TITLES[f].split(" (")[0]) for f in FAMILIES]
    positions, spans, total_w = group_positions(groups, _titles, 0.86 * FIG_PAGE, LEGEND_PT - 0.5)
    for conds, pp in zip(groups, positions):
        for c, pos in zip(conds, pp):
            ci, cl = COND_NAMES.index(c["name"]), COND_NAMES.index(lesion_of(c))
            rf, _ = rescue_fraction(CACHE[key][:, :, PB], ci, cl, floor)
            ok = np.isfinite(rf)
            col = condition_style(c)["color"]
            if ok.any():
                xj = pos + rng.uniform(-0.18, 0.18, int(ok.sum()))
                ax.scatter(xj, rf[ok], s=5, color=col, alpha=0.5, linewidths=0, zorder=3)
                lo, hi = cluster_boot_median(rf)
                ax.hlines(np.median(rf[ok]), pos - 0.3, pos + 0.3, color="0.1", lw=1.4, zorder=5)
                if np.isfinite(lo):
                    ax.vlines(pos, lo, hi, color="0.1", lw=0.9, zorder=5)
            xs.append(pos); labs.append(condition_tick_label(c))
    for span in [sp for sp in spans if sp is not None][:-1]:
        ax.axvline(span[1], color="0.8", lw=0.6)
    ax.set_xlim(-0.5, total_w - 0.5)
    ax.axhline(0, color=_ROLE["verm"], lw=0.8, ls="--")
    ax.axhline(1, color="0.1", lw=0.8, ls="--")
    ax.axhspan(-Q8.rf_equiv_margin, Q8.rf_equiv_margin, color=_ROLE["verm"], alpha=0.08, lw=0)
    ax.axhspan(1 - Q8.rf_full_margin, 1 + Q8.rf_full_margin, color="0.1", alpha=0.06, lw=0)
    ax.set_xticks(xs); ax.set_xticklabels(labs, rotation=90, ha="center", va="top", fontsize=LEGEND_PT - 0.5)
    ax.set_ylabel(ylab, fontsize=LEGEND_PT)
    ax.tick_params(axis="y", labelsize=LEGEND_PT)
    group_headers(ax, spans, _titles, total_w, 0.86 * FIG_PAGE, LEGEND_PT - 0.5)
    return xs

_ROLE = _ROLE_COLORS
fig, axes = plt.subplots(2, 1, figsize=(FIG_PAGE, 4.6), sharex=True)
_rf_dotplot(axes[0], "lshare", Q8.rf_gap_floor_log10, "(a) rescue fraction:\nregister share (log10)")
_rf_dotplot(axes[1], "ftop", 0.2, "(b) rescue fraction:\ntop-sink head fraction")
axes[0].scatter([], [], s=5, color="0.3", label="one generation (prompt, seed)")
axes[0].plot([], [], color="0.1", lw=1.4, label="median with prompt-clustered bootstrap 95% CI")
axes[0].plot([], [], color=_ROLE["verm"], lw=0.8, ls="--", label="0 = no rescue (lesion level); shaded: equivalence margin")
axes[0].plot([], [], color="0.1", lw=0.8, ls="--", label="1 = full rescue (clean level); shaded: full-rescue margin")
save_paper_fig(fig, q8_fig_stem("R8_rescue_fractions"), legend_ax=axes[0], ncol=2,
    title=(f"Rescue fractions of the register's incoming attention and top-sink head fraction at the first post-rescue "
           f"block {PB} ({MODEL_LABEL}, step {_STEP}, {_S} generations)"),
    caption=(f"Rescue fractions at block {PB} (the first attention call after the rescue) in {MODEL_LABEL}, step {_STEP}. For every "
             f"scenario the rescue fraction is (rescue minus lesion) / (clean minus lesion) of (a) the log10 incoming attention share "
             f"of the register (scenarios whose clean-lesion gap is below {Q8.rf_gap_floor_log10:g} log10 are excluded) and (b) the "
             f"fraction of heads whose top sink is the register (gap floor 0.2). Dots are generations, black bars the median with a "
             f"prompt-clustered bootstrap 95 percent confidence interval; 0 marks no rescue, 1 a complete rescue, and the shaded bands are "
             f"the pre-declared equivalence margins (+/- {Q8.rf_equiv_margin:g} around 0, +/- {Q8.rf_full_margin:g} around 1). Conditions "
             f"are grouped by lesion family."))

# ---- F3. dose-response over the v* fraction, if configured
if DOSE:
    fig, axes = plt.subplots(1, len(DOSE), figsize=(FIG_COL * len(DOSE), 2.4), squeeze=False)
    for pi, (fam, d) in enumerate(DOSE.items()):
        ax = axes[0, pi]
        rng = np.random.default_rng(1)
        fr = [f for f, _ in d["levels"]]; cols = [i for _, i in d["levels"]]
        for f, ci in d["levels"]:
            v = 10 ** CACHE["lshare"][:, ci, PB]
            ax.scatter(f + rng.uniform(-0.02, 0.02, len(v)), v, s=5, color=condition_style(CONDITIONS[ci])["color"], alpha=0.55, linewidths=0)
        ax.plot(fr, [10 ** np.nanmedian(CACHE["lshare"][:, ci, PB]) for ci in cols], color="0.1", lw=1.2, marker="o", ms=3,
                label="median over generations")
        ax.axhline(10 ** np.nanmedian(CACHE["lshare"][:, 0, PB]), color="0.1", ls="--", lw=0.8, label="baseline (clean) median")
        ax.set_yscale("log"); ax.set_xlabel("fraction of the clean v* projection restored", fontsize=LEGEND_PT)
        ax.set_ylabel(f"incoming attention at block {PB}\n(x uniform share)", fontsize=LEGEND_PT)
        ax.set_title(f"({'abc'[pi]}) {FAMILY_TITLES[fam]}", fontsize=_BASE_PT, pad=3)
        ax.tick_params(labelsize=LEGEND_PT)
    save_paper_fig(fig, q8_fig_stem("R8_dose_response"), legend_ax=axes[0, 0], ncol=2,
        title=(f"Dose-response of the register's incoming attention at block {PB} to the restored fraction of the clean "
               f"v* projection ({MODEL_LABEL}, step {_STEP}, {_S} generations)"),
        caption=(f"Dose-response of the v* rescue in {MODEL_LABEL}: the register's incoming attention at block {PB} (mean over heads, "
                 f"relative to uniform) when 0 (lesion only), a fraction, or all of the clean projection onto v* is restored at block "
                 f"{B_RESCUE}. Dots are generations, the line joins medians, the dashed line is the baseline median. "
                 + "; ".join(f"{FAMILY_TITLES[f]}: per-scenario Spearman rho median {d['rho_median']:+.2f}, sign test p = {d['p_sign']:.1e}" for f, d in DOSE.items()) + "."))

# ---- F4. heatmap of the primary readout, conditions x blocks from the lesion block on
_order = [COND_NAMES.index("baseline")] + ([COND_NAMES.index("sham")] if "sham" in COND_NAMES else []) + \
         [COND_NAMES.index(c["name"]) for fam in FAMILIES for c in _fam_conds(fam)]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    _hm = np.stack([np.nanmedian(CACHE["lshare"][:, ci, :], axis=0) for ci in _order])[:, B_LESION:]
def _row_label(ci):
    c = CONDITIONS[ci]
    if c["rescue_kind"] == "none" and c["family"] in LESION_KINDS:
        return f"{FAMILY_TITLES[c['family']].split(' (')[0]}: lesion only" + (" (maintained)" if c["lesion_hi"] > B_RESCUE - 1 else "")
    return COND_LABELS_SHORT[COND_NAMES[ci]]
fig, ax = plt.subplots(figsize=(FIG_PAGE, 0.16 * len(_order) + 1.3))
im = ax.imshow(_hm, aspect="auto", cmap="magma", vmin=np.log10(SHARE_FLOOR), vmax=max(float(np.nanmax(_hm)), 1.0),
               interpolation="nearest", rasterized=True, extent=(B_LESION - 0.5, _L - 0.5, len(_order) - 0.5, -0.5))
ax.set_yticks(range(len(_order))); ax.set_yticklabels([_row_label(ci) for ci in _order], fontsize=LEGEND_PT - 1)
ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.tick_params(axis="x", labelsize=LEGEND_PT)
ax.axvline(B_LESION + 0.5, color="w", ls=":", lw=0.9); ax.axvline(B_RESCUE + 0.5, color="w", ls="-.", lw=0.9)
_fam_bounds = np.cumsum([1 + int("sham" in COND_NAMES)] + [len(_fam_conds(f)) for f in FAMILIES])[:-1]
for yb in _fam_bounds:
    ax.axhline(yb - 0.5, color="w", lw=0.8)
cb = fig.colorbar(im, ax=ax, pad=0.01, shrink=0.9)
cb.set_label("log10 incoming attention (x uniform)", fontsize=LEGEND_PT); cb.ax.tick_params(labelsize=LEGEND_PT)
save_paper_fig(fig, q8_fig_stem("R8_share_heatmap"),
    title=(f"Register incoming attention per condition and transformer block ({MODEL_LABEL}, step {_STEP}, median over "
           f"{_S} generations; white lines mark the first blocks reading the lesioned and the rescued state)"),
    caption=(f"Median log10 incoming attention of the register (mean over heads, relative to uniform) for every condition (rows, grouped "
             f"by family) and block from the lesion block on (columns) in {MODEL_LABEL}, step {_STEP}, over {_S} scenarios. The column of "
             f"a block shows the attention computed inside that block; the dotted white line marks where the lesioned residual first "
             f"enters attention (block {B_LESION + 1}) and the dash-dotted line where the rescued residual first enters (block {B_RESCUE + 1})."))
print("[mediation] figures complete")


## R8b. Geometry, routing and attention maps

This cell looks at what happens around the primary endpoint. It plots the
register geometry across depth for each lesion family, and how far each
head's attention pattern moves from the clean run. It also tracks where each
head's strongest sink goes after the lesion (the register, another clean
high-norm token or an ordinary token) and whether it comes back after the
rescue.

It also draws the incoming-attention maps over the token grid at the primary
probe block for one scenario, and the register strength at the denoising
steps after the single-step intervention.


In [ ]:
# =====================================================================
# R8b. Geometry, routing, sink destination, attention maps, later steps
#
#   geometry      alpha, ||x_perp||, strength and c* value of the register
#                 across depth, one figure per family
#   routing       total-variation distance of each head's incoming attention
#                 to the clean run's, head mean, median over scenarios
#   destination   fraction of heads whose top sink is the register, another
#                 clean high-norm token or an ordinary token, per block
#   maps          incoming attention (head mean) on the token grid at the
#                 primary probe block for one scenario
#   later steps   register strength at each recorded step after the
#                 intervened step, to see whether the effect carries over
# =====================================================================
_side = int(round(math.sqrt(N_IMG)))
_KEYS_FAM = {fam: [c for c in CONDITIONS if c["family"] == fam] for fam in FAMILIES}

# ---- F5. geometry per family
for fam in FAMILIES:
    conds = _KEYS_FAM[fam]
    fig, axes = plt.subplots(1, 4, figsize=(FIG_PAGE, 2.5))
    panels = (("alpha", "(a) projection on v*", r"$\alpha = \langle x_r, v^*\rangle$", False),
              ("xperp", "(b) off-axis norm", r"$\|x_\perp\| = \|x_r - \alpha v^*\|$", True),
              ("strength", "(c) register strength", "register norm / median token norm", True),
              ("cstar", f"(d) value on channel {CH1}", f"activation of the register on ch. {CH1}", False))
    for pi, (key, ptitle, ylab, logy) in enumerate(panels):
        ax = axes[pi]
        mark_blocks(ax, first=(pi == 0))
        fl = log_floor(np.nanmedian(CACHE[key][:, :, B_LESION:], axis=0), frac=0.5) if logy else None
        _plot_conds(ax, key, conds, transform=(lambda m, fl=fl: np.maximum(m, fl)) if logy else None, first=(pi == 0))
        if logy:
            ax.set_yscale("log"); ax.set_ylim(bottom=fl)
        ax.set_xlim(B_LESION - 0.5, _L - 1 + 0.5)
        ax.set_title(wrap_label(ptitle, 24), fontsize=_BASE_PT, pad=3)
        ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.set_ylabel(ylab, fontsize=LEGEND_PT)
        ax.tick_params(labelsize=LEGEND_PT)
    save_paper_fig(fig, q8_fig_stem(f"R8b_geometry_{fam}"), legend_ax=axes[0], ncol=3,
        title=(f"Register geometry across depth under the {FAMILY_TITLES[fam].split(' (')[0]} lesion and its rescues "
               f"({MODEL_LABEL}, step {_STEP}, median over {_S} generations)"),
        caption=(f"Register geometry for the {FAMILY_TITLES[fam]} family in {MODEL_LABEL} (median over {_S} scenarios, step {_STEP}): "
                 f"(a) the projection of the register token onto v*, (b) the norm of its component orthogonal to v*, (c) its norm relative "
                 f"to the median token norm, and (d) its activation on channel {CH1}, across depth under the baseline, the lesion alone "
                 f"(block {B_LESION}, dotted) and each rescue (block {B_RESCUE}, dash-dotted; attention rescues leave the residual state "
                 f"unchanged by construction). The shaded region is the high-norm band."))

# ---- F6. routing distance to clean, one panel per family
fig, axes = plt.subplots(1, len(FAMILIES), figsize=(FIG_PAGE, 2.4), sharey=True, squeeze=False)
_hand, _labs = [], []
for fi, fam in enumerate(FAMILIES):
    ax = axes[0, fi]
    mark_blocks(ax, first=(fi == 0))
    _plot_conds(ax, "tv", _KEYS_FAM[fam], first=(fi == 0))
    ax.set_xlim(B_LESION - 0.5, _L - 1 + 0.5); ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"({'abcd'[fi]}) {wrap_label(FAMILY_TITLES[fam], max(14, int(FIG_PAGE / len(FAMILIES) * 0.8 * 72 / 4.6) - 4))}", fontsize=_BASE_PT, pad=3)
    ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.tick_params(labelsize=LEGEND_PT)
axes[0, 0].set_ylabel("routing distance to the clean run\n(total variation, mean over heads)", fontsize=LEGEND_PT)
_h_marks, _l_marks = axes[0, 0].get_legend_handles_labels()
_h_marks, _l_marks = _h_marks[:3], _l_marks[:3]
_h_ops, _l_ops = op_legend([c for fam in FAMILIES for c in _KEYS_FAM[fam]])
save_paper_fig(fig, q8_fig_stem("R8b_routing_distance"), legend_handles=_h_marks + _h_ops, legend_labels=_l_marks + _l_ops, ncol=3,
    title=(f"Total-variation distance of the incoming-attention distribution to the clean run across depth, per lesion "
           f"family ({MODEL_LABEL}, step {_STEP}, mean over heads, median over {_S} generations)"),
    caption=(f"Whole-pattern routing distance in {MODEL_LABEL}: for every head, the distribution of incoming image-to-image attention "
             f"over image keys (summed over image queries) is compared with the clean run's distribution at the same block by the "
             f"total-variation distance (0 = identical routing, 1 = disjoint); shown is the mean over heads, median over {_S} scenarios, "
             f"step {_STEP}, one panel per lesion family. Baseline is 0 by definition."))

# ---- F7. where the sink went, stacked areas for baseline, lesion and mediator rescue
_cls_cols = (_ROLE_COLORS["blue"], _ROLE_COLORS["orange"], "0.75")
_cls_labs = ("register (the lesioned token)", "another clean high-norm token", "an ordinary token")
_show = [("baseline", "baseline")]
for fam in FAMILIES:
    les = next((c["name"] for c in LESION_CONDS if c["family"] == fam and c["lesion_hi"] == B_RESCUE - 1), None)
    if les:
        _show.append((les, f"{FAMILY_TITLES[fam]}: lesion only"))
    med = _MEDIATOR.get(fam)
    if med in COND_BY_NAME:
        _show.append((med, f"{FAMILY_TITLES[fam]}: {COND_LABELS_SHORT[med].replace('+ ', '')}"))
fig, axs = panel_grid(len(_show), max_cols=4, panel_h=1.9, sharey=True)
for ax, (cn, lab) in zip(axs, _show):
    ci = COND_NAMES.index(cn)
    tc = np.nanmedian(CACHE["top_class"][:, ci, :, :], axis=0)                  # [L, 3]
    tc = np.where(np.isfinite(tc), tc, 0.0)
    ok = np.arange(_L) >= B_LESION
    ax.stackplot(_x[ok], tc[ok, 0], tc[ok, 1], tc[ok, 2], colors=_cls_cols, labels=(_cls_labs if ax is axs[0] else ("_a", "_b", "_c")), alpha=0.9, lw=0, step="mid")
    ax.axvline(B_LESION, color="0.25", ls=":", lw=0.8); ax.axvline(B_RESCUE, color="0.25", ls="-.", lw=0.7)
    ax.set_xlim(B_LESION - 0.5, _L - 1 + 0.5); ax.set_ylim(0, 1)
    ax.set_title(wrap_label(lab, 30), fontsize=LEGEND_PT, pad=2); ax.tick_params(labelsize=LEGEND_PT)
grid_labels(axs, 4, _XLAB, "fraction of heads")
save_paper_fig(fig, q8_fig_stem("R8b_sink_destination"), legend_ax=axs[0], ncol=3,
    title=(f"Destination of each head's strongest sink across depth under lesion and mediator rescue "
           f"({MODEL_LABEL}, step {_STEP}, median over {_S} generations)"),
    caption=(f"Destination of each head's strongest image-key sink across depth in {MODEL_LABEL} (median over {_S} scenarios, step "
             f"{_STEP}): the fraction of heads whose top sink is the lesioned register token, another token that is high-norm in the "
             f"clean run at that block (norm at least {Q8.target_norm_mult:g} times the median), or an ordinary token; one panel for the "
             f"baseline and for the lesion and the mediator rescue of each family. Dotted and dash-dotted lines mark the lesion and rescue blocks."))

# ---- F8. attention maps at the primary probe block for one scenario, families in rows and roles in columns
_ROLE_COLS = (("baseline", "baseline"), ("lesion", "lesion only"), ("mediator", "mediator rescue"),
              ("norm", "norm-only rescue"), ("full", "full-state rescue"))
def _role_cond(fam, role):
    if role == "baseline":
        return "baseline"
    if role == "lesion":
        return next((c["name"] for c in LESION_CONDS if c["family"] == fam and c["lesion_hi"] == B_RESCUE - 1), None)
    if role == "mediator":
        return _MEDIATOR.get(fam) if _MEDIATOR.get(fam) in COND_BY_NAME else None
    n = f"{fam}__{role}"
    return n if n in COND_BY_NAME else None
if "probe_rel" in np.load(scenario_paths(Q8_PATHS["runs"], _rid("baseline", _STEP, CLEAN_Q8[0][0]), False)["npz"], allow_pickle=False).files:
    _map_sid = next((s for s, p, sd, _t in CLEAN_Q8 if (p, sd) in DEEP_PAIRS), CLEAN_Q8[0][0])
    _pi = PROBE_BLOCKS_L.index(PRIMARY_PROBE)
    _rows = list(FAMILIES)
    fig, axes = plt.subplots(len(_rows), len(_ROLE_COLS), figsize=(FIG_PAGE, 0.92 * FIG_PAGE / len(_ROLE_COLS) * len(_rows) + 1.1), squeeze=False)
    _vmax = None; im = None
    for ri, fam in enumerate(_rows):
        for ci, (role, head) in enumerate(_ROLE_COLS):
            ax = axes[ri, ci]
            cn = _role_cond(fam, role)
            ax.set_xticks([]); ax.set_yticks([])
            if ci == 0:
                ax.set_ylabel(wrap_label(FAMILY_TITLES[fam], 22), fontsize=LEGEND_PT - 1)
            if cn is not None:
                _pt = ("baseline" if cn == "baseline" else ("lesion only" if COND_BY_NAME[cn]["rescue_kind"] == "none"
                       else COND_LABELS_SHORT[cn].replace("+ ", "")))
                ax.set_title((head + "\n" if (ri == 0 and head != _pt) else "") + wrap_label(_pt, 26), fontsize=LEGEND_PT - 1.5, pad=2)
            elif ri == 0:
                ax.set_title(head, fontsize=LEGEND_PT - 1.5, pad=2)
            if cn is None:
                ax.text(0.5, 0.5, "not in the\ncondition list", ha="center", va="center", fontsize=LEGEND_PT - 1, color="0.4", transform=ax.transAxes)
                for sp in ax.spines.values():
                    sp.set_visible(False)
                continue
            with np.load(scenario_paths(Q8_PATHS["runs"], _rid(cn, _STEP, _map_sid), False)["npz"], allow_pickle=False) as z:
                rel = z["probe_rel"][0, _pi].astype(np.float32).mean(axis=0)              # [N] head mean, x uniform
                tt = int(z["tracked_toks"][0, 0])
            if _vmax is None:
                _vmax = max(float(rel.max()), 10.0)
            im = ax.imshow(rel.reshape(_side, _side), cmap="magma", norm=LogNorm(vmin=0.1, vmax=_vmax), interpolation="nearest", rasterized=True)
            ax.scatter([tt % _side], [tt // _side], s=40, facecolors="none", edgecolors=_ROLE_COLORS["sky"], linewidths=0.8)
    cb = fig.colorbar(im, ax=axes.ravel().tolist(), pad=0.01, shrink=0.8)
    cb.set_label("incoming attention (x uniform, head mean)", fontsize=LEGEND_PT); cb.ax.tick_params(labelsize=LEGEND_PT)
    save_paper_fig(fig, q8_fig_stem("R8b_attention_maps"),
        title=(f"Incoming attention per image token at block {PRIMARY_PROBE} under lesion and rescue for one generation "
               f"({_map_sid}; {MODEL_LABEL}, step {_STEP}; the circle marks the register)"),
        caption=(f"Incoming image-to-image attention per image token (mean over heads, relative to the uniform share, log color scale) at "
                 f"block {PRIMARY_PROBE} for one scenario ({_map_sid}) in {MODEL_LABEL}, step {_STEP}: columns show the baseline, the lesion "
                 f"alone, the mediator rescue, the norm-only rescue and the full-state rescue; rows are the lesion families. The circle marks "
                 f"the register token; the {_side} by {_side} grid is the token raster (row-major)."))

# ---- F9. register strength difference to baseline at the steps after the intervention
_pairs = []
for fam in FAMILIES:
    les = next((c["name"] for c in LESION_CONDS if c["family"] == fam and c["lesion_hi"] == B_RESCUE - 1), None)
    med = _MEDIATOR.get(fam)
    for cn in (les, med):
        if cn in COND_BY_NAME:
            _pairs.append(cn)
if _pairs and Q8.store_tn_later_steps:
    steps_after = list(range(_STEP, _T))
    fig, axs = panel_grid(len(_pairs), max_cols=len(_pairs) if len(_pairs) <= 3 else 3, panel_h=2.0, sharey=True)
    vmax = 0.0
    hm = []
    for cn in _pairs:
        ci = COND_NAMES.index(cn)
        d = np.log10(np.maximum(CACHE["strength_steps"][:, ci, steps_after, :], 1e-2)) - np.log10(np.maximum(CACHE["strength_steps"][:, 0, steps_after, :], 1e-2))
        d = np.nanmedian(d, axis=0)                                                      # [n_steps_after, L]
        hm.append(d); vmax = max(vmax, float(np.nanmax(np.abs(d[:, B_LESION:]))) if np.isfinite(d[:, B_LESION:]).any() else 0.0)
    vmax = max(vmax, 0.1)
    for ax, cn, d in zip(axs, _pairs, hm):
        im = ax.imshow(d[:, B_LESION:], aspect="auto", origin="lower", cmap=SIGNED_CMAP, vmin=-vmax, vmax=vmax, interpolation="nearest",
                       rasterized=True, extent=(B_LESION - 0.5, _L - 0.5, steps_after[0] - 0.5, steps_after[-1] + 0.5))
        ax.set_title(wrap_label(COND_LABELS[cn], 34), fontsize=LEGEND_PT - 0.5, pad=2)
        ax.tick_params(labelsize=LEGEND_PT)
        from matplotlib.ticker import MaxNLocator
        ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=8)); ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    grid_labels(axs, len(_pairs) if len(_pairs) <= 3 else 3, _XLAB, "denoising step")
    cb = fig.colorbar(im, ax=axs, pad=0.01, shrink=0.9)
    cb.set_label("log10 register strength, condition minus baseline", fontsize=LEGEND_PT); cb.ax.tick_params(labelsize=LEGEND_PT)
    save_paper_fig(fig, q8_fig_stem("R8b_cross_step_persistence"),
        title=(f"Register strength relative to baseline at later denoising steps after a single-step lesion or rescue at "
               f"step {_STEP} ({MODEL_LABEL}, median over {_S} generations)"),
        caption=(f"Cross-step propagation of the single-step interventions in {MODEL_LABEL}: median over {_S} scenarios of the log10 "
                 f"register strength (register norm over median token norm) minus the baseline value, per block (from the lesion block "
                 f"{B_LESION}) and denoising step from the intervened step {_STEP} on, for the lesion alone and the mediator rescue of "
                 f"each family. The register identity is the baseline's at step {_STEP}. Effects at steps after {_STEP} arise only "
                 f"through the latent, because the operators act at step {_STEP} alone."))
print("[geometry/routing] complete")


## R9. Downstream rescue and verification

Here we check whether the rescues also repair what comes after the
attention. We compare the model's prediction at the intervened step (the
guidance-combined one for classifier-free guidance models), the state of
the ordinary tokens at the reference blocks and the final image (PSNR, plus
LPIPS when the package is available) with the clean run. Each gets rescue
fractions, and the sham gives the noise floor.

The cell also makes a contact sheet of example images and a set of plots
that confirm the interventions did what they should across depth (c\*
value, cosine with v\*, register strength, and the register key against the
clean key).


In [ ]:
# =====================================================================
# R9. Downstream rescue and verification views
#
#   trajectory    relative L2 distance of the model's prediction at the
#                 intervened step to the clean prediction. This is the
#                 velocity or noise estimate the sampler uses, guidance
#                 combined for CFG models. Rescue fractions use the negated
#                 distance, as for the routing endpoint.
#   state         median over ordinary tokens of the relative distance of the
#                 block output to the clean state at the reference blocks
#   image         PSNR of the final image to the baseline image (whole image,
#                 low-pass and high-pass parts), and LPIPS when available
#   verification  c* value, cosine with v* and strength of the register
#                 across depth, and the relative difference of the register
#                 key to the clean key for the attention rescues
# =====================================================================
_TRAJ_ROWS = []
for key, label, floor in (("traj_dist", "trajectory distance (relative L2 of the prediction at the intervened step)", 1e-4),
                          ("psnr", "image PSNR to baseline (dB)", 0.5)):
    vals = CACHE[key].astype(np.float64)
    sign = -1.0 if key == "traj_dist" else 1.0             # rescue fractions assume higher is cleaner
    rows = rf_table(sign * vals, "final", key, floor)
    for r in rows:
        r["metric_label"] = label; r["probe_role"] = "downstream"
    _TRAJ_ROWS += rows
    print_rf_rows(rows, f"DOWNSTREAM ENDPOINT: {label} (lesion effect vs clean; rescue fractions on the {'negated ' if sign < 0 else ''}metric)")
for ri, b in enumerate(STATE_REF_BLOCKS):
    vals = CACHE["state_dist_med"][:, :, ri].astype(np.float64)
    rows = rf_table(-vals, b, "state_dist_med", 1e-3)
    for r in rows:
        r["metric_label"] = "ordinary-token state distance to clean (median relative L2), negated"; r["probe_role"] = PROBE_ROLES[PROBE_BLOCKS_L.index(b)]
    _TRAJ_ROWS += rows
    print_rf_rows(rows, f"DOWNSTREAM ENDPOINT at block {b}: ordinary-token state distance to the clean run (negated)")
pd.DataFrame(_TRAJ_ROWS).to_csv(os.path.join(Q8_PATHS["analysis"], "downstream_endpoints.csv"), index=False)
_sham_i = COND_NAMES.index("sham") if "sham" in COND_NAMES else None
if _sham_i is not None:
    print(f"\n  noise floor (sham): trajectory distance median {np.nanmedian(CACHE['traj_dist'][:, _sham_i]):.2e} "
          f"(max {np.nanmax(CACHE['traj_dist'][:, _sham_i]):.2e}); PSNR median {np.nanmedian(CACHE['psnr'][:, _sham_i]):.1f} dB")

# ---- optional LPIPS (AlexNet) of each condition's image against the baseline image
LPIPS = None
try:
    if os.environ.get("Q8_SKIP_LPIPS") == "1":
        raise RuntimeError("disabled by the Q8_SKIP_LPIPS environment variable")
    try:
        import lpips as _lpips
    except ImportError:
        if "_pip" in globals():
            _pip("lpips")                      # quiet pip install helper from S0
        else:
            import subprocess as _sp, sys as _sys
            _sp.run([_sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)
        import lpips as _lpips
    _net = _lpips.LPIPS(net="alex", verbose=False).to(DEVICE).eval()
    LPIPS = np.full((_S, _NC), np.nan, np.float32)
    def _to_t(img):
        return torch.from_numpy(np.asarray(img, np.float32) / 127.5 - 1.0).permute(2, 0, 1)[None].to(DEVICE)
    with torch.no_grad():
        for si, (sid_base, pid, seed, _t) in enumerate(tqdm(CLEAN_Q8, desc="lpips")):
            base = _to_t(_Image.open(scenario_paths(Q8_PATHS["runs"], _rid("baseline", _STEP, sid_base), False)["png"]).convert("RGB"))
            for ci, cn in enumerate(COND_NAMES):
                if ci == 0:
                    LPIPS[si, ci] = 0.0; continue
                im = _to_t(_Image.open(scenario_paths(Q8_PATHS["runs"], _rid(cn, _STEP, sid_base), False)["png"]).convert("RGB"))
                LPIPS[si, ci] = float(_net(base, im).item())
    del _net
    np.save(os.path.join(Q8_PATHS["analysis"], "lpips.npy"), LPIPS)
    print("[downstream] LPIPS (AlexNet) computed for every run")
except Exception as _e:
    print(f"[downstream] LPIPS skipped ({type(_e).__name__}: {_e}); PSNR only")

# ---- F10. downstream dot plots, all conditions grouped by family
_conds_dl = [c for c in CONDITIONS if c["name"] != "baseline"]
def _dot_panel(ax, values, ylab, logy=False, hline=None):
    rng = np.random.default_rng(2)
    xs, labs = [], []
    groups = [("sham", [c for c in _conds_dl if c["is_sham"]])] + [(fam, [c for c in _conds_dl if c["family"] == fam]) for fam in FAMILIES]
    _titles = [(FAMILY_TITLES.get(g, g), FAMILY_TITLES.get(g, g).split(" (")[0]) for g, _ in groups]
    positions, spans, total_w = group_positions([g[1] for g in groups], _titles, 0.86 * FIG_PAGE, LEGEND_PT - 0.5)
    for (gname, conds), pp in zip(groups, positions):
        for c, pos in zip(conds, pp):
            ci = COND_NAMES.index(c["name"])
            v = values[:, ci]; ok = np.isfinite(v)
            col = condition_style(c)["color"]
            if ok.any():
                ax.scatter(pos + rng.uniform(-0.18, 0.18, int(ok.sum())), v[ok], s=5, color=col, alpha=0.5, linewidths=0, zorder=3)
                lo, hi = cluster_boot_median(v)
                ax.hlines(np.median(v[ok]), pos - 0.3, pos + 0.3, color="0.1", lw=1.4, zorder=5)
                if np.isfinite(lo):
                    ax.vlines(pos, lo, hi, color="0.1", lw=0.9, zorder=5)
            xs.append(pos); labs.append("sham" if c["is_sham"] else condition_tick_label(c))
    for span in [sp for sp in spans if sp is not None][:-1]:
        ax.axvline(span[1], color="0.8", lw=0.6)
    ax.set_xlim(-0.5, total_w - 0.5)
    if hline is not None:
        ax.axhline(hline, color="0.5", lw=0.7, ls="--")
    if logy:
        ax.set_yscale("log")
    ax.set_xticks(xs); ax.set_xticklabels(labs, rotation=90, ha="center", va="top", fontsize=LEGEND_PT - 0.5)
    ax.set_ylabel(ylab, fontsize=LEGEND_PT); ax.tick_params(axis="y", labelsize=LEGEND_PT)
    group_headers(ax, spans, _titles, total_w, 0.86 * FIG_PAGE, LEGEND_PT - 0.5)

_n_pan = 3 + (1 if LPIPS is not None else 0)
fig, axes = plt.subplots(_n_pan, 1, figsize=(FIG_PAGE, 2.1 * _n_pan + 0.8), sharex=True)
_tr = np.maximum(CACHE["traj_dist"], 1e-7)
_dot_panel(axes[0], _tr, f"(a) step-{_STEP} prediction:\nrelative distance to clean", logy=True,
           hline=(float(np.nanmedian(_tr[:, _sham_i])) if _sham_i is not None else None))
_sd = np.maximum(CACHE["state_dist_med"][:, :, -1], 1e-7)
_dot_panel(axes[1], _sd, f"(b) ordinary-token state at block {STATE_REF_BLOCKS[-1]}:\nmedian relative distance to clean", logy=True)
_dot_panel(axes[2], CACHE["psnr"], "(c) final image:\nPSNR to baseline (dB)")
if LPIPS is not None:
    _dot_panel(axes[3], LPIPS, "(d) final image:\nLPIPS to baseline")
axes[0].scatter([], [], s=5, color="0.3", label="one generation (prompt, seed)")
axes[0].plot([], [], color="0.1", lw=1.4, label="median with prompt-clustered bootstrap 95% CI")
axes[0].plot([], [], color="0.5", lw=0.7, ls="--", label="sham median (numerical noise floor)")
save_paper_fig(fig, q8_fig_stem("R9_downstream_rescue"), legend_ax=axes[0], ncol=3,
    title=(f"Downstream effects of lesion and rescue per condition: prediction distance at the intervened step, "
           f"ordinary-token state distance and image similarity ({MODEL_LABEL}, step {_STEP}, {_S} generations)"),
    caption=(f"Downstream readouts per condition in {MODEL_LABEL} ({_S} scenarios, intervention at step {_STEP}): (a) relative L2 "
             f"distance of the model's prediction at step {_STEP} to the clean prediction; (b) median over ordinary tokens of the "
             f"relative L2 distance of the block-{STATE_REF_BLOCKS[-1]} output to the clean state; (c) PSNR of the final image to the "
             f"baseline image" + ("; (d) LPIPS distance" if LPIPS is not None else "") + ". Dots are generations, black bars medians "
             f"with prompt-clustered bootstrap 95 percent confidence intervals; the dashed line in (a) is the sham median, the numerical "
             f"noise floor of a hook that writes the register back unchanged. Conditions are grouped by lesion family."))

# ---- F11. contact sheet for the pilot pairs, baseline plus lesion, mediator and full rescue per family
_sheet = [(s, p, sd) for s, p, sd, _t in CLEAN_Q8 if (p, sd) in DEEP_PAIRS][:5] or [(s, p, sd) for s, p, sd, _t in CLEAN_Q8[:5]]
_cols = ["baseline"]
for fam in FAMILIES:
    for cn in (fam, _MEDIATOR.get(fam), f"{fam}__full"):
        if cn in COND_BY_NAME and cn not in _cols:
            _cols.append(cn)
if _sheet:
    fig, axes = plt.subplots(len(_sheet), len(_cols), figsize=(FIG_PAGE, 0.95 * FIG_PAGE / len(_cols) * len(_sheet) + 0.6), squeeze=False)
    for ri, (sid_base, pid, seed) in enumerate(_sheet):
        for ci, cn in enumerate(_cols):
            png = scenario_paths(Q8_PATHS["runs"], _rid(cn, _STEP, sid_base), False)["png"]
            axes[ri, ci].imshow(_Image.open(png)); axes[ri, ci].set_xticks([]); axes[ri, ci].set_yticks([])
            if ri == 0:
                _fam_short = FAMILY_TITLES[COND_BY_NAME[cn]["family"]].split(" (")[0] if cn != "baseline" else ""
                _ttl = ("baseline" if cn == "baseline" else
                        (f"{_fam_short}: lesion only" if COND_BY_NAME[cn]["rescue_kind"] == "none"
                         else f"{_fam_short}: {COND_LABELS_SHORT[cn].replace('+ ', '')}"))
                axes[ri, ci].set_title(wrap_label(_ttl, 16), fontsize=LEGEND_PT - 1.5, pad=2)
            if ci == 0:
                axes[ri, ci].set_ylabel(sid_base, fontsize=LEGEND_PT - 1)
    save_paper_fig(fig, q8_fig_stem("R9_image_contact_sheet"),
        title=(f"Generated images under the baseline, the lesion, the mediator rescue and the full-state rescue of each "
               f"family for {len(_sheet)} generations ({MODEL_LABEL}, single-step interventions at step {_STEP})"),
        caption=(f"Final images for {len(_sheet)} scenarios (rows) of {MODEL_LABEL} under the baseline and, for each lesion family, the "
                 f"lesion alone, the mediator rescue and the full-state rescue (columns). All interventions act at denoising step "
                 f"{_STEP} only; every other step follows the clean trajectory."))

# ---- F12. verification views
fig, axes = plt.subplots(1, 4, figsize=(FIG_PAGE, 2.5))
_ver = (("cstar", "cstar_zero", f"(a) c* suppression:\nch. {CH1} of the register", f"activation on ch. {CH1}", False),
        ("cos", "vstar_destroy", "(b) v* destruction:\ncos(register, v*)", r"$\cos(x_r, v^*)$", False),
        ("strength", "reg_remove", "(c) register removal:\nregister strength", "register norm / median norm", True))
for pi, (key, fam, ptitle, ylab, logy) in enumerate(_ver):
    ax = axes[pi]
    mark_blocks(ax, first=(pi == 0))
    conds = _KEYS_FAM.get(fam, [])
    fl = log_floor(np.nanmedian(CACHE[key][:, :, B_LESION:], axis=0), frac=0.5) if logy else None
    _plot_conds(ax, key, conds, transform=(lambda m, fl=fl: np.maximum(m, fl)) if logy else None, first=False)
    if logy:
        ax.set_yscale("log"); ax.set_ylim(bottom=fl)
    ax.set_xlim(B_LESION - 0.5, _L - 1 + 0.5)
    ax.set_title(ptitle, fontsize=_BASE_PT, pad=3); ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.set_ylabel(ylab, fontsize=LEGEND_PT)
    ax.tick_params(labelsize=LEGEND_PT)
# (d) register key vs clean key for the attention rescues
ax = axes[3]
mark_blocks(ax, first=False)
_att = [c for c in CONDITIONS if c["rescue_kind"] in ATTN_RESCUES]
_kd_rows = []
for c in _att[:6]:
    diffs = np.full((_S, _L), np.nan, np.float32)
    for si, (sid_base, pid, seed, _t) in enumerate(CLEAN_Q8):
        with np.load(scenario_paths(Q8_PATHS["runs"], _rid(c["name"], _STEP, sid_base), False)["npz"], allow_pickle=False) as z, \
             np.load(scenario_paths(Q8_PATHS["runs"], _rid("baseline", _STEP, sid_base), False)["npz"], allow_pickle=False) as zb:
            if "k_cap" not in z.files:
                continue
            k = z["k_cap"][0, :, SELECT_ROW, :, 0, :].astype(np.float32)            # [L, H, Dh] conditional row, primary target
            kb = zb["k_cap"][_SI_BASE, :, SELECT_ROW, :, 0, :].astype(np.float32)
            diffs[si] = np.linalg.norm((k - kb).reshape(_L, -1), axis=1) / np.maximum(np.linalg.norm(kb.reshape(_L, -1), axis=1), 1e-9)
    med = np.nanmedian(diffs, axis=0)
    ax.plot(_x, med, label=COND_LABELS_SHORT[c["name"]], **line_kwargs(c))
    _kd_rows.append({"condition": c["name"], **{f"b{b}": float(med[b]) for b in range(B_LESION, _L)}})
ax.set_yscale("log"); ax.set_xlim(B_LESION - 0.5, _L - 1 + 0.5)
ax.set_title("(d) attention rescues:\nregister key vs. clean key", fontsize=_BASE_PT, pad=3)
ax.set_xlabel(_XLAB, fontsize=LEGEND_PT); ax.set_ylabel("relative L2 difference of the\nregister's final key to clean", fontsize=LEGEND_PT)
ax.tick_params(labelsize=LEGEND_PT)
_h0, _l0 = axes[0].get_legend_handles_labels()
_h0, _l0 = _h0[:3], _l0[:3]                                              # band, lesion block, rescue block
_plotted = {c["name"] for _k, f2, _t, _y, _l in _ver for c in _KEYS_FAM.get(f2, [])} | {c["name"] for c in _att[:6]}
_h_ops, _l_ops = op_legend([COND_BY_NAME[n] for n in COND_NAMES if n in _plotted])
_hand, _labs = list(_h0) + _h_ops, list(_l0) + _l_ops
pd.DataFrame(_kd_rows).to_csv(os.path.join(Q8_PATHS["analysis"], "key_difference_to_clean.csv"), index=False)
save_paper_fig(fig, q8_fig_stem("R9_verification"), legend_handles=_hand, legend_labels=_labs, ncol=3,
    title=(f"Verification of the interventions: channel {CH1}, cosine with v*, register strength and final-key difference "
           f"to the clean run across depth ({MODEL_LABEL}, step {_STEP}, median over {_S} generations)"),
    caption=(f"Verification of the interventions in {MODEL_LABEL} (median over {_S} scenarios, step {_STEP}): (a) the register's "
             f"activation on channel {CH1} under the c* suppression family (exactly zero at block {B_LESION}, restored at block {B_RESCUE} "
             f"by the v*, c* and full rescues), (b) its cosine with v* under the v* destruction family (zero at block {B_LESION}), (c) its "
             f"norm relative to the median token under the register removal family, and (d) the relative difference between the "
             f"register's final key (the key passed to attention) and the clean key inside and outside the "
             f"attention-rescue windows. Lines follow the condition styles of the ladder figures."))
print("[downstream] complete")


## R10. Run manifest

This writes `analysis_q8_rescue/q8_manifest.json`. It records the decision,
v\*, the condition definitions, the primary endpoint rows at the primary
probe block, the dose-response, the figure overlap check, storage use, and
a short guide for reading the rescue fractions. The secondary and downstream
endpoints stay in their CSV files, and the manifest points to them.


In [ ]:
# =====================================================================
# R10. Run manifest
#
# Writes analysis_q8_rescue/q8_manifest.json.
# =====================================================================
def _du_gib(path):
    total = 0
    for r, _d, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(r, f))
            except OSError:
                pass
    return total / 2**30

_prim = ENDPOINTS[(ENDPOINTS.metric == "lshare") & (ENDPOINTS.block == PRIMARY_PROBE)]
_manifest = {
    "experiment": "Q8 restoration and mediation: lesion-rescue pairs on the register token (v1)",
    "model_id": CFG.model_id, "model_family": MODEL_FAMILY,
    "decision": {k: Q8_DEC[k] for k in ("ch1", "ch2", "ch_ctrl", "ctrl_source", "l_star", "t_star", "band", "b_lesion", "b_rescue",
                                         "b_maint_end", "primary_probe", "probe_blocks", "state_ref_blocks", "intervention_steps",
                                         "n_targets", "vstar_sha", "dec_sha") if k in Q8_DEC},
    "decision_rules": Q8_DEC.get("rules"),
    "vstar": {k: VSTAR_INFO.get(k) for k in ("vstar_sha", "source", "n_rows", "n_scenarios", "abs_cos_with_e_ch1", "energy_on_ch1",
                                             "energy_on_ch2", "top_singular_energy", "split_half_abs_cos")},
    "conditions": [{k: c.get(k) for k in ("name", "family", "lesion_kind", "lesion_lo", "lesion_hi", "rescue_kind", "rescue_lo",
                                          "rescue_hi", "frac", "channel_role", "position", "source", "is_sham")} for c in CONDITIONS],
    "condition_labels": COND_LABELS,
    "scenario_scope": Q8.scenario_scope, "n_scenarios_clean": len(CLEAN_Q8), "n_prompts": len(_uniq_prompts),
    "primary_endpoint": {
        "definition": (f"log10 incoming attention share of the register (mean over heads, relative to uniform) at block {PRIMARY_PROBE}, "
                       f"step {T_STAR}; lesion effect = lesion minus clean; rescue fraction = (rescue minus lesion) / (clean minus lesion) "
                       f"per scenario with a gap floor of {Q8.rf_gap_floor_log10:g} log10; prompt-clustered bootstrap CIs"),
        "equivalence_margins": {"no_rescue_rf": Q8.rf_equiv_margin, "full_rescue_rf": Q8.rf_full_margin, "share_log10": Q8.share_equiv_log10},
        "baseline_median": float(np.nanmedian(CACHE["lshare"][:, 0, PRIMARY_PROBE])),
        "rows": _prim.to_dict(orient="records")},
    "secondary_endpoints_csv": "mediation_endpoints.csv",
    "downstream_endpoints_csv": "downstream_endpoints.csv",
    "dose_response": DOSE,
    "comparability": COMPARABILITY,
    "figure_overlap_audit": {k: v["overlaps"] for k, v in _FIG_CHECKS.items()},
    "q8_hash": Q8_HASH, "config_hash": CONFIG_HASH, "precision_mode": PRECISION_MODE, "lib_versions": LIB_VERSIONS,
    "storage_gib": {"runs": round(_du_gib(Q8_PATHS["runs"]), 2), "figures": round(_du_gib(Q8_PATHS["figs"]), 2),
                    "analysis": round(_du_gib(Q8_PATHS["analysis"]), 2)},
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "interpretation_guide": {
        "cstar_zero": ("RF(v*) high with RF(norm) near 0 and RF(full) near 1: the register's projection on v* mediates the effect of "
                       "c* on routing. RF(v*) well below RF(full): coordinates other than v*, changed during the gap, also matter. "
                       "RF(v*) and RF(norm) both near 0: the rescue point is too late or the network erases the restored state "
                       "(check the maintained variant)."),
        "vstar_destroy": ("key rescue restores routing throughout the patched range while the single-block key rescue loses the sink "
                          "at the next block: the key is recomputed from the residual direction at every block, so the chain v* -> key "
                          "-> sink holds. Trajectory and state rescued by the key alone as much as by the full state: the register's "
                          "downstream function is routing; value-only rescue mattering instead: content read from the register matters."),
        "reg_remove": ("RF(c*) high: one coordinate at the natural position rebuilds the sink on an ordinary residual; RF(c*) near 0 "
                       "with RF(full) near 1: the rest of the register state is needed (relevant to Q5); the random-position control "
                       "is expected to stay at the lesion level."),
    },
}
save_json_atomic(_manifest, os.path.join(Q8_PATHS["analysis"], "q8_manifest.json"))
_bad_figs = {k: v for k, v in _manifest["figure_overlap_audit"].items() if v}
print("=" * 70)
print("Q8 MANIFEST / HANDOFF")
print("=" * 70)
print(f"  model {MODEL_LABEL} | c* {CH1} | controls {CH2}, {CH_CTRL} | lesion {B_LESION} -> rescue {B_RESCUE} -> primary probe {PRIMARY_PROBE} | v* {VSTAR_SHA}")
print(f"  clean scenarios: {len(CLEAN_Q8)} ({len(_uniq_prompts)} prompts) | conditions: {len(CONDITIONS)} | steps {INT_STEPS}")
print(f"  {'condition':34s} {'RF median':>10s} {'CI':>18s}  verdict")
for r in _prim.itertuples():
    if r.kind == "rescue":
        print(f"  {r.condition:34s} {r.rf_median:+10.2f} [{r.rf_ci_lo:+.2f}, {r.rf_ci_hi:+.2f}]  {r.verdict}")
    else:
        print(f"  {r.condition:34s} lesion effect {r.median:+.3f} log10 (CI [{r.ci_lo:+.3f}, {r.ci_hi:+.3f}], Holm p {r.p_sign_holm:.1e})")
print(f"  figures with overlapping text: {len(_bad_figs)}" + (f" -> {list(_bad_figs)[:6]}" if _bad_figs else " (none)"))
print(f"  storage: " + ", ".join(f"{k} {v} GiB" for k, v in _manifest["storage_gib"].items()))
print(f"  -> {os.path.join(Q8_PATHS['analysis'], 'q8_manifest.json')}")
print("=" * 70)
